# ♻️ Reator de pirólise de plástico em 3D: escoamento, calor e química (Colab)

Simulação de um **reator agitado contínuo** que recebe **plástico pós-consumo derretido** (poliolefinas: HDPE, LDPE e PP) vindo de uma extrusora e o **craqueia a ~400 °C** em vapores de óleo e cera. Tudo em Python, com **GPU** no Colab, e com **validação** contra dados abertos da literatura.

| Etapa | O que faz |
|---|---|
| 1 | **Cinética e propriedades**: cisão aleatória de cadeias calibrada com termogravimetria (TGA), reologia do fundido (Cross + lei de Raju, ramo de ceras) |
| 2 | **Reator ideal 0-D**: curva de capacidade (vazão × temperatura da parede) |
| 3 | **CFD 3D** no referencial da fita helicoidal: Navier–Stokes, turbulência **k-ω SST** com lei de parede, viscosidade μ(γ̇, T, Mw), temperatura e estado do polímero em regime permanente, controle de nível |
| 4 | Resultados: cortes (T, Mw, viscosidade, velocidade), mapa de calor na parede, números do ponto de operação |
| 5 | **Validação** em três níveis (componentes, química, sistema) com veredito por item |
| 6 | **Imagem 3D** do reator em corte (ray marching na GPU) |
| 7 | **Vídeo em HD** com o resultado logo nos primeiros segundos (versão completa ~1 min e versão curta ~25 s) e texto sugerido para o post |

**Antes de rodar:** *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4*. Sem GPU use o preset `teste` (malha grossa; roda, mas demora mais e os números mudam alguns %).

Hipóteses, equações e referências: `docs/design_brief_en.md` e `docs/parameters.json` no repositório.

In [ ]:
#@title Ambiente: GPU e pacote `pyrokit`
import os, sys, torch
gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
print("GPU:", gpu or "nenhuma (use o preset 'teste')")
os.makedirs("pyrokit", exist_ok=True)
os.makedirs("saida", exist_ok=True)

In [ ]:
%%writefile pyrokit/__init__.py
"""pyrokit: reator de pirólise de plástico (tanque agitado) em 3D, com reologia do fundido e cinética."""
__version__ = "0.1.0"

In [ ]:
%%writefile pyrokit/ops.py
"""Operações de malha deslocada (MAC) compartilhadas pelos solvers."""
from __future__ import annotations

import torch


def sl(a, axis, start, stop=None):
    idx = [slice(None)] * a.ndim
    idx[axis] = slice(start, stop)
    return a[tuple(idx)]


def pad(a, axis, n=1):
    """Replica as células de borda (gradiente nulo)."""
    lo, hi = sl(a, axis, 0, 1), sl(a, axis, -1, None)
    return torch.cat([lo] * n + [a] + [hi] * n, dim=axis)


def avg(a, axis):
    return 0.5 * (sl(a, axis, 1, None) + sl(a, axis, 0, -1))


def diff(a, axis):
    return sl(a, axis, 1, None) - sl(a, axis, 0, -1)


def to_faces(c, axis):
    """Centro → faces ao longo de axis (faces de borda recebem o valor da célula de borda)."""
    return avg(pad(c, axis), axis)


def cgrad(q, axis, h):
    """Gradiente central de um campo nos centros (unilateral nas bordas)."""
    qp = pad(q, axis)
    return (sl(qp, axis, 2, None) - sl(qp, axis, 0, -2)) / (2 * h)


def extend_into_solid(q, fluid_mask, passes: int = 3):
    """Estende para dentro do sólido (máscara False) os valores do fluido vizinho, por médias sucessivas.

    Usado para a viscosidade: no sólido a taxa de cisalhamento é nula e um modelo pseudoplástico daria
    viscosidades enormes, que contaminariam a tensão nas faces da interface."""
    known = fluid_mask.to(q.dtype)
    val = q * known
    pool = lambda a: torch.nn.functional.avg_pool3d(a[None, None], 3, 1, 1, count_include_pad=False)[0, 0]  # noqa: E731
    for _ in range(passes):
        s, c = pool(val * known), pool(known)
        newly = (known == 0) & (c > 0)
        val = torch.where(newly, s / c.clamp(min=1e-12), val)
        known = torch.maximum(known, newly.to(q.dtype))
    return torch.where(known > 0, val, q)

In [ ]:
%%writefile pyrokit/geometry.py
"""Geometria do reator: vaso cilíndrico encamisado e agitador tipo âncora, por funções de distância.

Convenção do Tank: a função de distância com sinal (SDF) é positiva dentro do sólido e negativa no
líquido (as formas do agitador usam a convenção usual, negativa dentro, e o Tank troca o sinal).
O agitador é descrito no referencial que gira com ele (onde está parado); o vaso, no mesmo referencial,
é uma parede que gira em sentido contrário. Fundo plano em z = 0 (face da malha) e superfície livre plana
em z = H_L (sem vórtice central: hipótese usual para fundidos viscosos e vasos sem chicanas).
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field

import numpy as np


class _NP:
    """Operações das SDFs em NumPy (malha) — a mesma geometria roda em PyTorch na renderização (GPU)."""
    hypot, atan2, mod, abs, sin = np.hypot, np.arctan2, np.mod, np.abs, np.sin

    @staticmethod
    def clip(x, lo, hi):
        return np.clip(x, lo, hi)

    @staticmethod
    def max(a, b):
        return np.maximum(a, b)

    @staticmethod
    def min(a, b):
        return np.minimum(a, b)

    @staticmethod
    def stack(xs):
        return np.stack(xs, -1)

    @staticmethod
    def norm(q):
        return np.linalg.norm(q, axis=-1)

    @staticmethod
    def maxlast(q):
        return q.max(-1)

    @staticmethod
    def full(p, v):
        return np.full(p.shape[:-1], v)


class _TORCH:
    @staticmethod
    def _t(x, ref):
        return x if hasattr(x, "dtype") and not isinstance(x, (float, int)) else ref.new_tensor(float(x))

    hypot = staticmethod(lambda a, b: __import__("torch").hypot(a, b))
    atan2 = staticmethod(lambda a, b: __import__("torch").atan2(a, b))
    mod = staticmethod(lambda a, b: __import__("torch").remainder(a, b))
    abs = staticmethod(lambda a: a.abs())
    sin = staticmethod(lambda a: a.sin())

    @staticmethod
    def clip(x, lo, hi):
        import torch
        return torch.minimum(torch.maximum(x, _TORCH._t(lo, x)), _TORCH._t(hi, x))

    @staticmethod
    def max(a, b):
        import torch
        ref = a if hasattr(a, "new_tensor") else b
        return torch.maximum(_TORCH._t(a, ref), _TORCH._t(b, ref))

    @staticmethod
    def min(a, b):
        import torch
        ref = a if hasattr(a, "new_tensor") else b
        return torch.minimum(_TORCH._t(a, ref), _TORCH._t(b, ref))

    @staticmethod
    def stack(xs):
        import torch
        return torch.stack(xs, -1)

    @staticmethod
    def norm(q):
        import torch
        return torch.linalg.vector_norm(q, dim=-1)

    @staticmethod
    def maxlast(q):
        return q.max(-1).values

    @staticmethod
    def full(p, v):
        return p.new_full(p.shape[:-1], v)


def _xp(p):
    return _NP if isinstance(p, np.ndarray) else _TORCH


def _box_sdf(p, center, half):
    """SDF de uma caixa alinhada aos eixos (p: [..., 3], NumPy ou PyTorch)."""
    X = _xp(p)
    q = X.stack([X.abs(p[..., k] - center[k]) - half[k] for k in range(3)])
    outside = X.norm(X.max(q, 0.0))
    inside = X.min(X.maxlast(q), 0.0)
    return outside + inside


@dataclass
class Anchor:
    """Agitador âncora: dois braços verticais junto à parede, barra inferior e eixo central.

    D: diâmetro externo [m]; w: largura radial dos braços [m]; t: espessura (tangencial) [m];
    h_arm: altura dos braços a partir do fundo da barra [m]; c_b: folga no fundo [m];
    r_shaft: raio do eixo [m]."""
    D: float
    w: float
    t: float
    h_arm: float
    c_b: float
    r_shaft: float

    def sdf(self, p):
        Ra = 0.5 * self.D
        z0 = self.c_b
        zc = z0 + 0.5 * self.h_arm
        arms = [_box_sdf(p, (s * (Ra - 0.5 * self.w), 0.0, zc), (0.5 * self.w, 0.5 * self.t, 0.5 * self.h_arm))
                for s in (-1.0, 1.0)]
        bar = _box_sdf(p, (0.0, 0.0, z0 + 0.5 * self.w), (Ra, 0.5 * self.t, 0.5 * self.w))
        X = _xp(p)
        shaft = X.hypot(p[..., 0], p[..., 1]) - self.r_shaft
        shaft = X.max(shaft, z0 - p[..., 2])                # o eixo começa na barra inferior
        return X.min(X.min(arms[0], arms[1]), X.min(bar, shaft))


@dataclass
class HelicalRibbon:
    """Dupla fita helicoidal (dois filetes a 180°) com eixo e braços radiais de fixação.

    D: diâmetro externo [m]; w: largura radial da fita [m]; t: espessura [m]; pitch: passo [m];
    z0, z1: alturas inicial e final [m]; r_shaft: raio do eixo [m]; n_arms: níveis de braços."""
    D: float
    w: float
    t: float
    pitch: float
    z0: float
    z1: float
    r_shaft: float
    n_arms: int = 3
    phase: float = 0.0

    def sdf(self, p):
        X = _xp(p)
        x, y, z = p[..., 0], p[..., 1], p[..., 2]
        r = X.hypot(x, y)
        th = X.atan2(y, x)
        Ro = 0.5 * self.D
        Ri = Ro - self.w
        # distância normal à superfície helicoidal: Δθ ao filete mais próximo (dois filetes, período π),
        # projetado na normal da hélice dentro da superfície cilíndrica
        th_h = self.phase + 2 * math.pi * z / self.pitch
        dth = X.mod(th - th_h + 0.5 * math.pi, math.pi) - 0.5 * math.pi
        rm = X.clip(r, Ri, Ro)
        beta = X.atan2(self.pitch + 0 * rm, 2 * math.pi * rm)    # ângulo da hélice com a horizontal
        d_n = X.abs(rm * dth) * X.sin(beta) - 0.5 * self.t
        d_r = X.max(Ri - r, r - Ro)
        d_z = X.max(self.z0 - z, z - self.z1)
        q = X.stack([d_n, d_r, d_z])
        ribbon = X.norm(X.max(q, 0.0)) + X.min(X.maxlast(q), 0.0)
        out = X.min(ribbon, X.max(r - self.r_shaft, self.z0 - z))         # fitas + eixo
        # braços radiais nos níveis inferior, intermediário(s) e superior, alinhados aos filetes
        for zl in np.linspace(self.z0 + 0.5 * self.w, self.z1 - 0.5 * self.w, self.n_arms):
            ang = self.phase + 2 * math.pi * zl / self.pitch
            c, s = math.cos(ang), math.sin(ang)
            xr, yr = c * x + s * y, -s * x + c * y              # eixo x' ao longo do braço
            out = X.min(out, _box_sdf(X.stack([xr, yr, z]), (0.0, 0.0, zl),
                                      (Ro - 0.5 * self.w, 0.5 * self.t, 0.5 * self.w)))
        return out


@dataclass
class Shaft:
    """Só o eixo central (caso de verificação de Couette circular)."""
    r_shaft: float

    def sdf(self, p):
        return _xp(p).hypot(p[..., 0], p[..., 1]) - self.r_shaft


@dataclass
class Tank:
    """Vaso cilíndrico de diâmetro T [m] com líquido até H_L [m] e um agitador (Anchor ou Shaft)."""
    T: float
    H_L: float
    impeller: object = None
    rpm: float = 30.0
    meta: dict = field(default_factory=dict)

    @property
    def R(self):
        return 0.5 * self.T

    @property
    def omega(self):
        return 2.0 * math.pi * self.rpm / 60.0

    def vessel_sdf(self, p):
        """Positiva fora do raio interno (parede e camisa)."""
        return np.hypot(p[..., 0], p[..., 1]) - self.R

    def impeller_sdf(self, p):
        """Positiva dentro do agitador (as formas usam a convenção usual, negativa dentro)."""
        if self.impeller is None:
            return np.full(p.shape[:-1], -1e3)         # sem agitador: tudo fora do sólido
        return -self.impeller.sdf(p)

    def wall_distance(self, p):
        """Distância à parede mais próxima (lateral, fundo, agitador); a superfície livre não é parede."""
        d_side = -self.vessel_sdf(p)
        d_bot = p[..., 2]
        d_imp = np.abs(self.impeller_sdf(p))
        return np.maximum(np.minimum.reduce([d_side, d_bot, d_imp]), 0.0)

    @property
    def impeller_diameter(self):
        imp = self.impeller
        return getattr(imp, "D", 2 * getattr(imp, "r_shaft", 0.0))

    def describe(self) -> str:
        imp = self.impeller
        s = (f"vaso T = {self.T:.2f} m, líquido até {self.H_L:.2f} m "
             f"({math.pi * self.R ** 2 * self.H_L:.2f} m³); {self.rpm:.0f} rpm")
        if isinstance(imp, (Anchor, HelicalRibbon)):
            kind = "âncora" if isinstance(imp, Anchor) else "dupla fita helicoidal"
            s += (f"; {kind} D = {imp.D:.2f} m (D/T = {imp.D / self.T:.2f}, folga na parede "
                  f"{1e3 * (self.R - 0.5 * imp.D):.0f} mm)")
        return s


def ribbon_tank(T: float = 0.8, H_over_T: float = 1.0, D_over_T: float = 0.90, rpm: float = 30.0,
                w_over_D: float = 0.10, pitch_over_D: float = 1.0, t: float | None = None) -> Tank:
    """Vaso padrão com dupla fita helicoidal (p/d = 1, w/d = 0,1; folga c/T = 0,05 para ter ≥ 4 células no vão)."""
    H_L = H_over_T * T
    D = D_over_T * T
    c = 0.5 * (T - D)
    rib = HelicalRibbon(D=D, w=w_over_D * D, t=t if t is not None else 0.035 * T, pitch=pitch_over_D * D,
                        z0=c, z1=H_L - c, r_shaft=0.035 * T)
    return Tank(T=T, H_L=H_L, impeller=rib, rpm=rpm)


def standard_anchor_tank(T: float = 1.6, H_over_T: float = 1.0, D_over_T: float = 0.92, rpm: float = 30.0,
                         w_over_D: float = 0.10, t: float | None = None) -> Tank:
    """Vaso padrão com âncora (proporções típicas de reatores de fundido: D/T ≈ 0,9–0,95, w/D ≈ 0,1)."""
    H_L = H_over_T * T
    D = D_over_T * T
    w = w_over_D * D
    c_b = 0.5 * (T - D)                     # folga no fundo igual à folga na parede
    anc = Anchor(D=D, w=w, t=t if t is not None else 0.04 * T, h_arm=0.85 * H_L - c_b, c_b=c_b,
                 r_shaft=0.035 * T)
    return Tank(T=T, H_L=H_L, impeller=anc, rpm=rpm)

In [ ]:
%%writefile pyrokit/kinetics.py
"""Cinética de pirólise dos polímeros (literatura aberta) e validação por termogravimetria (TGA).

Modelo primário: cisão aleatória com comprimento mínimo não volátil L = 2 (Simha–Wall; forma de
Sánchez-Jiménez et al.), resolvido pela fração de ligações rompidas x:

    dx/dt = k(T)(1 − x),     α = x²  (fração volatilizada),     k = A exp(−E/RT)

Taxa específica de volatilização da massa que resta: 2 k x / (1 + x). Energias de ativação de Aboulkas
et al. (2010, isoconversional): HDPE 238, LDPE 215, PP 179 kJ/mol; PS 180 kJ/mol (meio da faixa da
literatura); fatores pré-exponenciais calibrados para os picos de DTG de consenso a 10 K/min (HDPE 475,
LDPE 465, PP 455, PS 420 °C). PET usa primeira ordem com 15% de resíduo (Dubdub e Al-Yaari, 2023).

Cada cisão física mapeia na cisão aparente por x_fís = 0,068 x (L = 20 unidades C2), o que dá a massa
molar média do fundido: Mn = m0 / (1/DPn0 + x_fís). Seletividade dos voláteis num tanque agitado a
425–450 °C: gás 0,07, óleo 0,70, cera 0,23 (polietileno/polipropileno); estireno 0,65 para PS.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field

import numpy as np

R_GAS = 8.314462618


@dataclass(frozen=True)
class Polymer:
    key: str
    name: str
    E: float               # energia de ativação aparente [J/mol]
    A: float               # fator pré-exponencial [1/s]
    model: str             # "rs2" (cisão aleatória, L = 2) ou "first" (primeira ordem)
    char: float            # fração da massa reagida que vira resíduo sólido
    tpeak_10: float        # pico de DTG de consenso a 10 K/min [°C] (alvo de validação)
    dH: float              # entalpia de pirólise + vaporização dos produtos [J/kg] (endotérmica)
    split: tuple           # seletividade dos voláteis (gás, óleo, cera) em massa
    m0: float = 0.028      # massa da unidade de cisão [kg/mol] (C2 para poliolefinas)
    Mn0: float = 30.0      # massa molar numérica da resina [kg/mol]
    sources: str = ""


POLYMERS = {
    "HDPE": Polymer("HDPE", "polietileno de alta densidade", 238e3, 5.2e14, "rs2", 0.0, 475.0, 1.0e6,
                    (0.07, 0.70, 0.23), 0.028, 30.0,
                    "E: Aboulkas et al. 2010; ΔH: Stoliarov & Walters (DSC, 920 ± 120 J/g); seletividade: tanque agitado"),
    "LDPE": Polymer("LDPE", "polietileno de baixa densidade", 215e3, 1.92e13, "rs2", 0.0, 465.0, 1.0e6,
                    (0.07, 0.70, 0.23), 0.028, 30.0, "E: Aboulkas et al. 2010"),
    "PP": Polymer("PP", "polipropileno", 179e3, 6.98e10, "rs2", 0.0, 455.0, 1.0e6,
                  (0.08, 0.80, 0.12), 0.042, 60.0, "E: Aboulkas et al. 2010; ΔH: Stoliarov & Walters (faixa 630–1310 J/g)"),
    "PS": Polymer("PS", "poliestireno", 180e3, 4.1e11, "rs2", 0.0, 420.0, 0.8e6,
                  (0.02, 0.98, 0.0), 0.104, 100.0, "E: meio da faixa 150–204 kJ/mol; produtos: estireno 0,65"),
    "PET": Polymer("PET", "poli(tereftalato de etileno)", 210e3, 2.59e13, "first", 0.15, 435.0, 0.4e6,
                   (0.47, 0.53, 0.0), 0.192, 20.0, "Dubdub & Al-Yaari 2023 (pico 680 K a 2 K/min)"),
}


def rate(p: Polymer, T):
    """k(T) [1/s] (T em K, escalar ou array)."""
    return p.A * np.exp(-p.E / (R_GAS * np.asarray(T, dtype=float)))


def volatilisation_specific(p: Polymer, x, T):
    """Taxa de volatilização por massa de fundido restante [1/s] para o estado x."""
    k = rate(p, T)
    if p.model == "rs2":
        return 2.0 * k * x / (1.0 + x)
    return k * (1.0 - p.char) * np.ones_like(np.asarray(x, dtype=float))


def Mn_of_x(p: Polymer, x):
    """Massa molar numérica do fundido [kg/mol] pela cisão aleatória (x_fís = 0,068 x)."""
    dpn0 = p.Mn0 / p.m0
    return p.m0 / (1.0 / dpn0 + 0.068 * np.asarray(x, dtype=float))


# ------------------------------------------------------------------------- TGA
def tga(p: Polymer, beta_K_min: float, T0: float = 400.0, T1: float = 950.0, n: int = 6000):
    """Termogravimetria a taxa constante: devolve T [K], α (fração volatilizada) e dα/dT [1/K]."""
    T = np.linspace(T0, T1, n)
    beta = beta_K_min / 60.0
    dT = T[1] - T[0]
    if p.model == "rs2":
        # dx/dT = (k/β)(1 − x)  →  integral exata por passo: ln(1−x) = −∫k dT/β
        k = rate(p, T)
        I = np.concatenate([[0.0], np.cumsum(0.5 * (k[1:] + k[:-1]) * dT)]) / beta
        x = 1.0 - np.exp(-I)
        alpha = x ** 2
    else:
        k = rate(p, T)
        I = np.concatenate([[0.0], np.cumsum(0.5 * (k[1:] + k[:-1]) * dT)]) / beta
        alpha = (1.0 - np.exp(-I)) * (1.0 - p.char)
    dadT = np.gradient(alpha, T)
    return T, alpha, dadT


def tga_peak(p: Polymer, beta_K_min: float) -> float:
    """Temperatura do pico de DTG [°C]."""
    T, _, d = tga(p, beta_K_min)
    i = int(np.argmax(d))
    # refino parabólico do máximo
    if 0 < i < len(T) - 1:
        y0, y1, y2 = d[i - 1], d[i], d[i + 1]
        den = y0 - 2 * y1 + y2
        off = 0.5 * (y0 - y2) / den if den != 0 else 0.0
        return float(T[i] + off * (T[1] - T[0]) - 273.15)
    return float(T[i] - 273.15)


def kissinger(p: Polymer, betas=(2, 5, 10, 20, 50)):
    """E recuperada pelo método de Kissinger [J/mol]: inclinação de ln(β/Tp²) × 1/Tp (ICTAC)."""
    Tp = np.array([tga_peak(p, b) + 273.15 for b in betas])
    y = np.log(np.asarray(betas) / Tp ** 2)
    slope = np.polyfit(1.0 / Tp, y, 1)[0]
    return -slope * R_GAS, Tp - 273.15


def half_life(p: Polymer, T_C: float) -> float:
    """Tempo para volatilizar 50% em isoterma [min] (limite cinético)."""
    k = float(rate(p, T_C + 273.15))
    if p.model == "rs2":
        x = math.sqrt(0.5)
        return -math.log(1 - x) / k / 60.0
    return -math.log(1 - 0.5 / (1 - p.char)) / k / 60.0


# ------------------------------------------------------------------- misturas
@dataclass
class Feed:
    """Carga do reator: frações mássicas por polímero (as poliolefinas dominam as cargas reais)."""
    name: str
    fractions: dict = field(default_factory=dict)
    source: str = ""

    def normalised(self):
        s = sum(self.fractions.values())
        return {k: v / s for k, v in self.fractions.items()}

    def polymers(self):
        return [(POLYMERS[k], w) for k, w in self.normalised().items()]

    def describe(self):
        return ", ".join(f"{k} {100 * w:.0f}%" for k, w in self.normalised().items())


FEEDS = {
    "HDPE": Feed("HDPE puro", {"HDPE": 1.0}),
    "PP": Feed("PP puro", {"PP": 1.0}),
    "poliolefinas": Feed("mistura de poliolefinas", {"LDPE": 0.40, "HDPE": 0.35, "PP": 0.25},
                         "mistura típica de resíduos pós-consumo sem PVC/PET"),
    "pos-consumo": Feed("plástico pós-consumo selecionado", {"HDPE": 0.30, "LDPE": 0.35, "PP": 0.25, "PS": 0.10},
                        "composição de demonstração da literatura; PVC e PET excluídos (cloro, oxigenados)"),
}


def tga_blend(feed: Feed, beta_K_min: float):
    """TGA de uma mistura, por aditividade das curvas α_i(T)."""
    T = None
    alpha = 0.0
    for p, w in feed.polymers():
        T, a, _ = tga(p, beta_K_min)
        alpha = alpha + w * a
    return T, alpha, np.gradient(alpha, T)

In [ ]:
%%writefile pyrokit/rheology.py
"""Reologia e propriedades térmicas do fundido que craqueia (literatura aberta; ver README).

Viscosidade (modelo de Cross com superposição tempo–temperatura e dependência da massa molar):

    η(γ̇, T, Mw) = η_piso + η0 / [1 + (η0 γ̇ / τ*)^(1−n)],      η0 = η_ref(Mw) · a_T(T)
    η_ref = K Mw^α  (Mw ≥ Mc)   ou   K Mc^α (Mw/Mc)  (Mw < Mc, regime de Rouse)
    a_T = exp[(Ea/R)(1/T − 1/T_ref)]

com Mw = Mn [2 + (PDI0 − 2)(Mn/Mn0)] (a cisão aleatória leva a polidispersão a 2). HDPE: K = 3,4e−15,
α = 3,6 a 190 °C (Raju et al. 1979), Mc = 3 800 g/mol, Ea = 27 kJ/mol; PP: η_ref = 1,45e−6 (Mw/1000)^3,745
a 230 °C, Ea = 42 kJ/mol; PS: K = 3,9e−15, α = 3,4 a 200 °C, Ea ≈ 55 kJ/mol (300–450 °C). n e τ*: faixas
típicas de bancos de dados de injeção. Misturas: média logarítmica ponderada pela massa.

Propriedades: densidade de Zoller & Walsh (isóbaras de Tait), cp do ATHAS (Wunderlich), condutividade
de fundido indo para a de cera líquida quando Mn cai abaixo de Mc.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np

R_GAS = 8.314462618


@dataclass(frozen=True)
class Rheo:
    K: float          # η_ref = K Mw^α  [Pa s, Mw em g/mol]
    alpha: float
    Mc: float         # massa molar crítica de emaranhamento [g/mol]
    Tref: float       # [K]
    Ea: float         # energia de ativação de fluxo [J/mol]
    n: float          # índice de pseudoplasticidade (Cross)
    tau_star: float   # [Pa]
    Mw0: float        # massa molar ponderal da resina [g/mol]
    PDI0: float

    @property
    def Mn0(self):
        return self.Mw0 / self.PDI0


RHEO = {
    "HDPE": Rheo(3.4e-15, 3.6, 3800.0, 463.15, 27e3, 0.35, 2e4, 150e3, 8.0),
    "LDPE": Rheo(3.4e-15, 3.6, 3800.0, 463.15, 55e3, 0.30, 3e4, 200e3, 10.0),
    "PP": Rheo(1.45e-6 * 1000.0 ** -3.745, 3.745, 7000.0, 503.15, 42e3, 0.30, 2e4, 300e3, 5.0),
    "PS": Rheo(3.9e-15, 3.4, 33000.0, 473.15, 55e3, 0.25, 3e4, 280e3, 2.5),
    "PET": Rheo(3.4e-15, 3.6, 3800.0, 553.15, 60e3, 0.70, 1e5, 40e3, 2.0),
}

ETA_FLOOR = 2e-4     # hidrocarboneto líquido quente [Pa s]
ETA_MAX = 1e5


def Mw_of_Mn(r: Rheo, Mn):
    """Massa molar ponderal a partir da numérica sob cisão aleatória [g/mol]."""
    Mn = np.asarray(Mn, dtype=float)
    return Mn * (2.0 + (r.PDI0 - 2.0) * np.clip(Mn / r.Mn0, 0.0, 1.0))


def eta0(r: Rheo, T, Mw, xp=np):
    """Viscosidade a taxa nula [Pa s] (T em K, Mw em g/mol). xp: numpy ou torch."""
    Mw = xp.clip(Mw, 1.0, None) if xp is np else Mw.clamp(min=1.0)
    above = r.K * Mw ** r.alpha
    below = r.K * r.Mc ** r.alpha * (Mw / r.Mc)
    ref = xp.where(Mw >= r.Mc, above, below)
    aT = xp.exp((r.Ea / R_GAS) * (1.0 / T - 1.0 / r.Tref))
    return ref * aT


def viscosity(r: Rheo, gamma, T, Mw, xp=np):
    """Modelo de Cross com piso [Pa s]."""
    e0 = eta0(r, T, Mw, xp)
    if xp is np:
        e = ETA_FLOOR + e0 / (1.0 + (e0 * np.asarray(gamma) / r.tau_star) ** (1.0 - r.n))
        return np.clip(e, ETA_FLOOR, ETA_MAX)
    e = ETA_FLOOR + e0 / (1.0 + (e0 * gamma / r.tau_star) ** (1.0 - r.n))
    return e.clamp(ETA_FLOOR, ETA_MAX)


# ------------------------------------------------------------- propriedades
def density(key: str, T, Mn=None):
    """Massa específica do líquido [kg/m³] (T em K); abaixo de 2 kg/mol tende à de alcanos líquidos."""
    Tc = np.asarray(T, dtype=float) - 273.15
    if key in ("HDPE", "LDPE"):
        rho = 1000.0 / (1.1595 + 8.0e-4 * Tc)
    elif key == "PP":
        rho = 738.6 / (1.0 + 6.7e-4 * (Tc - 230.0))
    elif key == "PS":
        rho = 1000.0 / (0.9287 * np.exp(5.131e-4 * Tc))
    else:
        rho = 1300.0 * (1.0 - 6e-4 * (Tc - 25.0))
    if Mn is not None:
        f = np.clip(np.asarray(Mn, dtype=float) / 2000.0, 0.0, 1.0)
        rho = f * rho + (1 - f) * 610.0
    return rho


def heat_capacity(key: str, T):
    """cp do líquido [J/(kg K)]."""
    T = np.asarray(T, dtype=float)
    if key in ("HDPE", "LDPE"):
        return (17.48 + 0.0412 * T) / 0.014027
    return {"PP": 3000.0, "PS": 2300.0, "PET": 1900.0}.get(key, 2800.0) * np.ones_like(T)


def conductivity(key: str, Mn=None, Mc=3800.0):
    """Condutividade térmica [W/(m K)]: fundido → cera líquida quando Mn < Mc."""
    k_melt = {"HDPE": 0.24, "LDPE": 0.22, "PP": 0.17, "PS": 0.16, "PET": 0.20}.get(key, 0.2)
    if Mn is None:
        return k_melt
    return 0.10 + (k_melt - 0.10) * np.clip(np.asarray(Mn, dtype=float) / Mc, 0.0, 1.0)


def boiling_cut_molar_mass(T):
    """Massa molar [g/mol] do n-alcano que ferve a T (1 atm), pela correlação de Riazi:
    Tb = 1070 − exp(6,98291 − 0,02013 M^(2/3)). Frações mais leves saem como vapor."""
    T = np.asarray(T, dtype=float)
    return ((6.98291 - np.log(1070.0 - T)) / 0.02013) ** 1.5


def normal_boiling_point(M):
    """Tb [K] do n-alcano de massa molar M [g/mol] (Riazi)."""
    return 1070.0 - np.exp(6.98291 - 0.02013 * np.asarray(M, dtype=float) ** (2.0 / 3.0))

In [ ]:
%%writefile pyrokit/chemistry.py
"""Química da fase líquida: fechamento ψ + 3 classes (cisão aleatória com evaporação) e reator 0-D.

Em cada polímero i, por kg de líquido, transportam-se Y_i (massa derivada de i), Z_i = Y_i ψ_i (mols de
cadeias, aditivo na mistura) e as massas nas classes emaranhadas E_i1, E_i2, E_i3 (Mw ≥ 64, 16–64 e
4–16 kg/mol). A cisão aleatória rompe ligações à taxa k_c = f A exp(−E/RT) (o mesmo A, E da TGA, com o
fator f do mapeamento da cisão aparente na física); os fragmentos menores que N* = M*(T)/m0, o corte de
ebulição a T (Riazi), evaporam:

    S_v,i = ρ k_c N*² m0 Z_i                                  (vapor, kg/(m³ s))
    w_Z,i = ρ k_c Y_i/m0 − 2 S_v,i/(m0 N*)                     (cisões criam cadeias; vapor leva fragmentos)
    w_Y,i = −S_v,i (1 + s_C),   w_sólidos = s_C Σ S_v,i        (s_C: coque por kg de vapor)
    w_E1 = −r1 ρ E1,  w_E2 = ρ(r1 E1 − r2 E2),  w_E3 = ρ(r2 E2 − r3 E3)

e a massa molar ponderal sai das classes (Mw_i), governando a viscosidade. Conferido contra o balanço
populacional exato (massa retida 1–3%, Mw 13–16%) no documento de projeto (docs/design_brief_en.md).

O reator 0-D (tanque perfeitamente misturado, evaporativo, com nível constante e purga de sólidos)
dá a condição inicial do 3-D e a curva de capacidade: vazão processada e temperatura do fundido em
função da temperatura da parede.
"""
from __future__ import annotations

import math
from dataclasses import dataclass, field

import numpy as np

from . import rheology as rh
from .kinetics import POLYMERS, R_GAS

M_LO = (64.0, 16.0, 4.0)          # limites inferiores das classes [kg/mol]
MBAR = tuple(1.848 * m for m in M_LO)


@dataclass(frozen=True)
class Closure:
    key: str
    f: float           # k_c = f k_app (cisão física por ligação)
    m0: float          # massa por ligação do esqueleto [kg/mol]
    Mw0: float         # [kg/mol]
    PDI0: float
    dH: float          # calor de gaseificação por kg de vapor (DSC, já inclui a dessorção) [J/kg]
    k_melt: float      # condutividade do fundido [W/(m K)]
    rheo: str

    @property
    def Mn0(self):
        return self.Mw0 / self.PDI0


CLOSURE = {
    "HDPE": Closure("HDPE", 0.0347, 0.014027, 150.0, 8.0, 920e3, 0.24, "HDPE"),
    "LDPE": Closure("LDPE", 0.0347, 0.014027, 200.0, 10.0, 920e3, 0.22, "LDPE"),
    "PP": Closure("PP", 0.053, 0.021, 300.0, 5.0, 1310e3, 0.17, "PP"),
}

S_CHAR = 0.005            # coque por kg de vapor
SOLIDS_TARGET = 0.10      # fração de sólidos mantida no líquido pela purga


@dataclass
class FeedSpec:
    name: str
    poly: dict                      # frações mássicas dos polímeros (base sem cinzas)
    ash: float = 0.0                # cinzas na entrada do reator (fração mássica)
    T_feed: float = 573.15          # fundido vindo da extrusora [K]
    note: str = ""

    def items(self):
        s = sum(self.poly.values())
        return [(CLOSURE[k], v / s) for k, v in self.poly.items()]

    def describe(self):
        s = sum(self.poly.values())
        return ", ".join(f"{k} {100 * v / s:.0f}%" for k, v in self.poly.items()) + \
            (f"; cinzas {100 * self.ash:.1f}%" if self.ash else "")


FEEDS = {
    "F1": FeedSpec("HDPE puro", {"HDPE": 1.0}, ash=0.002, note="caso de validação"),
    "F2": FeedSpec("PP puro", {"PP": 1.0}, ash=0.01),
    "F3": FeedSpec("poliolefinas pós-consumo (Brasil)", {"HDPE": 0.60, "LDPE": 0.05, "PP": 0.35}, ash=0.01,
                   note="São Carlos: HDPE 59,4 / LDPE 4,0 / PP 36,5 (Matos, 2006), renormalizado"),
}


# ------------------------------------------------------------------ cinética
def k_c(c: Closure, T):
    p = POLYMERS[c.key]
    return c.f * p.A * np.exp(-p.E / (R_GAS * np.asarray(T, dtype=float)))


def n_star(c: Closure, T, f_cut: float = 1.0):
    """Comprimento de corte (ligações do esqueleto) dos fragmentos que evaporam a T."""
    return f_cut * rh.boiling_cut_molar_mass(T) / 1000.0 / c.m0


def class_rates(c: Closure, T):
    kc = k_c(c, T)
    r1 = (kc / (2 * c.m0)) / (1.0 / 64.0 - 1.0 / c.Mw0)
    r2 = kc * M_LO[1] / (1.5 * c.m0)
    r3 = kc * M_LO[2] / (1.5 * c.m0)
    return r1, r2, r3


def Mw_of_state(c: Closure, Y, Z, E):
    """Massa molar ponderal [kg/mol] a partir de Y, Z e das massas das classes."""
    Y = np.maximum(Y, 1e-12)
    Es = sum(E)
    matrix = np.maximum(Y - Es, 1e-12)
    zm = np.maximum(Z - sum(2 * e / m for e, m in zip(E, MBAR)), 1e-12)
    Mn_m = matrix / zm
    return (sum(e * m for e, m in zip(E, MBAR)) + matrix * 1.5 * Mn_m) / Y


def vapour_rate(c: Closure, T, Y, Z, rho, f_cut=1.0):
    """S_v [kg/(m³ s)], com ψ limitado a 1/(N* m0)."""
    kc = k_c(c, T)
    ns = n_star(c, T, f_cut)
    Zc = np.minimum(Z, Y / (ns * c.m0))
    return rho * kc * ns ** 2 * c.m0 * Zc


def gas_selectivity(T):
    return 0.06 * np.exp(-(60e3 / R_GAS) * (1.0 / np.asarray(T, float) - 1.0 / 693.15))


def product_slate(T, f_cut=1.0):
    """Frações mássicas do vapor que sai do líquido: gás (C1–C4), óleo (C5–C20), cera (C21+), coque."""
    sG = gas_selectivity(T)
    Nc = rh.boiling_cut_molar_mass(T) * f_cut / 14.027
    oil_frac = (20 ** 2 - 4 ** 2) / max(Nc ** 2 - 4 ** 2, 1.0)
    rest = 1.0 - sG - S_CHAR
    return {"gás": float(sG), "óleo": float(rest * min(oil_frac, 1.0)),
            "cera": float(rest * max(1.0 - oil_frac, 0.0)), "coque": S_CHAR}


# --------------------------------------------------------------- propriedades
def liquid_density(c: Closure, T, Mn_kgmol):
    return rh.density(c.key, T, Mn_kgmol * 1000.0)


def liquid_viscosity(feed_items, states, T, gamma=0.0):
    """Viscosidade da mistura [Pa s]: η0 de cada polímero (Raju acima, cera POLYWAX abaixo) com
    média logarítmica pela massa, e Cross com os parâmetros do componente majoritário."""
    lnmix, wsum = 0.0, 0.0
    main = max(feed_items, key=lambda cw: cw[1])[0]
    for (c, _), st in zip(feed_items, states):
        Y, Z, E = st["Y"], st["Z"], st["E"]
        Mw = Mw_of_state(c, Y, Z, E) * 1000.0
        r = rh.RHEO[c.rheo]
        ent = rh.eta0(r, T, Mw)
        wax = 0.015 * (Mw / 1080.0) ** 1.9 * np.exp((27e3 / R_GAS) * (1.0 / T - 1.0 / 422.15))
        e0 = np.maximum(ent, wax)
        lnmix = lnmix + Y * np.log(np.maximum(e0, 1e-6))
        wsum = wsum + Y
    e0 = np.exp(lnmix / np.maximum(wsum, 1e-12))
    r = rh.RHEO[main.rheo]
    return rh.ETA_FLOOR + e0 / (1.0 + (e0 * gamma / r.tau_star) ** (1.0 - r.n))


# ---------------------------------------------------------------- reator 0-D
@dataclass
class CSTRResult:
    T: float                 # temperatura do fundido [K]
    feed: float              # vazão de alimentação [kg/s]
    vapour: float            # vapor [kg/s]
    drain: float             # purga de resíduo [kg/s]
    duty: float              # calor pela parede [W]
    holdup: float            # massa de líquido [kg]
    states: list = field(default_factory=list)
    Mw: float = 0.0          # massa molar ponderal média [kg/mol]
    Mn: float = 0.0
    eta: float = 0.0         # viscosidade [Pa s]
    rho: float = 0.0
    slate: dict = field(default_factory=dict)

    @property
    def residence_h(self):
        return self.holdup / self.feed / 3600.0


def _coefficients(feed: FeedSpec, T, f_cut=1.0):
    """Constantes de cada polímero que só dependem de T (calculadas uma vez por temperatura)."""
    out = []
    for c, w in feed.items():
        out.append((c, w, float(k_c(c, T)), float(n_star(c, T, f_cut)), [float(r) for r in class_rates(c, T)]))
    return out


def _species_given_s(feed: FeedSpec, T, Sf, s, rho, f_cut=1.0, co=None):
    """Estado de cada polímero para alimentação Sf e vapor total s [kg/(m³ s)] (sistema linear 2×2)."""
    out, Sv = [], []
    d = Sf - s
    for c, w, kc, ns, (r1, r2, r3) in (co or _coefficients(feed, T, f_cut)):
        a = rho * kc * ns ** 2 * c.m0
        Zf = w / c.Mn0
        # Y d + (1+s_C) a Z = Sf w ;  −(ρ k_c/m0) Y + (d + 2ρ k_c N*) Z = Sf Zf
        A11, A12, A21, A22 = d, (1 + S_CHAR) * a, -rho * kc / c.m0, d + 2 * rho * kc * ns
        det = A11 * A22 - A12 * A21
        Y = (Sf * w * A22 - A12 * Sf * Zf) / det
        Z = (A11 * Sf * Zf - A21 * Sf * w) / det
        if Z > Y / (ns * c.m0):                       # ψ no limite: cadeias já no tamanho de corte
            Y = Sf * w / (d + (1 + S_CHAR) * rho * kc * ns)
            Z = Y / (ns * c.m0)
        sv = a * min(Z, Y / (ns * c.m0))
        E1 = Sf * w / (d + rho * r1)
        E2 = rho * r1 * E1 / (d + rho * r2)
        E3 = rho * r2 * E2 / (d + rho * r3)
        out.append({"Y": Y, "Z": Z, "E": [E1, E2, E3]})
        Sv.append(sv)
    return out, Sv


def _steady_species(feed: FeedSpec, T, Sf, rho, f_cut=1.0, co=None):
    """Estado estacionário do tanque perfeitamente misturado: acha o vapor total s = Σ S_v,i(s) em [0, Sf)."""
    co = co or _coefficients(feed, T, f_cut)
    lo, hi = 0.0, Sf * (1 - 1e-9)
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        _, Sv = _species_given_s(feed, T, Sf, mid, rho, f_cut, co)
        if sum(Sv) > mid:
            lo = mid
        else:
            hi = mid
    st, Sv = _species_given_s(feed, T, Sf, 0.5 * (lo + hi), rho, f_cut, co)
    return st, sum(Sv), Sv


def solve_cstr(feed: FeedSpec, volume: float, area: float, h_in: float, T_wall: float | None = None,
               feed_rate: float | None = None, rho: float = 620.0, cp: float = 3240.0, shaft_power: float = 0.0,
               f_cut: float = 1.0) -> CSTRResult:
    """Reator 0-D com nível constante. Modo B (padrão): dada a temperatura da parede, acha a vazão de
    alimentação e a temperatura do fundido; modo A: dada a vazão, acha a temperatura da parede."""
    ash = feed.ash
    items = feed.items()
    dH_mix = sum(w * c.dH for c, w in items)

    def at_T(T):
        # a vazão é tal que alimentação = vapor + purga (sólidos mantidos em SOLIDS_TARGET)
        co = _coefficients(feed, T, f_cut)

        def resid(Sf):
            st, Sv, _ = _steady_species(feed, T, Sf, rho, f_cut, co)
            F = Sf * volume
            V = Sv * volume
            D = (F * ash + S_CHAR * V) / SOLIDS_TARGET
            return F - V - D, st, Sv
        lo, hi = 1e-7, 10.0
        for _ in range(56):              # bisseção em log
            mid = math.sqrt(lo * hi)
            r, _, _ = resid(mid)
            if r > 0:
                hi = mid
            else:
                lo = mid
        Sf = math.sqrt(lo * hi)
        _, st, Sv = resid(Sf)
        F, V = Sf * volume, Sv * volume
        D = (F * ash + S_CHAR * V) / SOLIDS_TARGET
        sens = F * cp * (T - feed.T_feed)
        Q = sens + V * dH_mix - shaft_power
        return F, V, D, Q, st

    if feed_rate is not None:            # modo A
        lo, hi = 600.0, 760.0
        for _ in range(40):
            mid = 0.5 * (lo + hi)
            F = at_T(mid)[0]
            if F > feed_rate:
                hi = mid
            else:
                lo = mid
        T = 0.5 * (lo + hi)
        F, V, D, Q, st = at_T(T)
        T_wall = T + Q / (h_in * area)
    else:                                # modo B
        lo, hi = 600.0, min(T_wall, 780.0)
        for _ in range(40):
            mid = 0.5 * (lo + hi)
            F, V, D, Q, st = at_T(mid)
            if h_in * area * (T_wall - mid) > Q:
                lo = mid
            else:
                hi = mid
        T = 0.5 * (lo + hi)
        F, V, D, Q, st = at_T(T)
    res = CSTRResult(T=T, feed=F, vapour=V, drain=D, duty=Q, holdup=rho * volume, states=st)
    Ysum = sum(s["Y"] for s in st)
    res.Mw = sum(s["Y"] * Mw_of_state(c, s["Y"], s["Z"], s["E"]) for (c, _), s in zip(items, st)) / Ysum
    res.Mn = Ysum / sum(s["Z"] for s in st)
    res.eta = float(liquid_viscosity(items, st, T))
    res.rho = float(sum(s["Y"] * liquid_density(c, T, res.Mn) for (c, _), s in zip(items, st)) / Ysum)
    res.slate = product_slate(T, f_cut)
    res.T_wall = T_wall
    return res


def capacity_curve(feed: FeedSpec, volume, area, h_in, walls_C=(440, 460, 480, 500, 520, 540), **kw):
    """Vazão processada e temperatura do fundido em função da temperatura da parede."""
    return [solve_cstr(feed, volume, area, h_in, T_wall=Tw + 273.15, **kw) for Tw in walls_C]


def nagata_h(rho, N, d, mu, cp, k, T_vessel, mu_wall=None):
    """Coeficiente interno pela correlação de Nagata para fita helicoidal (Re > 1000):
    Nu = h T/k = 0,37 Re^(2/3) Pr^(1/3) (μ/μ_w)^0,14."""
    Re = rho * N * d ** 2 / mu
    Pr = cp * mu / k
    vis = 1.0 if mu_wall is None else (mu / mu_wall) ** 0.14
    return 0.37 * Re ** (2 / 3) * Pr ** (1 / 3) * vis * k / T_vessel, Re, Pr

In [ ]:
%%writefile pyrokit/flow.py
"""Escoamento do fundido no tanque agitado (PyTorch; GPU se houver).

Formulação:
  * Navier–Stokes incompressível no referencial que gira com o agitador (Ω = ω ẑ): o agitador fica
    parado e a parede do vaso gira com velocidade −Ω × r. Num vaso sem chicanas, com paredes de
    revolução, o problema é estacionário nesse referencial; Coriolis −2Ω × u explícito, centrífuga
    absorvida na pressão modificada (densidade constante);
  * viscosidade variável μ(γ̇, …) de fluido newtoniano generalizado, na forma ∇·(μ∇u) + (∇u)ᵀ·∇μ,
    mais a viscosidade turbulenta do modelo RANS quando houver;
  * malha MAC uniforme; advecção upwind de 3ª ordem; RK2 (SSP) com projeção em cada estágio;
  * fronteiras imersas por forçamento direto (fração sólida suavizada em 1 célula) para o vaso e o
    agitador, aplicado antes e depois de cada projeção; o torque no agitador vem do impulso total do
    forçamento (que inclui a pressão) e a potência é P = ω·T;
  * Poisson direto pela diagonalização dos operadores 1D (Neumann em todas as faces).

Fundo em z = 0 (parede que gira, ou deslizamento no caso de verificação) e topo em z = H_L com
deslizamento (superfície livre plana).
"""
from __future__ import annotations

import math
import time
from dataclasses import dataclass

import numpy as np
import scipy.linalg
import torch

from .geometry import Tank
from .ops import avg, cgrad, diff, extend_into_solid, pad, sl, to_faces


@dataclass
class FlowConfig:
    n_diam: int = 64             # células ao longo do diâmetro do vaso
    rho: float = 750.0           # massa específica do fundido [kg/m³]
    cfl: float = 0.45
    dt_max: float = 0.02
    bottom: str = "wall"         # "wall" (parede que gira) ou "slip" (verificação)
    upwind: float = 0.0          # mistura com upwind de 1ª ordem (0 = 3ª ordem pura)
    mu_max: float = 1e9          # teto de viscosidade [Pa s] (estabilidade do passo explícito)
    spin0: float = 0.0           # partida: núcleo girando a spin0·Ω no referencial fixo (0 = repouso)
    vessel_ibm: str = "diffuse"  # "diffuse" (sem deslizamento, laminar) ou "wallfn" (lei de parede, RANS)
    advection: str = "conservative"  # "conservative" (fluxos, conserva momento) ou "advective" (u·∇u)
    device: str = "auto"
    dtype: str = "float32"


class TankFlow:
    def __init__(self, tank: Tank, cfg: FlowConfig, mu_fn=None):
        """mu_fn(gamma_dot) → μ [Pa s] nos centros (tensor); None = newtoniano com μ = 1 Pa s."""
        self.tank, self.cfg = tank, cfg
        dev = cfg.device
        if dev == "auto":
            dev = "cuda" if torch.cuda.is_available() else "cpu"
        self.dev = torch.device(dev)
        self.ft = getattr(torch, cfg.dtype)
        t = lambda a: torch.as_tensor(np.asarray(a, dtype=np.float64), dtype=self.ft, device=self.dev)  # noqa: E731
        self._t = t
        self.mu_fn = mu_fn if mu_fn is not None else (lambda g: torch.ones_like(g))
        self.rho = cfg.rho
        self.omega = tank.omega

        # ---- malha uniforme no plano, espaçamento próprio em z (H_L/nz)
        h = tank.T / cfg.n_diam
        n_xy = cfg.n_diam + 4                                   # duas células de parede de cada lado
        half = 0.5 * n_xy * h
        self.h = h
        self.nx = self.ny = n_xy
        self.nz = max(4, int(round(tank.H_L / h)))
        self.hz = tank.H_L / self.nz
        self.H = (h, h, self.hz)
        self.xf = -half + h * np.arange(n_xy + 1)
        self.yf = self.xf.copy()
        self.zf = self.hz * np.arange(self.nz + 1)
        self.xc, self.yc, self.zc = [0.5 * (f[1:] + f[:-1]) for f in (self.xf, self.yf, self.zf)]
        self.vol = h * h * self.hz
        self.inv_d2 = 2.0 / h ** 2 + 1.0 / self.hz ** 2

        # ---- geometria nos centros e nas faces
        def grid(xs, ys, zs):
            X, Y, Z = np.meshgrid(xs, ys, zs, indexing="ij")
            return np.stack([X, Y, Z], -1)

        self.P_c = grid(self.xc, self.yc, self.zc)
        P_u, P_v, P_w = grid(self.xf, self.yc, self.zc), grid(self.xc, self.yf, self.zc), grid(self.xc, self.yc, self.zf)
        frac = lambda s: np.clip(0.5 + s / h, 0.0, 1.0)  # noqa: E731   fração sólida (SDF > 0 no sólido)
        self.chi_v = [t(frac(tank.vessel_sdf(P))) for P in (P_u, P_v, P_w)]      # parede do vaso
        self.chi_i = [t(frac(tank.impeller_sdf(P))) for P in (P_u, P_v, P_w)]    # agitador
        w = self.omega
        self.uwall = [t(w * P_u[..., 1]), t(-w * P_v[..., 0]), torch.zeros_like(self.chi_v[2])]  # −Ω×r
        self.vtarget = list(self.uwall)
        self.vghost = None             # faces do vaso profundo (sem troca de momento por advecção)
        self.wallfn = cfg.vessel_ibm == "wallfn"
        if self.wallfn:
            # Lei de parede no vaso (RANS). Numa malha cartesiana a parede cilíndrica vira uma escada; no
            # referencial do agitador ela gira, e qualquer forçamento sem deslizamento (suavizado ou nítido)
            # faz os degraus empurrarem o fluido como dentes de engrenagem, com um torque que não diminui
            # com o refinamento (verificado em 32, 48 e 64 células). Aqui:
            #  * além de Δ/2 dentro da parede, a região fantasma gira como corpo rígido com a velocidade
            #    angular média (por altura) do anel de fluido junto à parede: a escada entre os dois fica
            #    "invisível", porque os dois lados andam juntos (sem velocidade radial: impermeável);
            #  * a tensão τ_w = ρu_τ² (lei de Spalding) entra como força distribuída nesse anel (δ = 1/Δ
            #    numa faixa de uma célula), com a área verdadeira do cilindro.
            s_f = [tank.vessel_sdf(P) for P in (P_u, P_v, P_w)]
            self.chi_v = [torch.zeros_like(c) for c in self.chi_v]
            self.ghost_face = [t((sf > 0.5 * h).astype(float)) > 0.5 for sf in s_f]
            s_c = tank.vessel_sdf(self.P_c)
            self.ring = t(((s_c > -h) & (s_c <= 0.0)).astype(float))      # anel de fluido junto à parede
            # δ da interface [1/m]: faixa de uma célula do lado do fluido, para que a força caia em faces
            # que não são reimpostas (as faces além de Δ/2 da parede são fantasmas)
            band = (s_c > -h) & (s_c <= 0.0)
            per_layer = band[..., 0].sum() * h * h                     # volume da faixa por unidade de altura
            self.delta_w = t(np.where(band, 2 * math.pi * tank.R / max(per_layer, 1e-30), 0.0))  # ∫δ = área real
            self.beyond_wall = t((s_c > 0.5 * h).astype(float))              # além da parede: μ ≈ 0
            Xc, Yc = self.P_c[..., 0], self.P_c[..., 1]
            rc = np.hypot(Xc, Yc) + 1e-12
            self.n_c = (t(Xc / rc), t(Yc / rc))
            self.uwall_c = (t(w * Yc), t(-w * Xc))
            self.tau_w = None
        self.chi_i = [ci * (1 - cv) for ci, cv in zip(self.chi_i, self.chi_v)]
        # casca da parede do vaso (onde há fluido por perto): só ela entra no torque da parede
        self.shell = [t((tank.vessel_sdf(P) < 1.5 * h).astype(float)) for P in (P_u, P_v)]
        self.xu, self.yu = t(P_u[..., 0]), t(P_u[..., 1])
        self.xv, self.yv = t(P_v[..., 0]), t(P_v[..., 1])
        self.fluid = t(1.0 - np.maximum(frac(tank.vessel_sdf(self.P_c)), frac(tank.impeller_sdf(self.P_c))))
        self.wall_dist = t(tank.wall_distance(self.P_c))
        self.r_c = t(np.hypot(self.P_c[..., 0], self.P_c[..., 1]))
        # velocidade da parede do fundo nas faces u e v da primeira camada
        self.ubot = self.uwall[0][:, :, :1]
        self.vbot = self.uwall[1][:, :, :1]

        # ---- campos (velocidade relativa ao agitador); partida: líquido em repouso no referencial fixo
        self.u = self.uwall[0].clone() * (1 - self.chi_i[0]) * (1 - cfg.spin0)
        self.v = self.uwall[1].clone() * (1 - self.chi_i[1]) * (1 - cfg.spin0)
        self.w = torch.zeros_like(self.chi_v[2])
        self.P = torch.zeros(self.nx, self.ny, self.nz, dtype=self.ft, device=self.dev)   # p/ρ
        self.mu = torch.full_like(self.P, float(self.mu_fn(torch.zeros(1, dtype=self.ft, device=self.dev))[0]))
        self.mu_t = torch.zeros_like(self.P)
        self.gamma = torch.zeros_like(self.P)
        self.body = None            # forças de corpo extras (ex.: empuxo), lista de 3 tensores nas faces
        self.turb = None            # modelo RANS (turbulence.SST), opcional
        self.mu_bottom = None       # viscosidade da face-fantasma do fundo (função de parede)

        self._poisson_setup()
        self.time, self.step_n, self.dt, self.wall = 0.0, 0, cfg.dt_max, 0.0
        self.torque_imp = 0.0
        self.torque_wall = 0.0
        self.history = {k: [] for k in ("t", "torque", "torque_wall", "dt", "ke")}

    # -------------------------------------------------------------- Poisson
    def _lap1d(self, n, h):
        B = np.zeros((n, n))
        i = np.arange(n - 1)
        B[i, i + 1] = B[i + 1, i] = 1.0 / h ** 2
        d = np.full(n, -2.0 / h ** 2)
        d[0] += 1.0 / h ** 2
        d[-1] += 1.0 / h ** 2                     # Neumann nas duas pontas
        B[np.arange(n), np.arange(n)] = d
        lam, V = scipy.linalg.eigh(B)
        return self._t(lam), self._t(V)

    def _poisson_setup(self):
        (lx, Vx), (ly, Vy), (lz, Vz) = (self._lap1d(self.nx, self.h), self._lap1d(self.ny, self.h),
                                        self._lap1d(self.nz, self.hz))
        self.V = (Vx, Vy, Vz)
        lam = lx.view(-1, 1, 1) + ly.view(1, -1, 1) + lz.view(1, 1, -1)
        k0 = tuple(int(torch.argmin(l.abs())) for l in (lx, ly, lz))   # modo constante (autovalor nulo)
        lam[k0] = 1.0                              # pressão definida a menos de uma constante
        self.lam = lam
        self.zero_mode = torch.ones_like(lam)
        self.zero_mode[k0] = 0.0

    def poisson(self, rhs):
        Vx, Vy, Vz = self.V
        r = torch.einsum("ia,ijk->ajk", Vx, rhs)
        r = torch.einsum("jb,ajk->abk", Vy, r)
        r = torch.einsum("kc,abk->abc", Vz, r)
        r = r / self.lam * self.zero_mode
        r = torch.einsum("ia,abc->ibc", Vx, r)
        r = torch.einsum("jb,ibc->ijc", Vy, r)
        return torch.einsum("kc,ijc->ijk", Vz, r)

    def divergence(self, u, v, w):
        return diff(u, 0) / self.h + diff(v, 1) / self.h + diff(w, 2) / self.hz

    def grad_face(self, Pc, axis):
        """Gradiente nas faces internas; nas faces de borda (Neumann) é zero."""
        g = diff(Pc, axis) / self.H[axis]
        z = torch.zeros_like(sl(Pc, axis, 0, 1))
        return torch.cat([z, g, z], dim=axis)

    def project(self, u, v, w, coef):
        phi = self.poisson(self.divergence(u, v, w) / coef)
        return (u - coef * self.grad_face(phi, 0), v - coef * self.grad_face(phi, 1),
                w - coef * self.grad_face(phi, 2), phi)

    # ---------------------------------------------------------- contornos
    def _pad_comp(self, q, comp, axis, n=2):
        """Células-fantasma de uma componente de velocidade ao longo de axis.

        Em x e y o domínio termina dentro da parede (sólido): replica. Em z: fundo com parede que gira
        (fantasma = 2·u_parede − u) ou deslizamento; topo com deslizamento (espelho); w é nulo nas faces
        de fundo e topo (espelho antissimétrico)."""
        if axis != 2:
            return pad(q, axis, n)
        if comp == 2:
            lo = [-sl(q, 2, k, k + 1) for k in range(n, 0, -1)]
            hi = [-sl(q, 2, -1 - k, -k) for k in range(1, n + 1)]
            return torch.cat(lo + [q] + hi, dim=2)
        if self.cfg.bottom == "wall":
            ub = self.ubot if comp == 0 else self.vbot
            lo = [2 * ub - sl(q, 2, k - 1, k) for k in range(n, 0, -1)]
        else:
            lo = [sl(q, 2, k - 1, k) for k in range(n, 0, -1)]
        hi = [sl(q, 2, q.shape[2] - k, q.shape[2] - k + 1) for k in range(1, n + 1)]
        return torch.cat(lo + [q] + hi, dim=2)

    # ---------------------------------------------------------- viscosidade
    def strain(self, u, v, w):
        """Gradiente de velocidade nos centros g[i][j] = ∂u_i/∂x_j e taxa de cisalhamento γ̇ = √(2 S:S)."""
        uc, vc, wc = avg(u, 0), avg(v, 1), avg(w, 2)
        g = [[None] * 3 for _ in range(3)]
        for i, (q, qc) in enumerate(((u, uc), (v, vc), (w, wc))):
            for j in range(3):
                g[i][j] = diff(q, j) / self.H[j] if j == i else cgrad(qc, j, self.H[j])
        SS = 0.0
        for i in range(3):
            for j in range(i, 3):
                Sij = 0.5 * (g[i][j] + g[j][i])
                SS = SS + (1.0 if i == j else 2.0) * Sij * Sij
        return g, torch.sqrt(2.0 * SS + 1e-30)

    def update_viscosity(self, relax: float = 1.0):
        g, gam = self.strain(self.u, self.v, self.w)
        self.gamma = gam
        mu = extend_into_solid(self.mu_fn(gam).clamp(max=self.cfg.mu_max), self.fluid > 0.5)
        self.mu = mu if relax >= 1.0 else relax * mu + (1 - relax) * self.mu
        return g

    # -------------------------------------------------------------- momento
    def _advect_conservative(self, comp, q, vel):
        """∇·(u q) em forma de fluxo nos volumes de controle da componente comp, com interpolação upwind de
        3ª ordem nas faces. Pelas faces impermeáveis (velocidade normal nula) não passa momento: num vaso
        fechado o momento só é trocado com as paredes pela tensão e pela pressão."""
        beta = self.cfg.upwind
        out = 0.0
        for a in range(3):
            qp = self._pad_comp(q, comp, a, 2)
            nf = q.shape[a] + 1
            l2, l1, r1, r2 = (sl(qp, a, k, k + nf) for k in range(4))
            U = 0.5 * (l1 + r1) if a == comp else to_faces(vel[a], comp)
            qf = torch.where(U >= 0, (-l2 + 5 * l1 + 2 * r1) / 6.0, (2 * l1 + 5 * r1 - r2) / 6.0)
            if beta > 0:
                qf = (1 - beta) * qf + beta * torch.where(U >= 0, l1, r1)
            F = U * qf
            if self.vghost is not None and comp < 2:
                # vaso profundo (reimposto a cada estágio): nenhum fluxo de momento entra ou sai dele
                gp = pad(self.vghost[comp].to(q.dtype), a)
                F = F * (1 - torch.maximum(sl(gp, a, 0, nf), sl(gp, a, 1, nf + 1)))
            out = out + (sl(F, a, 1, None) - sl(F, a, 0, -1)) / self.H[a]
        return out

    def _adv_visc(self, comp, q, adv, mu_q, mu_bot=None, qv=None, vel=None):
        """Upwind de 3ª ordem e ∇·(μ∇q)/ρ para a componente comp (nas suas faces); mu_bot: viscosidade
        da face-fantasma do fundo (função de parede), no lugar da réplica da primeira camada; qv: campo
        usado no termo viscoso (com a velocidade da parede nas faces sólidas, na parede nítida)."""
        beta = self.cfg.upwind
        A, Vs = 0.0, 0.0
        qv = q if qv is None else qv
        for axis, a in enumerate(adv):
            h = self.H[axis]
            qp = self._pad_comp(q, comp, axis, 2)
            qvp = qp if qv is q else self._pad_comp(qv, comp, axis, 1)
            n = q.shape[axis]
            qm2, qm1 = sl(qp, axis, 0, n), sl(qp, axis, 1, n + 1)
            qp1, qp2 = sl(qp, axis, 3, n + 3), sl(qp, axis, 4, n + 4)
            dpos = (2 * qp1 + 3 * q - 6 * qm1 + qm2) / (6 * h)
            dneg = (-qp2 + 6 * qp1 - 3 * q - 2 * qm1) / (6 * h)
            d3 = torch.where(a >= 0, dpos, dneg)
            if beta > 0:
                d1 = torch.where(a >= 0, (q - qm1) / h, (qp1 - q) / h)
                d3 = (1 - beta) * d3 + beta * d1
            if vel is None:
                A = A + a * d3
            mp = pad(mu_q, axis)
            if axis == 2 and mu_bot is not None:
                mp = torch.cat([mu_bot, sl(mp, 2, 1, None)], dim=2)
            mu_p = 0.5 * (mu_q + sl(mp, axis, 2, None))
            mu_m = 0.5 * (mu_q + sl(mp, axis, 0, -2))
            if qv is q:
                Vs = Vs + (mu_p * (qp1 - q) - mu_m * (q - qm1)) / (h * h)
            else:
                k0 = 1 if qvp is not qp else 2
                vm1, vp1 = sl(qvp, axis, k0 - 1, n + k0 - 1), sl(qvp, axis, k0 + 1, n + k0 + 1)
                Vs = Vs + (mu_p * (vp1 - qv) - mu_m * (qv - vm1)) / (h * h)
        if vel is not None:
            A = self._advect_conservative(comp, q, vel)
        return A - Vs / self.rho

    def momentum_rhs(self, u, v, w, g=None):
        """F tal que ∂u/∂t = −F − ∇P (sem a pressão)."""
        mu = self.mu + self.mu_t
        mb = self.mu_bottom
        if self.wallfn:                       # além da parede: sem atrito (a tensão é a da lei de parede)
            mu = torch.where(self.beyond_wall > 0, 1e-3 * self.mu, mu)
            if mb is not None:
                mb = torch.where(self.beyond_wall[:, :, :1] > 0, 1e-3 * mb, mb)
        uc, vc, wc = avg(u, 0), avg(v, 1), avg(w, 2)
        vel = (u, v, w) if self.cfg.advection == "conservative" else None
        Fu = self._adv_visc(0, u, (u, to_faces(vc, 0), to_faces(wc, 0)), to_faces(mu, 0),
                            None if mb is None else to_faces(mb, 0), vel=vel)
        Fv = self._adv_visc(1, v, (to_faces(uc, 1), v, to_faces(wc, 1)), to_faces(mu, 1),
                            None if mb is None else to_faces(mb, 1), vel=vel)
        Fw = self._adv_visc(2, w, (to_faces(uc, 2), to_faces(vc, 2), w), to_faces(mu, 2), vel=vel)
        # termo transposto (∇u)ᵀ·∇μ: importante onde a viscosidade varia muito (reologia, turbulência)
        if g is not None:
            dmu = [cgrad(mu, j, self.H[j]) for j in range(3)]
            Tt = [sum(dmu[j] * g[j][i] for j in range(3)) for i in range(3)]
            Fu = Fu - to_faces(Tt[0], 0) / self.rho
            Fv = Fv - to_faces(Tt[1], 1) / self.rho
            Fw = Fw - to_faces(Tt[2], 2) / self.rho
        # Coriolis: −2Ω×u = (2ωv, −2ωu, 0)
        om = self.omega
        Fu = Fu - 2 * om * to_faces(vc, 0)
        Fv = Fv + 2 * om * to_faces(uc, 1)
        if self.body is not None:
            Fu, Fv, Fw = Fu - self.body[0], Fv - self.body[1], Fw - self.body[2]
        if self.wallfn and self.tau_w is not None:
            fx, fy, fz = self.tau_faces
            Fu, Fv, Fw = Fu - fx, Fv - fy, Fw - fz
        return Fu, Fv, Fw

    def wall_shear(self):
        """Lei de parede no vaso: u_τ de Spalding com a velocidade tangencial relativa à parede na faixa
        da interface (y = Δ) e a aceleração −u_τ² δ t̂ [m/s²] nos centros (congelada durante o passo)."""
        from .turbulence import spalding_utau
        uc, vc, wc = avg(self.u, 0), avg(self.v, 1), avg(self.w, 2)
        nx, ny = self.n_c
        ux, uy = uc - self.uwall_c[0], vc - self.uwall_c[1]
        un = ux * nx + uy * ny
        tx, ty, tz = ux - un * nx, uy - un * ny, wc
        Ut = torch.sqrt(tx * tx + ty * ty + tz * tz).clamp(min=1e-9)
        nu = (self.mu / self.rho).clamp(min=1e-9)
        ut = spalding_utau(Ut, torch.full_like(Ut, self.h), nu)
        a = -(ut ** 2) * self.delta_w / Ut
        self.u_tau_w = torch.where(self.delta_w > 0, ut, torch.zeros_like(ut))
        self.tau_w = (a * tx, a * ty, a * tz)
        # nas faces (as fantasmas são reimpostas, então a força ali se perderia): renormaliza o torque de
        # cada camada para o da tensão modelada, para que a força aplicada seja a da área verdadeira
        X, Y = self.n_c[0] * self.r_c, self.n_c[1] * self.r_c
        keep = [(~g).to(self.ft) for g in self.ghost_face]
        fx, fy, fz = (to_faces(c, k) * keep[k] for k, c in enumerate(self.tau_w))
        t_cell = (X * self.tau_w[1] - Y * self.tau_w[0]).sum((0, 1))
        t_face = (self.xv * fy).sum((0, 1)) - (self.yu * fx).sum((0, 1))
        sc = torch.where(t_face.abs() > 1e-30, t_cell / t_face, torch.ones_like(t_cell)).clamp(0.5, 2.0).view(1, 1, -1)
        self.tau_faces = (fx * sc, fy * sc, fz)
        return self.tau_w

    def _slip_ghost(self, u, v, w):
        """Região além da parede: gira como corpo rígido com a velocidade angular média (por altura) do
        primeiro anel de fluido, sem componente radial nem axial. Um campo de rotação rígida tem divergência
        nula, e a troca de momento por advecção com o líquido junto à parede fica neutra em média."""
        uc, vc = avg(u, 0), avg(v, 1)
        X, Y = self.n_c[0] * self.r_c, self.n_c[1] * self.r_c
        om = (X * vc - Y * uc) / self.r_c.clamp(min=1e-9) ** 2
        wz = (om * self.ring).sum((0, 1)) / self.ring.sum((0, 1)).clamp(min=1.0)       # [nz]
        wz = wz.view(1, 1, -1)
        u = torch.where(self.ghost_face[0], -wz * self.yu, u)
        v = torch.where(self.ghost_face[1], wz * self.xv, v)
        w = torch.where(self.ghost_face[2], torch.zeros_like(w), w)
        return u, v, w

    def _force(self, q, comp):
        """Forçamento direto: impõe a velocidade do sólido; devolve (q, impulso do agitador, da parede)."""
        ci, cv = self.chi_i[comp], self.chi_v[comp]
        di = ci * (0.0 - q)
        dv = cv * (self.vtarget[comp] - q)
        return q + di + dv, di, dv

    def _apply_bc(self, w):
        w = torch.cat([torch.zeros_like(w[:, :, :1]), w[:, :, 1:-1], torch.zeros_like(w[:, :, :1])], 2)
        return w

    def compute_dt(self):
        uc, vc, wc = avg(self.u, 0).abs(), avg(self.v, 1).abs(), avg(self.w, 2).abs()
        rate = (uc / self.h + vc / self.h + wc / self.hz).max().item()
        dt = self.cfg.cfl / max(rate, 1e-9)
        nu_max = (self.mu + self.mu_t).max().item() / self.rho
        if nu_max > 0:
            dt = min(dt, 0.45 / (2.0 * nu_max * self.inv_d2))
        dt = min(dt, 0.45 / max(self.omega, 1e-9), self.cfg.dt_max)
        return dt

    def _torques(self, du, dv, dt, chi_sel):
        """Torque (em z) aplicado ao fluido pelo impulso (du, dv) nas faces [N m]."""
        return self.rho * self.vol * float(((self.xv * dv).sum() - (self.yu * du).sum()).item()) / dt

    def step(self, update_mu: bool = True):
        """Passo RK2 (SSP). Pressão não incremental; forçamento antes e depois de cada projeção (o
        impulso total no sólido inclui a pressão e dá o torque)."""
        t0 = time.perf_counter()
        g = self.update_viscosity() if update_mu else self.strain(self.u, self.v, self.w)[0]
        if self.turb is not None:
            if not update_mu:
                self.gamma = self.strain(self.u, self.v, self.w)[1]
            self.turb.update(self.dt, g, self.gamma)
        dt = self.compute_dt()
        self.dt = dt
        if self.wallfn:
            self.wall_shear()
        u0, v0, w0 = self.u, self.v, self.w

        def stage(uu, vv, ww, coef):
            if self.wallfn:
                uu, vv, ww = self._slip_ghost(uu, vv, ww)
            uu, ai_u, av_u = self._force(uu, 0)
            vv, ai_v, av_v = self._force(vv, 1)
            ww, _, _ = self._force(ww, 2)
            ww = self._apply_bc(ww)
            uu, vv, ww, phi = self.project(uu, vv, ww, coef)
            if self.wallfn:
                uu, vv, ww = self._slip_ghost(uu, vv, ww)
            uu, bi_u, bv_u = self._force(uu, 0)              # reforço: o sólido fica exatamente na sua velocidade
            vv, bi_v, bv_v = self._force(vv, 1)
            ww, _, _ = self._force(ww, 2)
            ww = self._apply_bc(ww)
            return uu, vv, ww, phi, (ai_u + bi_u, ai_v + bi_v, av_u + bv_u, av_v + bv_v)

        # ---- estágio 1
        Fu, Fv, Fw = self.momentum_rhs(u0, v0, w0, g)
        u1, v1, w1, _, I1 = stage(u0 - dt * Fu, v0 - dt * Fv, w0 - dt * Fw, dt)
        # ---- estágio 2
        g1 = self.strain(u1, v1, w1)[0]
        Fu, Fv, Fw = self.momentum_rhs(u1, v1, w1, g1)
        u2, v2, w2, phi2, I2 = stage(0.5 * (u0 + u1 - dt * Fu), 0.5 * (v0 + v1 - dt * Fv),
                                     0.5 * (w0 + w1 - dt * Fw), 0.5 * dt)
        self.u, self.v, self.w = u2, v2, w2
        self.P = phi2
        # ---- torques: impulso efetivo no passo (o estágio 1 entra com peso 1/2)
        Iu_i, Iv_i, Iu_v, Iv_v = [0.5 * a + b for a, b in zip(I1, I2)]
        self.torque_imp = self._torques(Iu_i, Iv_i, dt, "i")
        if self.wallfn:
            # lei de parede: o torque da parede é o da tensão modelada (a remoção da componente normal
            # é radial e não exerce torque)
            ax, ay, _ = self.tau_w
            X, Y = self._t(self.P_c[..., 0]), self._t(self.P_c[..., 1])
            self.torque_wall = self.rho * self.vol * float((X * ay - Y * ax).sum())
            if self.cfg.bottom == "wall":
                self.torque_wall += self.bottom_torque()
        else:
            self.torque_wall = self._torques(Iu_v * self.shell[0], Iv_v * self.shell[1], dt, "v")
            if self.cfg.bottom == "wall":
                self.torque_wall += self.bottom_torque()
        if not torch.isfinite(self.u).all():
            raise FloatingPointError(f"o escoamento divergiu em t = {self.time:.3f} s")
        self.time += dt
        self.step_n += 1
        if self.dev.type == "cuda":
            torch.cuda.synchronize()
        self.wall += time.perf_counter() - t0
        return dt

    def angular_momentum(self):
        """Momento angular do fluido em torno do eixo, no referencial do laboratório [kg m²/s]."""
        ul, vl, _ = self.lab_velocity_centers()
        X, Y = self._t(self.P_c[..., 0]), self._t(self.P_c[..., 1])
        return self.rho * float(((X * vl - Y * ul) * self.fluid).sum()) * self.vol

    def bottom_torque(self):
        """Torque da parede do fundo sobre o fluido (tensão viscosa na primeira camada)."""
        mu_c = (self.mu + self.mu_t)[:, :, :1]
        if self.mu_bottom is not None:                       # mesma média da face usada no momento
            mu_c = 0.5 * (mu_c + self.mu_bottom)
        if self.wallfn:
            mu_c = mu_c * (self.fluid[:, :, :1] > 0.5)
        mu_u = to_faces(mu_c, 0)
        mu_v = to_faces(mu_c, 1)
        tu = mu_u * (self.ubot - self.u[:, :, :1]) / (0.5 * self.hz) * (1 - self.chi_v[0][:, :, :1])
        tv = mu_v * (self.vbot - self.v[:, :, :1]) / (0.5 * self.hz) * (1 - self.chi_v[1][:, :, :1])
        A = self.h * self.h
        return float(((self.xv[:, :, :1] * tv).sum() - (self.yu[:, :, :1] * tu).sum()).item()) * A

    # ------------------------------------------------------------ diagnósticos
    def record(self):
        ke = 0.5 * float(((avg(self.u, 0) ** 2 + avg(self.v, 1) ** 2 + avg(self.w, 2) ** 2) * self.fluid).sum()) * self.vol
        for k, val in (("t", self.time), ("torque", self.torque_imp), ("torque_wall", self.torque_wall),
                       ("dt", self.dt), ("ke", ke)):
            self.history[k].append(val)

    def power(self, n_avg: int = 200):
        """Potência média do agitador nas últimas n_avg medições [W] (P = ω T)."""
        T = np.asarray(self.history["torque"][-n_avg:])
        return self.omega * float(T.mean()) if len(T) else 0.0

    def lab_velocity_centers(self):
        """Velocidade no referencial do laboratório nos centros: u_lab = u_rel + Ω × r."""
        uc, vc, wc = avg(self.u, 0), avg(self.v, 1), avg(self.w, 2)
        X, Y = self._t(self.P_c[..., 0]), self._t(self.P_c[..., 1])
        om = self.omega
        return uc - om * Y, vc + om * X, wc

    def summary(self) -> str:
        n = self.nx * self.ny * self.nz
        return (f"malha {self.nx}×{self.ny}×{self.nz} = {n / 1e6:.2f} M células, Δ = {1e3 * self.h:.1f} mm, "
                f"{self.dev.type.upper()}")

In [ ]:
%%writefile pyrokit/turbulence.py
"""Turbulência RANS (k-ω SST de Menter, 2003) e funções de parede para fronteiras imersas.

Por que RANS no fundido: depois de craqueado, o conteúdo do reator é uma cera com viscosidade de
~1–10 mPa s a 400–440 °C, e o Reynolds do agitador fica em 10⁴–10⁶ (turbulento); só a pluma de
alimentação, ainda com o polímero longo, é viscosa. O SST se laminariza sozinho onde ν_t ≪ ν.

Paredes (vaso, fundo e agitador) ficam em células de uma malha cartesiana, com distância e normal dadas
pelas funções de distância analíticas. Na primeira célula de fluido:
  * lei de Spalding (válida para todo y⁺) → u_τ a partir da velocidade tangencial relativa à parede;
  * a tensão de parede entra por uma viscosidade efetiva nas faces entre a célula e o sólido
    (μ_w = ρ u_τ² Δ / |u_t|), que substitui o gradiente não resolvido;
  * ω = √(ω_vis² + ω_log²) (tratamento automático de parede) e produção de k pela lei log;
  * calor: função de parede de Kader (1981), T⁺(y⁺, Pr), h = ρ c_p u_τ / T⁺.
"""
from __future__ import annotations

import math

import numpy as np
import torch

from .ops import avg, cgrad, diff, extend_into_solid, pad, sl, to_faces

KAPPA, B_LOG = 0.41, 5.2
BETA_STAR, A1 = 0.09, 0.31
SIG_K1, SIG_K2, SIG_W1, SIG_W2 = 0.85, 1.0, 0.5, 0.856
BETA1, BETA2 = 0.075, 0.0828
ALPHA1, ALPHA2 = 5.0 / 9.0, 0.44


# --------------------------------------------------------------- leis de parede
def spalding_utau(U, y, nu, iters: int = 12):
    """u_τ pela lei de Spalding: y⁺ = u⁺ + e^{−κB}[e^{κu⁺} − 1 − κu⁺ − (κu⁺)²/2 − (κu⁺)³/6]."""
    U = U.clamp(min=1e-9)
    ut = torch.sqrt(nu * U / y).clamp(min=1e-9)                 # chute viscoso
    ut = torch.maximum(ut, KAPPA * U / torch.log(1 + 9.0 * (y * U / nu).clamp(min=1.0)))
    ekb = math.exp(-KAPPA * B_LOG)
    for _ in range(iters):
        up = U / ut
        ku = (KAPPA * up).clamp(max=60.0)
        f_up = up + ekb * (torch.exp(ku) - 1 - ku - ku ** 2 / 2 - ku ** 3 / 6)
        f = y * ut / nu - f_up
        dfu = 1 + ekb * KAPPA * (torch.exp(ku) - 1 - ku - ku ** 2 / 2)        # d(f_up)/du⁺
        df = y / nu + dfu * U / ut ** 2
        ut = (ut - f / df).clamp(min=1e-9)
    return ut


def kader_Tplus(yplus, Pr):
    """Temperatura adimensional de Kader (1981), válida da subcamada viscosa à região log."""
    beta = (3.85 * Pr ** (1.0 / 3.0) - 1.3) ** 2 + 2.12 * math.log(Pr) if not torch.is_tensor(Pr) else \
        (3.85 * Pr ** (1.0 / 3.0) - 1.3) ** 2 + 2.12 * torch.log(Pr)
    G = 0.01 * (Pr * yplus) ** 4 / (1 + 5 * Pr ** 3 * yplus)
    return Pr * yplus * torch.exp(-G) + (2.12 * torch.log(1 + yplus) + beta) * torch.exp(-1.0 / G.clamp(min=1e-12))


# ------------------------------------------------------- correção de rotação/curvatura
def rotation_curvature_factor(f, g, w_turb, frame_omega, cr1=1.0, cr2=2.0, cr3=1.0):
    """f_r1 de Spalart–Shur na forma de Smirnov & Menter (2009) para o SST, no referencial que gira com
    Ω = frame_omega ẑ. Suprime a produção de k em escoamentos com rotação estabilizante (o núcleo de um
    tanque sem chicanas gira quase como corpo rígido) e a aumenta onde a curvatura desestabiliza."""
    S = [[0.5 * (g[i][j] + g[j][i]) for j in range(3)] for i in range(3)]
    W = [[0.5 * (g[i][j] - g[j][i]) for j in range(3)] for i in range(3)]
    om = frame_omega
    # vorticidade absoluta: Ω_ij + ε_mji Ω_m  (Ω_m = ω δ_m3) → Ω_12 −= ω, Ω_21 += ω
    W[0][1] = W[0][1] - om
    W[1][0] = W[1][0] + om
    S2 = 2 * sum(S[i][j] ** 2 for i in range(3) for j in range(3))
    W2 = 2 * sum(W[i][j] ** 2 for i in range(3) for j in range(3))
    Sm, Wm = torch.sqrt(S2 + 1e-30), torch.sqrt(W2 + 1e-30)
    rstar = Sm / Wm
    # derivada material do tensor de deformação (escoamento estacionário no referencial girante)
    uc, vc, wc = avg(f.u, 0), avg(f.v, 1), avg(f.w, 2)
    vel = (uc, vc, wc)
    DS = [[None] * 3 for _ in range(3)]
    for i in range(3):
        for j in range(i, 3):
            d = sum(vel[a] * cgrad(S[i][j], a, f.H[a]) for a in range(3))
            DS[i][j] = DS[j][i] = d
    # termo de rotação do referencial: (ε_imn S_jn + ε_jmn S_in) Ω_m, só com m = 3
    def rot(i, j):
        t = 0.0
        if i == 0:
            t = t - S[j][1]
        if i == 1:
            t = t + S[j][0]
        if j == 0:
            t = t - S[i][1]
        if j == 1:
            t = t + S[i][0]
        return om * t
    num = 0.0
    for i in range(3):
        for j in range(3):
            Aij = DS[i][j] + rot(i, j)
            for k in range(3):
                num = num + 2 * W[i][k] * S[j][k] * Aij
    D2 = torch.maximum(S2, 0.09 * w_turb ** 2)
    rtil = num / (Wm * D2 ** 1.5 + 1e-30)
    fr = (1 + cr1) * 2 * rstar / (1 + rstar) * (1 - cr3 * torch.atan(cr2 * rtil)) - cr1
    return fr.clamp(0.0, 1.25)


# ------------------------------------------------------------------------ SST
class SST:
    def __init__(self, flow, intensity: float = 0.05, L_frac: float = 0.07, curvature: bool = True):
        f = self.f = flow
        tank = flow.tank
        t = flow._t
        self.rho = flow.rho
        fl = flow.fluid > 0.5
        self.fl = fl
        self.flf = fl.to(flow.ft)
        P = flow.P_c
        # ---- paredes: distância, normal e velocidade da parede mais próxima
        d_side = -tank.vessel_sdf(P)
        d_bot = P[..., 2] if flow.cfg.bottom == "wall" else np.full(P.shape[:-1], 1e3)
        d_imp = -tank.impeller_sdf(P)
        D = np.stack([d_side, d_bot, np.abs(d_imp)], 0)
        which = np.argmin(D, 0)
        h = flow.h
        y = np.clip(D.min(0), 0.25 * h, None)
        self.y = t(y)
        # normais (para o fluido) por diferenças do SDF de cada parede
        nrm = np.zeros(P.shape)
        r = np.hypot(P[..., 0], P[..., 1]) + 1e-12
        n_side = np.stack([-P[..., 0] / r, -P[..., 1] / r, 0 * r], -1)
        n_bot = np.stack([0 * r, 0 * r, 1 + 0 * r], -1)
        gi = np.stack(np.gradient(d_imp, flow.xc, flow.yc, flow.zc), -1)
        n_imp = gi / (np.linalg.norm(gi, axis=-1, keepdims=True) + 1e-12)
        for k, n in enumerate((n_side, n_bot, n_imp)):
            nrm = np.where((which == k)[..., None], n, nrm)
        self.n = t(nrm)
        om = flow.omega
        uw = np.stack([om * P[..., 1], -om * P[..., 0], 0 * r], -1)
        self.uw = t(np.where((which == 2)[..., None], 0.0, uw))     # agitador parado; vaso e fundo: −Ω×r
        self.side = t(which == 0) > 0.5                               # parede lateral é a mais próxima
        # células de parede: fluido com vizinho sólido (ou no fundo) e área de parede por volume
        sol = (~fl).to(flow.ft)
        nb = 0.0
        for axis in range(3):
            sp = pad(sol, axis)
            nb = nb + sl(sp, axis, 0, -2) * flow.H[axis] ** -1 + sl(sp, axis, 2, None) * flow.H[axis] ** -1
        if flow.cfg.bottom == "wall":
            bot = torch.zeros_like(sol)
            bot[:, :, 0] = 1.0 / flow.hz
            nb = nb + bot
        self.a_w = nb * self.flf                                      # [1/m] área de parede / volume
        self.wallcell = (self.a_w > 0) & fl
        # parede em escada: área das faces / área verdadeira ≈ |n_x| + |n_y| + |n_z| (em média, 4/π no vaso);
        # a tensão de parede é dividida por esse fator para que a força total seja a da superfície real
        self.stair = self.n.abs().sum(-1).clamp(1.0, 3.0 ** 0.5)
        self.wallzone = ((self.y < 1.5 * h) & fl) | self.wallcell
        # ---- campos k, ω
        U = om * tank.R
        k0 = 1.5 * (intensity * U) ** 2 + 1e-8
        w0 = math.sqrt(k0) / (BETA_STAR ** 0.25 * L_frac * tank.T)
        self.k = torch.full_like(flow.P, k0) * self.flf + 1e-10
        self.w = torch.full_like(flow.P, w0)
        self.nu_t = torch.zeros_like(flow.P)
        self.u_tau = torch.zeros_like(flow.P)
        self.F1 = torch.ones_like(flow.P)
        self.curvature = curvature
        self.nu_t_max_ratio = 1e5
        self.fr1 = torch.ones_like(flow.P)

    # ---------------------------------------------------------------- auxiliares
    def _faces_both_fluid(self, axis):
        return sl(self.flf, axis, 1, None) * sl(self.flf, axis, 0, -1)

    def _adv_diff(self, q, Q, Gam):
        """−∇·(u q) + q∇·u (upwind 1ª ordem, fluxos só entre células de fluido) + ∇·(Γ∇q)."""
        f = self.f
        out = 0.0
        divq = 0.0
        for axis in range(3):
            lo, hi = sl(q, axis, 0, -1), sl(q, axis, 1, None)
            m = self._faces_both_fluid(axis)
            Qa = sl(Q[axis], axis, 1, -1) * m
            g_lo, g_hi = sl(Gam, axis, 0, -1), sl(Gam, axis, 1, None)
            Gf = 0.5 * (g_lo + g_hi) * m
            F = Qa * torch.where(Qa >= 0, lo, hi) - Gf * (hi - lo) / f.H[axis]
            z = torch.zeros_like(sl(q, axis, 0, 1))
            Ff = torch.cat([z, F, z], dim=axis)
            Qf = torch.cat([z, Qa, z], dim=axis)
            out = out - (sl(Ff, axis, 1, None) - sl(Ff, axis, 0, -1)) / f.H[axis]
            divq = divq + (sl(Qf, axis, 1, None) - sl(Qf, axis, 0, -1)) / f.H[axis]
        return out + q * divq

    def wall_velocity_rel(self):
        """Velocidade tangencial relativa à parede nas células (vetor) e seu módulo."""
        f = self.f
        uc, vc, wc = avg(f.u, 0), avg(f.v, 1), avg(f.w, 2)
        U = torch.stack([uc, vc, wc], -1) - self.uw
        un = (U * self.n).sum(-1, keepdim=True)
        Ut = U - un * self.n
        return Ut, torch.linalg.vector_norm(Ut, dim=-1)

    # -------------------------------------------------------------------- passo
    def update(self, dt, g=None, gamma=None):
        f = self.f
        rho = self.rho
        nu = (f.mu / rho).clamp(min=1e-8)
        if gamma is None:
            g, gamma = f.strain(f.u, f.v, f.w)
        k, w = self.k, self.w
        y = self.y
        S2 = gamma ** 2
        # misturas F1, F2
        dk = [cgrad(k, a, f.H[a]) for a in range(3)]
        dw = [cgrad(w, a, f.H[a]) for a in range(3)]
        kw = sum(a * b for a, b in zip(dk, dw))
        CD = torch.clamp(2 * SIG_W2 * kw / w, min=1e-10)
        sk = torch.sqrt(k.clamp(min=0))
        arg1 = torch.minimum(torch.maximum(sk / (BETA_STAR * w * y), 500 * nu / (y ** 2 * w)),
                             4 * SIG_W2 * k / (CD * y ** 2))
        F1 = torch.tanh(arg1 ** 4)
        arg2 = torch.maximum(2 * sk / (BETA_STAR * w * y), 500 * nu / (y ** 2 * w))
        F2 = torch.tanh(arg2 ** 2)
        bl = lambda a, b: F1 * a + (1 - F1) * b  # noqa: E731
        nu_t = A1 * k / torch.maximum(A1 * w, gamma * F2)
        Pk = torch.minimum(nu_t * S2, 10 * BETA_STAR * k * w)
        if self.curvature:
            if g is None:
                g = f.strain(f.u, f.v, f.w)[0]
            self.fr1 = rotation_curvature_factor(f, g, w, f.omega)
            Pk = Pk * self.fr1
        # paredes: u_τ (Spalding), produção pela lei log e ω automático. Na faixa junto às paredes
        # (y < 1,5Δ) o gradiente da fronteira imersa não é físico: a produção vem da lei de parede
        Ut, Um = self.wall_velocity_rel()
        ut = spalding_utau(Um, y, nu)
        self.u_tau = torch.where(self.wallcell, ut, torch.zeros_like(ut))
        Pk = torch.where(self.wallzone, ut ** 3 / (KAPPA * y), Pk)
        Gk = nu + bl(SIG_K1, SIG_K2) * nu_t
        Gw = nu + bl(SIG_W1, SIG_W2) * nu_t
        Rk = self._adv_diff(k, (f.u, f.v, f.w), Gk) + Pk
        Rw = self._adv_diff(w, (f.u, f.v, f.w), Gw) + bl(ALPHA1, ALPHA2) * Pk / nu_t.clamp(min=1e-12) \
            + (1 - F1) * 2 * SIG_W2 * kw / w
        k_new = ((k + dt * Rk) / (1 + dt * BETA_STAR * w)).clamp(min=1e-10)
        w_new = ((w + dt * Rw) / (1 + dt * bl(BETA1, BETA2) * w)).clamp(min=1e-4)
        w_vis = 6 * nu / (BETA1 * y ** 2)
        w_log = ut / (math.sqrt(BETA_STAR) * KAPPA * y)
        w_new = torch.where(self.wallcell, torch.sqrt(w_vis ** 2 + w_log ** 2), w_new)
        self.k = torch.where(self.fl, k_new, torch.full_like(k_new, 1e-10))
        self.w = torch.where(self.fl, w_new, w_vis)
        self.F1 = F1
        nu_t = A1 * self.k / torch.maximum(A1 * self.w, gamma * F2)
        self.nu_t = torch.minimum(nu_t, self.nu_t_max_ratio * nu) * self.flf
        # ---- viscosidade para o momento. Nas células de parede, a viscosidade efetiva é a da lei de parede,
        # μ_w = ρu_τ²Δ/|U_t|, que dá τ_w = ρu_τ² na face com o sólido; a mesma μ_w é estendida para as
        # células sólidas vizinhas, de modo que a média na face seja μ_w qualquer que seja o ν_t transportado
        # (com k difundido do núcleo, ν_t na primeira célula pode ser muito maior que κu_τy, e a tensão na
        # parede sairia multiplicada).
        mu_w = (rho * ut ** 2 * f.h / Um.clamp(min=1e-6) / self.stair).clamp(min=f.mu)
        # com a lei de parede do escoamento (getattr wallfn), a parede lateral já recebe a tensão como
        # força distribuída: ali a viscosidade fica a natural, e a do sólido além da parede é anulada
        wc = self.wallcell & ~self.side if getattr(f, "wallfn", False) else self.wallcell
        mu_eff = torch.where(wc, mu_w, f.mu + rho * self.nu_t)
        ghost = torch.where(wc, mu_w, torch.where(self.wallcell, mu_eff, torch.zeros_like(mu_w)))
        solid_mu = extend_into_solid(ghost, self.wallcell, passes=1)
        f.mu_t = torch.where(self.fl, mu_eff - f.mu, (solid_mu - f.mu).clamp(min=-0.999 * f.mu))
        if f.cfg.bottom == "wall":
            # fundo (face da malha): viscosidade da face-fantasma que dá τ_w na primeira camada
            mwb = rho * ut[:, :, :1] ** 2 * (0.5 * f.hz) / Um[:, :, :1].clamp(min=1e-6)
            f.mu_bottom = (2 * mwb - mu_eff[:, :, :1]).clamp(min=0.05 * mwb)
        return self

    def wall_heat_transfer(self, cp, k_mol, Pr=None):
        """Coeficiente de troca nas células de parede pela função de Kader: h = ρ c_p u_τ / T⁺ [W/(m² K)]."""
        f = self.f
        nu = (f.mu / self.rho).clamp(min=1e-8)
        if Pr is None:
            Pr = (f.mu * cp / k_mol)
        yp = (self.y * self.u_tau / nu).clamp(min=1e-3)
        Tp = kader_Tplus(yp, torch.as_tensor(Pr, dtype=f.ft, device=f.dev) * torch.ones_like(yp))
        h_wf = self.rho * cp * self.u_tau / Tp.clamp(min=1e-6)
        h_lam = k_mol / self.y                                      # limite de condução pura
        return torch.where(self.wallcell, torch.maximum(h_wf, h_lam), torch.zeros_like(h_wf))

In [ ]:
%%writefile pyrokit/scalar.py
"""Transporte estacionário de escalares (temperatura, estado do polímero) sobre o escoamento convergido.

As escalas de tempo são muito diferentes: o escoamento no tanque se estabelece em segundos, enquanto a
pirólise e o tempo de residência do fundido são de dezenas de minutos. Por isso a temperatura e a
composição são resolvidas diretamente no regime permanente, no referencial do agitador (onde o
escoamento é estacionário):

    ∇·(u φ) − φ ∇·u − ∇·(Γ ∇φ) + S_p φ = S_c

com advecção upwind (monótona), difusão com Γ variável (molecular + turbulenta), células sólidas com
valor imposto (parede aquecida) ou isoladas (agitador), e solução por BiCGSTAB sem matriz explícita,
precondicionado por Jacobi (GPU).
"""
from __future__ import annotations

import math

import torch

from .ops import pad, sl


def sl_pad(face_coef, axis):
    """Soma, em cada célula, dos coeficientes das suas faces internas ao longo de axis."""
    z = torch.zeros_like(sl(face_coef, axis, 0, 1))
    return torch.cat([face_coef, z], dim=axis) + torch.cat([z, face_coef], dim=axis)


class SteadyScalar:
    def __init__(self, flow, u=None, v=None, w=None):
        """flow: TankFlow (malha, máscaras); (u, v, w): velocidades nas faces (padrão: as atuais)."""
        self.f = flow
        h, hz = flow.h, flow.hz
        self.H = (h, h, hz)
        self.A = (h * hz, h * hz, h * h)          # áreas das faces normais a x, y, z
        self.V = flow.vol
        fl = (flow.fluid > 0.5)
        self.fluid = fl.to(flow.ft)
        u = flow.u if u is None else u
        v = flow.v if v is None else v
        w = flow.w if w is None else w
        # vazões volumétricas nas faces internas entre duas células de fluido (zero junto ao sólido)
        self.Q = []
        self.both = []
        for axis, vel in enumerate((u, v, w)):
            both = sl(self.fluid, axis, 1, None) * sl(self.fluid, axis, 0, -1)
            q = sl(vel, axis, 1, -1) * both * self.A[axis]
            self.Q.append(q)
            self.both.append(both)
        # Cortar as faces junto ao sólido deixa um resíduo de divergência nas células de interface (a
        # fronteira imersa é difusa); uma projeção só no fluido torna o campo de vazões conservativo, para
        # que o balanço global de massa e energia feche.
        self.div_before = float(self._div_flux(self.Q).abs().sum())
        self.Q = self._make_conservative(self.Q)
        self.divQ = self._div_flux([q for q in self.Q])     # ∑ vazões que saem (≈ 0 após a projeção)

    # ------------------------------------------------------------ operadores
    def _lap_fluid(self, psi):
        """−∑ (A/Δ)(ψ_viz − ψ) nas faces fluido–fluido (Laplaciano de Neumann no fluido, SPD a menos de
        constantes por região conexa)."""
        F = [self.both[a] * self.A[a] / self.H[a] * (sl(psi, a, 1, None) - sl(psi, a, 0, -1)) for a in range(3)]
        return -self._div_flux(F)

    def _make_conservative(self, Q, tol: float = 1e-6, maxit: int = 5000):
        div = self._div_flux(Q) * self.fluid
        if float(div.abs().sum()) == 0.0:
            return Q
        dg = sum(sl_pad(self.both[a] * self.A[a] / self.H[a], a) for a in range(3)) * self.fluid
        Minv = torch.where(dg > 0, 1.0 / dg.clamp(min=1e-30), torch.zeros_like(dg))
        # CG para L ψ = div, L = −∇·(A∇) (SPD semidefinida; o lado direito é compatível em cada região
        # conexa); então ∇·(Q + A∇ψ) = div − Lψ = 0
        psi = torch.zeros_like(div)
        r = div.clone()
        z = Minv * r
        p = z.clone()
        rz = float((r * z).sum())
        r0 = float(torch.linalg.vector_norm(r)) + 1e-300
        best, best_psi = 1.0, psi
        for _ in range(maxit):
            Ap = self._lap_fluid(p)
            pAp = float((p * Ap).sum())
            if pAp <= 0:
                break
            al = rz / pAp
            psi = psi + al * p
            r = r - al * Ap
            rn = float(torch.linalg.vector_norm(r)) / r0
            if rn < best:
                best, best_psi = rn, psi
            if rn < tol or rn > 1e3 * best:          # convergiu, ou o arredondamento tomou conta
                break
            z = Minv * r
            rz_new = float((r * z).sum())
            p = z + (rz_new / rz) * p
            rz = rz_new
        psi = best_psi
        return [Q[a] + self.both[a] * self.A[a] / self.H[a] * (sl(psi, a, 1, None) - sl(psi, a, 0, -1))
                for a in range(3)]

    def _div_flux(self, F):
        """Soma dos fluxos que saem de cada célula, a partir dos fluxos nas faces internas."""
        out = 0.0
        for axis, Fa in enumerate(F):
            z = torch.zeros_like(sl(self.fluid, axis, 0, 1))
            Ff = torch.cat([z, Fa, z], dim=axis)
            out = out + sl(Ff, axis, 1, None) - sl(Ff, axis, 0, -1)
        return out

    def setup(self, Gamma, Sp, Sc, dirichlet_mask=None, dirichlet_value=None, bottom_value=None, wall_cond=None):
        """Γ nas células [unid·m²/s ou W/(m K)/(ρc)], S_p ≥ 0 e S_c por volume; dirichlet_mask: células
        sólidas com valor imposto (o resto do sólido é isolado); bottom_value: valor imposto no fundo
        (z = 0), se a parede do fundo também troca; wall_cond: condutância de parede por área nas células
        de fluido [m/s] (função de parede), no lugar de Γ/(Δ/2) nas faces com a parede."""
        f = self.f
        fl = self.fluid
        dm = torch.zeros_like(fl) if dirichlet_mask is None else dirichlet_mask.to(fl.dtype) * (1 - fl)
        self.dm = dm
        self.dv = torch.zeros_like(fl) if dirichlet_value is None else dirichlet_value * torch.ones_like(fl)
        # condutâncias difusivas nas faces internas: fluido–fluido e fluido–sólido com valor imposto
        self.Dc, self.Dw = [], []
        for axis in range(3):
            g_lo, g_hi = sl(Gamma, axis, 0, -1), sl(Gamma, axis, 1, None)
            gf = 2 * g_lo * g_hi / (g_lo + g_hi).clamp(min=1e-30)       # média harmônica
            a_lo, a_hi = sl(fl, axis, 0, -1), sl(fl, axis, 1, None)
            d_lo, d_hi = sl(dm, axis, 0, -1), sl(dm, axis, 1, None)
            ff = a_lo * a_hi
            fs = a_lo * d_hi + d_lo * a_hi                                # face entre fluido e parede
            if wall_cond is None:
                c_fs = 2 * torch.where(a_lo > 0, g_lo, g_hi) / self.H[axis]     # meia célula: Γ/(Δ/2)
            else:
                c_fs = torch.where(a_lo > 0, sl(wall_cond, axis, 0, -1), sl(wall_cond, axis, 1, None))
            self.Dc.append(ff * gf * self.A[axis] / self.H[axis] + fs * c_fs * self.A[axis])
            self.Dw.append((fs * c_fs * self.A[axis], a_lo))
        # fundo: troca com a parede em z = 0 (meia célula)
        self.bottom = None
        if bottom_value is not None:
            cb = 2 * sl(Gamma, 2, 0, 1) / self.H[2] if wall_cond is None else sl(wall_cond, 2, 0, 1)
            gb = cb * sl(fl, 2, 0, 1) * self.A[2]
            self.bottom = (gb, bottom_value)
        self.Sp, self.Sc = Sp * self.V, Sc * self.V
        self.extra = 0.0                    # diagonal extra (passo pseudo-transiente)
        # diagonal (para o precondicionador)
        diag = self._diag()
        self.diag = torch.where(fl > 0, diag, torch.ones_like(diag))
        return self

    def _diag(self):
        d = self.Sp * self.fluid
        for axis in range(3):
            Q, D = self.Q[axis], self.Dc[axis]
            # célula "lo" de cada face: saída para +; célula "hi": saída para −
            z = torch.zeros_like(sl(self.fluid, axis, 0, 1))
            out_lo = torch.cat([Q.clamp(min=0) + D, z], dim=axis)
            out_hi = torch.cat([z, (-Q).clamp(min=0) + D], dim=axis)
            d = d + out_lo + out_hi
        d = d - (self.divQ * self.fluid)
        if self.bottom is not None:
            gb = self.bottom[0]
            d = d + torch.cat([gb, torch.zeros_like(sl(d, 2, 1, None))], dim=2)
        return d

    def apply(self, phi):
        """A φ (só nas células de fluido; nas demais, identidade)."""
        fl = self.fluid
        r = (self.Sp + self.extra) * phi
        F = []
        for axis in range(3):
            lo, hi = sl(phi, axis, 0, -1), sl(phi, axis, 1, None)
            Q, D = self.Q[axis], self.Dc[axis]
            up = torch.where(Q >= 0, lo, hi)
            F.append(Q * up - D * (hi - lo))
        r = r + self._div_flux(F) - phi * self.divQ
        if self.bottom is not None:
            gb = self.bottom[0]
            r = r + torch.cat([gb * sl(phi, 2, 0, 1), torch.zeros_like(sl(phi, 2, 1, None))], dim=2)
        return torch.where(fl > 0, r, phi)

    def rhs(self):
        """Termo independente: fontes e parede do fundo; nas células sólidas com valor imposto, o valor
        (as paredes entram pelo acoplamento com essas células, que são incógnitas com linha identidade)."""
        fl, dm = self.fluid, self.dm
        b = self.Sc.clone()
        if self.bottom is not None:
            gb, Tb = self.bottom
            b = b + torch.cat([gb * Tb, torch.zeros_like(sl(b, 2, 1, None))], dim=2)
        return torch.where(fl > 0, b, self.dv * dm)

    # --------------------------------------------------------------- solver
    def rhs_norm(self):
        """Norma do lado direito efetivo nas células de fluido (fontes + valores impostos nas paredes,
        como se as linhas de Dirichlet fossem eliminadas): escala para os resíduos relativos."""
        b = self.rhs()
        xD = torch.where(self.fluid > 0, torch.zeros_like(b), b)
        return float(torch.linalg.vector_norm((b - self.apply(xD)) * self.fluid)) + 1e-30

    def solve(self, x0=None, tol: float = 1e-8, maxit: int = 4000, rel_r0: bool = False):
        """BiCGSTAB precondicionado por Jacobi. Devolve (φ, resíduo relativo, iterações). As células fora
        do fluido (linhas identidade) começam no valor exato e não mudam; o resíduo é relativo ao lado
        direito efetivo do fluido ou, com rel_r0, ao resíduo inicial (partida a quente)."""
        b = self.rhs()
        x = torch.zeros_like(b) if x0 is None else x0.clone()
        x = torch.where(self.fluid > 0, x, b)
        Minv = 1.0 / self.diag
        r = b - self.apply(x)
        r0 = r.clone()
        bn = (float(torch.linalg.vector_norm(r)) if rel_r0 else self.rhs_norm()) + 1e-30
        rho = alpha = omega = 1.0
        vv = torch.zeros_like(b)
        p = torch.zeros_like(b)
        res = float(torch.linalg.vector_norm(r)) / bn
        it = 0
        for it in range(1, maxit + 1):
            rho_new = float((r0 * r).sum())
            if abs(rho_new) < 1e-300:
                break
            beta = (rho_new / rho) * (alpha / omega)
            p = r + beta * (p - omega * vv)
            ph = Minv * p
            vv = self.apply(ph)
            r0v = float((r0 * vv).sum())
            if r0v == 0.0 or not math.isfinite(r0v):           # quebra do BiCGSTAB
                break
            alpha = rho_new / r0v
            s = r - alpha * vv
            sh = Minv * s
            t = self.apply(sh)
            tt = float((t * t).sum())
            omega = float((t * s).sum()) / tt if tt > 0 else 0.0
            x = x + alpha * ph + omega * sh
            r = s - omega * t
            rho = rho_new
            res = float(torch.linalg.vector_norm(r)) / bn
            if res < tol or omega == 0.0:
                break
        return x, res, it

    def wall_flux(self, phi):
        """Fluxo que entra no fluido pelas paredes com valor imposto (lateral e fundo), por célula de
        fluido [unid·m³/s]; para a temperatura, × ρc_p dá o calor em W."""
        out = torch.zeros_like(phi)
        for axis in range(3):
            Dw, a_lo = self.Dw[axis]
            lo, hi = sl(phi, axis, 0, -1), sl(phi, axis, 1, None)
            q = Dw * (hi - lo)                                    # do lado hi para o lo
            z = torch.zeros_like(sl(phi, axis, 0, 1))
            out = out + torch.cat([q * a_lo, z], dim=axis) - torch.cat([z, q * (1 - a_lo)], dim=axis)
        if self.bottom is not None:
            gb, vb = self.bottom
            out = out + torch.cat([gb * (vb - sl(phi, 2, 0, 1)), torch.zeros_like(sl(phi, 2, 1, None))], dim=2)
        return out * self.fluid

    def wall_loss(self):
        """A·1_fluido por célula: o que um deslocamento uniforme do fluido (paredes fixas) perde para as
        paredes e os sumidouros."""
        return self.apply(self.fluid) * self.fluid

    def solve_ptc(self, x0, tol: float = 5e-4, dtau0: float = 1.0, growth: float = 2.0, dtau_max: float = 300.0,
                  steps: int = 30, inner_tol: float = 1e-2, inner_maxit: int = 300, verbose: bool = False):
        """Continuação pseudo-transiente (Euler implícito em τ, passo crescendo até dtau_max) com BiCGSTAB
        em cada passo e correção do modo médio global: num vaso fechado com recirculação, o nível médio do
        escalar só relaxa na escala do tempo de residência (horas); um deslocamento uniforme ótimo resolve
        esse modo de uma vez, e o passo τ limitado mantém cada sistema linear bem condicionado. Um passo
        que piora o resíduo é rejeitado e refeito com dτ/4."""
        fl = self.fluid
        bn = self.rhs_norm()
        # trabalha com o desvio em relação à média inicial no fluido (x = c + y): os termos de advecção,
        # grandes, passam a agir sobre números pequenos, o que baixa o piso de arredondamento em float32
        c = float((x0 * fl).sum() / fl.sum().clamp(min=1.0))
        Sc_orig = self.Sc
        self.Sc = Sc_orig - self.apply(c * fl) * fl
        b0 = self.rhs()
        x = torch.where(fl > 0, x0 - c, b0)
        base_diag = self.diag.clone()
        loss = self.wall_loss()
        den = float(loss.sum())

        def residual(z):
            return float(torch.linalg.vector_norm((b0 - self.apply(z)) * fl)) / bn

        def mean_fix(z):
            if den > 0 and bool(torch.isfinite(z).all()):
                z = z + float(((b0 - self.apply(z)) * fl).sum()) / den * fl
            return z

        x = mean_fix(x)                                   # a partida já satisfaz o balanço global
        res = residual(x)
        dtau = dtau0
        n = 0
        for n in range(steps):
            self.extra = self.V / dtau * fl
            self.diag = torch.where(fl > 0, base_diag + self.extra, base_diag)
            Sc0 = self.Sc
            self.Sc = Sc0 + self.extra * x
            xn, _, _ = self.solve(x, tol=inner_tol, maxit=inner_maxit, rel_r0=True)
            self.Sc = Sc0
            self.extra = 0.0
            self.diag = base_diag
            xn = mean_fix(xn)                             # correção global (0-D) do nível médio
            rn = residual(xn) if bool(torch.isfinite(xn).all()) else float("inf")
            if verbose:
                print(f"    ptc {n}: dτ = {dtau:.3g} s, resíduo {rn:.2e}")
            if not rn < 2.0 * res:                         # passo ruim: rejeita e reduz dτ
                dtau = max(dtau / 4.0, 1e-3)
                continue
            x, res = xn, rn
            if res < tol:
                break
            dtau = min(dtau * growth, dtau_max)
        self.Sc = Sc_orig
        return torch.where(fl > 0, x + c, x), res, n + 1

In [ ]:
%%writefile pyrokit/reactor.py
"""Reator de pirólise 3-D: escoamento RANS no referencial do agitador + temperatura e química estacionárias.

Laço externo (Picard), como no documento de projeto:
  0. reator 0-D (tanque ideal) → campos uniformes iniciais e vazão de alimentação;
  1. escoamento com propriedades congeladas (viscosidade μ(γ̇, T, Mw) e massa específica ρ(T, Mn)),
     algumas voltas do agitador no referencial girante até o torque estabilizar;
  2. temperatura e espécies em regime permanente sobre o escoamento (implícito, GPU);
  3. controle de nível: alimentação = vapor + purga (modo B, parede com temperatura imposta) ou ajuste da
     temperatura da parede para uma vazão dada (modo A);
  4. atualiza viscosidade (relaxada em log) e empuxo e volta a 1.
"""
from __future__ import annotations

import math
import time
from dataclasses import dataclass, field

import numpy as np
import torch

from . import rheology as rh
from .chemistry import (CLOSURE, M_LO, MBAR, S_CHAR, SOLIDS_TARGET, FeedSpec, class_rates, nagata_h,
                        product_slate, solve_cstr)
from .flow import FlowConfig, TankFlow
from .kinetics import POLYMERS, R_GAS
from .ops import avg, extend_into_solid, to_faces
from .scalar import SteadyScalar
from .turbulence import BETA_STAR, SST

G = 9.81


@dataclass
class ReactorConfig:
    n_diam: int = 64                 # células no diâmetro do vaso
    T_wall_C: float = 480.0          # temperatura da parede (modo B)
    feed_kgph: float | None = None   # vazão imposta (modo A): a parede vira incógnita
    turbulence: str = "sst"          # "sst" ou "laminar"
    mu_cap: float = 3.0              # teto da viscosidade no momento [Pa s] (passo explícito)
    spin0: float = 0.9               # partida: líquido girando a 0,9 Ω (a fita arrasta quase todo o líquido)
    feed_r: float = 0.20             # raio do anel de alimentação [m]
    rho0: float = 620.0              # massa específica de referência [kg/m³]
    Sc_t: float = 0.7
    Pr_t: float = 0.85
    buoyancy: bool = True
    revs_first: float = 15.0         # voltas do agitador na primeira passada do escoamento
    revs_next: float = 4.0           # voltas nas passadas seguintes
    outer: int = 4                   # iterações externas
    device: str = "auto"
    dtype: str = "float32"


def _riazi_Mstar(T):
    """Massa molar [g/mol] do n-alcano que ferve a T (torch)."""
    return ((6.98291 - torch.log(1070.0 - T)) / 0.02013) ** 1.5


class Reactor:
    def __init__(self, tank, feed: FeedSpec, cfg: ReactorConfig):
        self.tank, self.feed, self.cfg = tank, feed, cfg
        self.items = feed.items()
        V = math.pi * tank.R ** 2 * tank.H_L
        A = 2 * math.pi * tank.R * tank.H_L + math.pi * tank.R ** 2
        self.V_liq, self.A_wall = V, A
        self.T_wall = cfg.T_wall_C + 273.15
        # ---- 0-D: ponto de partida (h pela correlação de Nagata, iterado com a viscosidade do fundido)
        N, d = tank.rpm / 60.0, tank.impeller_diameter
        h = 300.0
        for _ in range(4):
            r0 = solve_cstr(feed, V, A, h, T_wall=self.T_wall, feed_rate=None if cfg.feed_kgph is None
                            else cfg.feed_kgph / 3600.0, rho=cfg.rho0)
            h = nagata_h(r0.rho, N, d, r0.eta, 3240.0, 0.11, tank.T)[0]
        self.cstr0, self.h_nagata0 = r0, h
        if cfg.feed_kgph is not None:
            self.T_wall = r0.T_wall
        # ---- escoamento
        fcfg = FlowConfig(n_diam=cfg.n_diam, rho=cfg.rho0, mu_max=cfg.mu_cap, spin0=cfg.spin0, device=cfg.device,
                          dtype=cfg.dtype, vessel_ibm="wallfn" if cfg.turbulence == "sst" else "diffuse")
        self.flow = TankFlow(tank, fcfg, mu_fn=self._mu)
        f = self.flow
        self.flm = (f.fluid > 0.5).to(f.ft)               # células de fluido (máscara dos escalares)
        if cfg.turbulence == "sst":
            f.turb = SST(f)
        t = f._t
        # ---- campos escalares uniformes do 0-D
        P0 = f.P
        self.T = torch.full_like(P0, r0.T)
        self.state = []
        for s in r0.states:
            self.state.append({"Y": torch.full_like(P0, s["Y"]), "Z": torch.full_like(P0, s["Z"]),
                               "E": [torch.full_like(P0, e) for e in s["E"]]})
        self.F = r0.feed                                   # alimentação [kg/s]
        # anel de alimentação junto à superfície (no referencial girante, o bocal fixo vira um anel)
        r = f.r_c
        zc = t(f.P_c[..., 2])
        ring = ((r - cfg.feed_r).abs() < f.h) & (zc > tank.H_L - 2.0 * f.hz) & (f.fluid > 0.5)
        self.feed_mask = ring.to(f.ft)
        self.V_feed = float(self.feed_mask.sum()) * f.vol
        # paredes aquecidas: células sólidas do vaso (não do agitador) e fundo
        self.vessel_solid = t((tank.vessel_sdf(f.P_c) > 0).astype(float)) * (f.fluid < 0.5).to(f.ft)
        self._side_area_factor = None
        self.cp0 = self._cp_bulk(r0.T)
        self.history = []
        self.flow_rho_body()

    # ---------------------------------------------------------------- propriedades
    def _cp_bulk(self, T):
        cps = {"HDPE": rh.heat_capacity("HDPE", T), "LDPE": rh.heat_capacity("LDPE", T),
               "PP": rh.heat_capacity("PP", T)}
        return float(sum(w * cps[c.key] for c, w in self.items))

    def _k_c(self, c, T):
        p = POLYMERS[c.key]
        return c.f * p.A * torch.exp(-p.E / (R_GAS * T))

    def _n_star(self, c, T):
        return _riazi_Mstar(T) / 1000.0 / c.m0

    def _Mw(self, c, s):
        Y = s["Y"].clamp(min=1e-12)
        E = s["E"]
        Es = E[0] + E[1] + E[2]
        matrix = (Y - Es).clamp(min=1e-12)
        zm = (s["Z"] - sum(2 * e / m for e, m in zip(E, MBAR))).clamp(min=1e-12)
        Mn_m = matrix / zm
        return (sum(e * m for e, m in zip(E, MBAR)) + matrix * 1.5 * Mn_m) / Y          # kg/mol

    def eta0_field(self):
        """Viscosidade a taxa nula da mistura [Pa s] (média logarítmica pela massa)."""
        T = self.T
        lnmix, wsum = 0.0, 0.0
        for (c, _), s in zip(self.items, self.state):
            Mw = (self._Mw(c, s) * 1000.0).clamp(100.0, 1e7)
            r = rh.RHEO[c.rheo]
            ent = rh.eta0(r, T, Mw, xp=torch)
            wax = 0.015 * (Mw / 1080.0) ** 1.9 * torch.exp((27e3 / R_GAS) * (1.0 / T - 1.0 / 422.15))
            e0 = torch.nan_to_num(torch.maximum(ent, wax), nan=1e-3, posinf=1e6).clamp(1e-6, 1e8)
            Yw = s["Y"].clamp(min=0.0)
            lnmix = lnmix + Yw * torch.log(e0)
            wsum = wsum + Yw
        e = torch.exp(lnmix / wsum.clamp(min=1e-12))
        # fora do líquido (paredes, agitador): o valor do fluido vizinho
        return extend_into_solid(torch.where(wsum > 1e-9, e, torch.full_like(e, 1e-3)), self.flm > 0, passes=4)

    def _mu(self, gamma):
        if not hasattr(self, "_eta0"):
            return torch.full_like(gamma, max(self.cstr0.eta, rh.ETA_FLOOR))
        main = max(self.items, key=lambda cw: cw[1])[0]
        r = rh.RHEO[main.rheo]
        e0 = self._eta0
        mu = rh.ETA_FLOOR + e0 / (1.0 + (e0 * gamma / r.tau_star) ** (1.0 - r.n))
        return mu.clamp(rh.ETA_FLOOR, self.cfg.mu_cap)

    def density_field(self):
        T = self.T
        rho, wsum = 0.0, 0.0
        Tc = T - 273.15
        for (c, _), s in zip(self.items, self.state):
            Mn = s["Y"] / s["Z"].clamp(min=1e-12) * 1000.0                     # g/mol
            if c.key in ("HDPE", "LDPE"):
                ri = 1000.0 / (1.1595 + 8.0e-4 * Tc)
            else:
                ri = 738.6 / (1.0 + 6.7e-4 * (Tc - 230.0))
            fb = ((Mn - 500.0) / 1500.0).clamp(0.0, 1.0)
            ri = fb * ri + (1 - fb) * 610.0
            rho = rho + s["Y"] * ri
            wsum = wsum + s["Y"]
        return rho / wsum.clamp(min=1e-12)

    def flow_rho_body(self, relax: float = 1.0):
        """Congela viscosidade (relaxada em log) e empuxo para a próxima passada do escoamento."""
        e0 = self.eta0_field()
        if hasattr(self, "_eta0") and relax < 1.0:
            e0 = torch.exp(relax * torch.log(e0) + (1 - relax) * torch.log(self._eta0))
        self._eta0 = e0
        f = self.flow
        if self.cfg.buoyancy:
            rho = self.density_field()
            rho_bulk = float((rho * f.fluid).sum() / f.fluid.sum())
            b = (rho - rho_bulk) / self.cfg.rho0 * (f.fluid > 0.5).to(f.ft)
            om2 = f.omega ** 2
            X, Y = f._t(f.P_c[..., 0]), f._t(f.P_c[..., 1])
            f.body = [to_faces(b * om2 * X, 0), to_faces(b * om2 * Y, 1), to_faces(-G * b, 2)]
        f.update_viscosity()

    # ------------------------------------------------------------------ escoamento
    def run_flow(self, revs: float, log_every: float = 1.0, callback=None):
        f = self.flow
        N = self.tank.rpm / 60.0
        t_end = f.time + revs / N
        t_avg0 = t_end - min(1.0, revs) / N                    # média na última volta
        acc, n_acc = None, 0
        next_log = f.time + log_every / N
        while f.time < t_end:
            f.step()
            f.record()
            if f.time >= t_avg0:
                cur = (f.u, f.v, f.w)
                acc = [c.clone() for c in cur] if acc is None else [a + c for a, c in zip(acc, cur)]
                n_acc += 1
            if callback is not None and f.time >= next_log:
                callback(self)
                next_log += log_every / N
        self.mean_uvw = [a / n_acc for a in acc]
        return self

    def power_number(self, n_avg=200):
        f = self.flow
        N, d = self.tank.rpm / 60.0, self.tank.impeller_diameter
        return f.power(n_avg) / (self.cfg.rho0 * N ** 3 * d ** 5)

    # ------------------------------------------------------------------ escalares
    def _wall_cond(self):
        """Condutância de parede por área [W/(m² K)] nas células de fluido junto ao vaso."""
        f = self.flow
        k_mol = 0.11
        if f.turb is not None:
            mu = f.mu
            h = f.turb.wall_heat_transfer(self.cp0, k_mol)
        else:
            h = k_mol / f.wall_dist.clamp(min=0.25 * f.h)
        return h

    def _side_factor(self, sc):
        """Correção da área em escada da parede lateral: área exata / área das faces."""
        if self._side_area_factor is None:
            f = self.flow
            A_true = 2 * math.pi * self.tank.R * self.tank.H_L
            fl = sc.fluid
            dm = self.vessel_solid
            A_st = 0.0
            for axis in (0, 1):
                from .ops import sl
                a_lo, a_hi = sl(fl, axis, 0, -1), sl(fl, axis, 1, None)
                d_lo, d_hi = sl(dm, axis, 0, -1), sl(dm, axis, 1, None)
                A_st += float((a_lo * d_hi + d_lo * a_hi).sum()) * sc.A[axis]
            self._side_area_factor = A_true / max(A_st, 1e-12)
        return self._side_area_factor

    def _vapour(self, T):
        """S_v de cada polímero [kg/(m³ s)] e d S_v/dT, com ψ limitado ao comprimento de corte."""
        rho0 = self.cfg.rho0
        Sv, dSv = [], []
        for (c, _), s in zip(self.items, self.state):
            kc = self._k_c(c, T)
            ns = self._n_star(c, T)
            Zc = torch.minimum(s["Z"], s["Y"] / (ns * c.m0))
            sv = rho0 * kc * ns ** 2 * c.m0 * Zc * self.flm
            p = POLYMERS[c.key]
            dlnN = 1.5 * (1.0 / (1070.0 - T)) / (6.98291 - torch.log(1070.0 - T))     # d ln N*/dT
            Sv.append(sv)
            dSv.append(sv * (p.E / (R_GAS * T ** 2) + 2 * dlnN))
        return Sv, dSv

    def solve_scalars(self, picard: int = 6, relax: float = 0.6, verbose: bool = False):
        """Temperatura e estado do polímero em regime permanente sobre o escoamento médio (Picard
        sub-relaxado: espécies → temperatura → controle de nível)."""
        f = self.flow
        cfg = self.cfg
        rho0 = cfg.rho0
        sc = SteadyScalar(f, *self.mean_uvw)
        nu_t = f.turb.nu_t if f.turb is not None else torch.zeros_like(f.P)
        Gs = nu_t / cfg.Sc_t + 1e-9
        k_mol = 0.11
        GT = k_mol / (rho0 * self.cp0) + nu_t / cfg.Pr_t
        hw = self._wall_cond()
        fac = self._side_factor(sc)
        # bottom: área exata (faces alinhadas); lateral: corrigida pela área em escada
        wall_cond = hw * fac / (rho0 * self.cp0)
        bottom_cond = hw / (rho0 * self.cp0)
        Tf = self.feed.T_feed
        info = {}
        fl = self.flm > 0
        for it in range(picard):
            Sf = self.F * self.feed_mask / self.V_feed                 # kg/(m³ s)
            T = self.T
            Sv, _ = self._vapour(T)
            Svt = sum(Sv)
            # ---- espécies (Picard nos acoplamentos, PTC + BiCGSTAB em cada uma)
            for (c, w), s, sv in zip(self.items, self.state, Sv):
                kc = self._k_c(c, T)
                ns = self._n_star(c, T)
                r1, r2, r3 = class_rates_torch(c, kc)
                spf = Sf / rho0
                ex = Svt / rho0
                Yf, Zf = w * (1 - self.feed.ash), w * (1 - self.feed.ash) / c.Mn0
                Y = self._solve(sc, Gs, spf, spf * Yf - (1 + S_CHAR) * sv / rho0 + ex * s["Y"], s["Y"])
                Z = self._solve(sc, Gs, spf + 2 * kc * ns, spf * Zf + kc * Y / c.m0 + ex * s["Z"], s["Z"])
                E1 = self._solve(sc, Gs, spf + r1, spf * Yf + ex * s["E"][0], s["E"][0])
                E2 = self._solve(sc, Gs, spf + r2, r1 * E1 + ex * s["E"][1], s["E"][1])
                E3 = self._solve(sc, Gs, spf + r3, r2 * E2 + ex * s["E"][2], s["E"][2])
                mix = lambda new, old: torch.where(fl, old + relax * (new - old), old)  # noqa: E731
                s["Y"] = mix(Y.clamp(min=1e-9), s["Y"])
                s["Z"] = torch.minimum(mix(Z.clamp(min=1e-12), s["Z"]), s["Y"] / (ns * c.m0))
                s["E"] = [mix(e.clamp(min=0), e0) for e, e0 in zip((E1, E2, E3), s["E"])]
                Es = s["E"][0] + s["E"][1] + s["E"][2]                 # as classes cabem na massa do polímero
                scale = torch.where(Es > 0.98 * s["Y"], 0.98 * s["Y"] / Es.clamp(min=1e-30), torch.ones_like(Es))
                s["E"] = [e * scale for e in s["E"]]
            # ---- energia (linearização de Newton do sumidouro endotérmico, com o estado novo)
            Sv, dSv = self._vapour(T)
            dH = [c.dH for c, _ in self.items]
            sinkT = sum(h_ * d for h_, d in zip(dH, dSv))
            src = -sum(h_ * s_ for h_, s_ in zip(dH, Sv)) + sinkT * T + self._dissipation()
            SpT = Sf / rho0 + sinkT / (rho0 * self.cp0)
            ScT = Sf * Tf / rho0 + src / (rho0 * self.cp0)
            sc.setup(GT, SpT, ScT, dirichlet_mask=self.vessel_solid > 0, dirichlet_value=self.T_wall,
                     bottom_value=self.T_wall, wall_cond=self._blend_wall(wall_cond, bottom_cond))
            Tn, res, nit = sc.solve_ptc(torch.where(fl, self.T, torch.full_like(self.T, self.T_wall)), dtau0=5.0)
            Tn = torch.where(fl, self.T + relax * (Tn - self.T), Tn)
            # fora do fluido: parede do vaso na temperatura imposta; agitador com a do fluido vizinho
            Ts = extend_into_solid(Tn, fl, passes=3)
            self.T = torch.where(self.vessel_solid > 0, torch.full_like(Tn, self.T_wall), Ts)
            self._last_sc = sc
            # ---- controle de nível com o vapor da temperatura e do estado novos
            V = float(sum(self._vapour(self.T)[0]).sum()) * f.vol
            F_lvl = V * (1 + S_CHAR / SOLIDS_TARGET) / (1 - self.feed.ash / SOLIDS_TARGET)
            if cfg.feed_kgph is None:
                self.F = self.F + relax * (F_lvl - self.F)
            else:                                   # modo A: ajusta a parede para a vazão pedida
                target = cfg.feed_kgph / 3600.0
                self.T_wall += relax * (target - F_lvl) / max(F_lvl, 1e-9) * 40.0
                self.F = target
            info = {"vapour": V, "feed_level": F_lvl, "res_T": res, "it_T": nit}
            if verbose:
                print(f"  química {it}: vapor {3600 * V:.1f} kg/h, alimentação {3600 * self.F:.1f} kg/h, "
                      f"T média {self.T_mean() - 273.15:.1f} °C, resíduo T {res:.1e}")
        return info

    def _blend_wall(self, side, bottom):
        """Condutância de parede: lateral (corrigida) em geral, a do fundo na primeira camada."""
        out = side.clone()
        out[:, :, 0] = bottom[:, :, 0]
        return out

    def _solve(self, sc, Gamma, Sp, Sc, x0):
        sc.setup(Gamma, Sp * torch.ones_like(x0), Sc * torch.ones_like(x0))
        x, _, _ = sc.solve_ptc(x0 * sc.fluid, dtau0=5.0)
        return torch.where(sc.fluid > 0, x.clamp(min=0.0), x0)

    def _dissipation(self):
        """Φ = 2μS:S + ρβ*kω [W/m³] (a potência do agitador vira calor)."""
        f = self.flow
        phi = f.mu * f.gamma ** 2
        if f.turb is not None:
            phi = phi + self.cfg.rho0 * BETA_STAR * f.turb.k * f.turb.w
        return phi * (f.fluid > 0.5)

    # ------------------------------------------------------------------ resultados
    def T_mean(self):
        return float((self.T * self.flm).sum() / self.flm.sum())

    def kpis(self) -> dict:
        """Indicadores do reator e fechamento dos balanços de massa e energia."""
        f = self.flow
        fl = self.flm > 0
        sc = self._last_sc
        T = self.T
        rho0, cp = self.cfg.rho0, self.cp0
        Sv = []
        for (c, _), s in zip(self.items, self.state):
            kc = self._k_c(c, T)
            ns = self._n_star(c, T)
            Sv.append(rho0 * kc * ns ** 2 * c.m0 * torch.minimum(s["Z"], s["Y"] / (ns * c.m0)) * self.flm)
        Svt = sum(Sv)
        V = float(Svt.sum()) * f.vol
        T_vap = float((Svt * T).sum() / Svt.sum().clamp(min=1e-30))
        # energia: calor que entra pelas paredes (+ dissipação) = aquecimento da carga + pirólise
        Qw = rho0 * cp * float(sc.wall_flux(T).sum())
        Phi = float(self._dissipation().sum()) * f.vol
        Sf = self.F * self.feed_mask / self.V_feed
        Q_sens = float((Sf * cp * (T - self.feed.T_feed) * self.flm).sum()) * f.vol
        Q_react = sum(float(s_.sum()) * f.vol * c.dH for (c, _), s_ in zip(self.items, Sv))
        e_err = (Qw + Phi - Q_sens - Q_react) / max(abs(Qw), 1e-9)
        drain = (self.F * self.feed.ash + S_CHAR * V) / SOLIDS_TARGET
        eta = self._mu(f.gamma) if hasattr(self, "_eta0") else None
        Mw = sum(s["Y"] * self._Mw(c, s) for (c, _), s in zip(self.items, self.state)) / \
            sum(s["Y"] for s in self.state).clamp(min=1e-12)
        N, d = self.tank.rpm / 60.0, self.tank.impeller_diameter
        n_fl = float(self.flm.sum())
        mu_b = float((eta * self.flm).sum()) / n_fl if eta is not None else self.cstr0.eta
        Tb = self.T_mean()
        return {
            "T_bulk_C": Tb - 273.15, "T_max_C": float(T[fl].max()) - 273.15, "T_min_C": float(T[fl].min()) - 273.15,
            "T_wall_C": self.T_wall - 273.15, "feed_kgph": 3600 * self.F, "vapour_kgph": 3600 * V,
            "drain_kgph": 3600 * drain, "mass_balance_err": (self.F - V - drain) / max(self.F, 1e-12),
            "duty_kW": Qw / 1e3, "dissipation_kW": Phi / 1e3, "energy_balance_err": e_err,
            "h_mean": Qw / (self.A_wall * max(self.T_wall - Tb, 1e-6)),
            "Np": self.power_number(), "power_kW": f.power() / 1e3,
            "Mw_bulk_kgmol": float((Mw * self.flm).sum()) / n_fl,
            "eta_bulk": mu_b, "Re": rho0 * N * d ** 2 / mu_b,
            "slate": product_slate(T_vap), "T_vapour_C": T_vap - 273.15,
            "residence_h": rho0 * self.V_liq / max(self.F, 1e-12) / 3600.0,
        }

    def run(self, verbose: bool = True, callback=None):
        cfg = self.cfg
        t0 = time.perf_counter()
        for k in range(cfg.outer):
            revs = cfg.revs_first if k == 0 else cfg.revs_next
            self.run_flow(revs, callback=callback)
            info = self.solve_scalars(verbose=verbose)
            self.flow_rho_body(relax=0.5)
            kp = self.kpis()
            kp.update(info)
            kp["outer"] = k
            kp["wall_s"] = time.perf_counter() - t0
            self.history.append(kp)
            if verbose:
                print(f"[{k + 1}/{cfg.outer}] T fundido {kp['T_bulk_C']:.1f} °C (máx {kp['T_max_C']:.1f}), "
                      f"alimentação {kp['feed_kgph']:.0f} kg/h, calor {kp['duty_kW']:.1f} kW, "
                      f"Np {kp['Np']:.2f}, μ {kp['eta_bulk']:.2e} Pa s, {kp['wall_s'] / 60:.1f} min")
        return self.history[-1]


def class_rates_torch(c, kc):
    r1 = (kc / (2 * c.m0)) / (1.0 / 64.0 - 1.0 / c.Mw0)
    r2 = kc * M_LO[1] / (1.5 * c.m0)
    r3 = kc * M_LO[2] / (1.5 * c.m0)
    return r1, r2, r3

In [ ]:
%%writefile pyrokit/video.py
"""Vídeo em HD (1080p): cartões de título, texto e números-chave, figuras com legenda e montagem final.

Mesma linguagem visual do estudo do flare: fundo escuro, textos com PIL (vírgula decimal), entrada suave e
codificação H.264 pelo ffmpeg, quadro a quadro, sem passar pelo matplotlib.
"""
from __future__ import annotations

import math
import re
import shutil
import subprocess
from pathlib import Path

import numpy as np

_DEC = re.compile(r"(?<=\d)\.(?=\d)")


def br(text: str) -> str:
    """Vírgula decimal (norma brasileira) em todo número de um texto: '6.8 m/s' → '6,8 m/s'."""
    return _DEC.sub(",", text) if isinstance(text, str) else text


def _font(size: int, bold: bool = False):
    from PIL import ImageFont
    import matplotlib
    name = "DejaVuSans-Bold.ttf" if bold else "DejaVuSans.ttf"
    path = Path(matplotlib.get_data_path()) / "fonts" / "ttf" / name
    try:
        return ImageFont.truetype(str(path), size)
    except OSError:
        return ImageFont.load_default()


def _mono(size: int, bold: bool = False):
    from PIL import ImageFont
    import matplotlib
    name = "DejaVuSansMono-Bold.ttf" if bold else "DejaVuSansMono.ttf"
    try:
        return ImageFont.truetype(str(Path(matplotlib.get_data_path()) / "fonts" / "ttf" / name), size)
    except OSError:
        return _font(size, bold)


class FFmpegPipe:
    """Escreve quadros RGB (H, W, 3) uint8 direto no ffmpeg (H.264, CRF baixo, yuv420p)."""

    def __init__(self, path: str, size: tuple, fps: int = 30, crf: int = 14):
        ff = shutil.which("ffmpeg")
        if ff is None:
            raise RuntimeError("ffmpeg não encontrado")
        W, H = size
        self.p = subprocess.Popen(
            [ff, "-y", "-loglevel", "error", "-f", "rawvideo", "-pix_fmt", "rgb24", "-s", f"{W}x{H}",
             "-r", str(fps), "-i", "-", "-c:v", "libx264", "-preset", "slow", "-crf", str(crf),
             "-tune", "film", "-pix_fmt", "yuv420p", "-movflags", "+faststart", path],
            stdin=subprocess.PIPE)
        self.path = path

    def write(self, frame: np.ndarray):
        self.p.stdin.write(np.ascontiguousarray(frame, dtype=np.uint8).tobytes())

    def close(self):
        self.p.stdin.close()
        if self.p.wait() != 0:
            raise RuntimeError(f"ffmpeg falhou ao gravar {self.path}")
        return self.path



_GLUE = re.compile(r"^(\([A-Z]{2}\)|m|m/s|s|kg/h|kg/s|kW|MW|t/a|W/m²K|Pa\u00a0s|mPa\u00a0s|rpm|°C|K|%|h)[.,;:)]?$")


def _words(text):
    """Palavras para quebra de linha, sem separar '(SP)' do nome nem a unidade do número."""
    out = []
    for w in text.split():
        if out and _GLUE.match(w):
            out[-1] += "\u00a0" + w
        else:
            out.append(w)
    return out


def _wrap(draw, text, font, width):
    words, lines, cur = _words(text), [], ""
    for w in words:
        nxt = (cur + " " + w).strip()
        if draw.textlength(nxt, font=font) <= width:
            cur = nxt
        else:
            if cur:
                lines.append(cur)
            cur = w
    return lines + ([cur] if cur else [])


def _wrap_balanced(draw, text, font, width):
    """Como _wrap, mas com linhas de comprimento parecido (evita uma palavra sozinha na última linha)."""
    lines = _wrap(draw, text, font, width)
    if len(lines) < 2:
        return lines
    lo, hi = 0.3 * width, float(width)
    for _ in range(18):                      # menor largura que mantém o mesmo número de linhas
        mid = 0.5 * (lo + hi)
        if len(_wrap(draw, text, font, mid)) <= len(lines):
            hi = mid
        else:
            lo = mid
    return _wrap(draw, text, font, hi)


def _caption_alpha(captions, t, fade=0.45):
    """Legenda ativa no instante t (s) e sua opacidade (entrada e saída suaves)."""
    for t0, t1, text in captions:
        if t0 <= t < t1:
            return text, float(np.clip(min(t - t0, t1 - t) / fade, 0, 1))
    return None, 0.0


def even_captions(texts, duration, lead=0.3):
    """Distribui as legendas igualmente na duração do segmento: [(t0, t1, texto)]."""
    texts = [t for t in texts if t]
    if not texts:
        return []
    d = (duration - lead) / len(texts)
    return [(lead + k * d, lead + (k + 1) * d, t) for k, t in enumerate(texts)]


def draw_lower_third(im, text, alpha, s, accent=(255, 176, 64), y_bottom=None, max_w=None):
    """Legenda em caixa escura translúcida no terço inferior (PIL RGBA)."""
    from PIL import Image, ImageDraw
    if not text or alpha <= 0:
        return im
    W, H = im.size
    font = _font(int(34 * s))
    layer = Image.new("RGBA", im.size, (0, 0, 0, 0))
    d = ImageDraw.Draw(layer)
    max_w = max_w or int(W * 0.62)
    lines = _wrap(d, br(text), font, max_w)
    lh = int(font.size * 1.28)
    pad = int(22 * s)
    box_h = lh * len(lines) + 2 * pad
    box_w = max(d.textlength(ln, font=font) for ln in lines) + 2 * pad + int(10 * s)
    y1 = y_bottom if y_bottom is not None else H - int(150 * s)
    y0 = y1 - box_h
    x0 = int(60 * s)
    d.rounded_rectangle([x0, y0, x0 + box_w, y1], radius=int(10 * s), fill=(6, 9, 15, int(185 * alpha)))
    d.rectangle([x0, y0, x0 + int(6 * s), y1], fill=accent + (int(255 * alpha),))
    for k, ln in enumerate(lines):
        d.text((x0 + pad + int(10 * s), y0 + pad + k * lh), ln, font=font, fill=(240, 244, 250, int(255 * alpha)))
    im.alpha_composite(layer)
    return im



def _gradient(W: int, H: int) -> np.ndarray:
    bg = np.zeros((H, W, 3), np.float32)
    yy = np.linspace(0, 1, H)[:, None]
    for c, (a, b) in enumerate(zip((8, 11, 20), (14, 20, 34))):
        bg[..., c] = a + (b - a) * yy
    return bg


def _bright_cx(im: np.ndarray) -> float:
    """Posição horizontal (fração da largura) do centro luminoso da imagem."""
    lum = np.asarray(im, np.float32).mean(2)
    wgt = np.clip(lum - np.percentile(lum, 50), 0, None)
    return float((wgt.sum(0) * np.arange(lum.shape[1])).sum() / max(wgt.sum(), 1e-6)) / lum.shape[1]


def _compose(bg: np.ndarray, im: np.ndarray, place: float, strength: float, ramp_from: float,
             cx: float) -> np.ndarray:
    """Chama deslocada para a fração `place` da largura, esmaecida e com rampa da esquerda para a direita."""
    W = bg.shape[1]
    im = np.asarray(im, np.float32)
    sh = int((place - cx) * W)
    out = np.zeros_like(im)
    if sh >= 0:
        out[:, sh:] = im[:, :W - sh]
    else:
        out[:, :W + sh] = im[:, -sh:]
    ramp = np.clip((np.linspace(0, 1, W) - ramp_from) / 0.35, 0, 1)[None, :, None] ** 1.5
    return np.clip(np.maximum(bg, strength * ramp * out), 0, 255)


def _card_background(size, backdrop=None, place: float = 0.78, strength: float = 0.5,
                     ramp_from: float = 0.42) -> np.ndarray:
    """Fundo escuro em gradiente; com `backdrop` (quadro HD do reator, sem textos), a imagem é deslocada
    para a fração `place` da largura e entra esmaecida, sem competir com o texto à esquerda."""
    W, H = size
    bg = _gradient(W, H)
    if backdrop is not None:
        from PIL import Image
        im = np.asarray(Image.fromarray(np.asarray(backdrop, np.uint8)).resize((W, H), Image.LANCZOS), np.float32)
        bg = _compose(bg, im, place, strength, ramp_from, _bright_cx(im))
    return np.clip(bg, 0, 255)


def _probe(src: str):
    """(largura, altura, duração [s]) de um vídeo, lidos da saída do ffmpeg."""
    txt = subprocess.run([shutil.which("ffmpeg"), "-i", src], capture_output=True, text=True).stderr
    m = re.search(r", (\d{2,5})x(\d{2,5})", txt)
    d = re.search(r"Duration: (\d+):(\d+):([\d.]+)", txt)
    dur = 3600 * int(d.group(1)) + 60 * int(d.group(2)) + float(d.group(3)) if d else 0.0
    return int(m.group(1)), int(m.group(2)), dur


class Backdrop:
    """Fundo dos cartões. Aceita None, um quadro RGB (fundo fixo) ou um vídeo limpo (sem textos) do
    reator girando: aí a imagem se move atrás dos textos, em câmera lenta (`speed`) e em
    laço sem emenda (fusão de `cross` quadros). O relógio continua de um segmento para o outro. Os quadros
    ficam na memória em resolução reduzida (`width`), o bastante para um fundo esmaecido."""

    def __init__(self, src=None, size=(1920, 1080), speed: float = 0.5, width: int = 960, max_mb: float = 500.0,
                 cross: int = 24):
        from PIL import Image
        self.size, self.speed, self.k = tuple(size), speed, 0
        self.still, self.frames, self._cache = None, None, {}
        if src is None:
            return
        if not isinstance(src, (str, Path)):
            self.still = np.asarray(src, np.uint8)
            return
        src = str(src)
        if src.lower().endswith((".png", ".jpg", ".jpeg")):
            self.still = np.asarray(Image.open(src).convert("RGB"))
            return
        w0, h0, dur = _probe(src)
        bw = min(width, w0)
        bh = max(2, int(round(h0 * bw / w0 / 2)) * 2)
        nmax = max(8, int(max_mb * 1e6 / (bw * bh * 3)))
        fps = min(30.0, nmax / max(dur, 1e-3))          # clipes longos são subamostrados para caber
        p = subprocess.Popen([shutil.which("ffmpeg"), "-loglevel", "error", "-i", src, "-vf",
                              f"fps={fps:.5f},scale={bw}:{bh}", "-f", "rawvideo", "-pix_fmt", "rgb24", "-"],
                             stdout=subprocess.PIPE)
        n, fr = bw * bh * 3, []
        while len(fr) < nmax:
            buf = p.stdout.read(n)
            if len(buf) < n:
                break
            fr.append(np.frombuffer(buf, np.uint8).reshape(bh, bw, 3))
        p.stdout.close()
        p.wait()
        if not fr:
            raise ValueError(f"nenhum quadro lido de {src}")
        self.frames = fr
        self.speed = speed * fps / 30.0                  # mesma velocidade aparente com quadros mais espaçados
        self.cross = min(cross, len(fr) // 3)
        self.period = len(fr) - self.cross
        self.cx = _bright_cx(np.mean([f.astype(np.float32) for f in fr[::max(1, len(fr) // 8)]], axis=0))
        self.grad = _gradient(bw, bh)

    @property
    def animated(self) -> bool:
        return self.frames is not None

    def _seq(self, j: int) -> np.ndarray:
        """Quadro j do laço: os primeiros `cross` quadros fundem o fim do clipe ao começo."""
        F, C, Lp = self.frames, self.cross, self.period
        j %= Lp
        if j < C:
            w = j / C
            return (1 - w) * F[Lp + j].astype(np.float32) + w * F[j].astype(np.float32)
        return F[j].astype(np.float32)

    def next(self, place: float = 0.78, strength: float = 0.5, ramp_from: float = 0.42) -> np.ndarray:
        """Próximo fundo (uint8, H×W×3); cada chamada avança um quadro do vídeo final."""
        from PIL import Image
        if not self.animated:
            key = (place, strength, ramp_from)
            if key not in self._cache:
                self._cache[key] = _card_background(self.size, self.still, place, strength,
                                                    ramp_from).astype(np.uint8)
            return self._cache[key]
        pos = self.k * self.speed
        self.k += 1
        i0, w = int(pos), pos - int(pos)
        im = (1 - w) * self._seq(i0) + w * self._seq(i0 + 1)
        bg = _compose(self.grad, im, place, strength, ramp_from, self.cx)
        return np.asarray(Image.fromarray(bg.astype(np.uint8)).resize(self.size, Image.BILINEAR))


def _backdrop(backdrop, size) -> Backdrop:
    return backdrop if isinstance(backdrop, Backdrop) else Backdrop(backdrop, size)


def still_of(src: str, path: str, t: float) -> str:
    """Salva o quadro do instante t [s] de um vídeo como PNG (ex.: capa do post)."""
    subprocess.run([shutil.which("ffmpeg"), "-loglevel", "error", "-y", "-ss", f"{t:.2f}", "-i", src,
                    "-frames:v", "1", path], check=True)
    return path


def render_title_card(path: str, title: str, subtitle: str, credit: str = "", note: str = "",
                      size=(1920, 1080), fps: int = 30, seconds: float = 4.5, backdrop=None,
                      accent=(255, 176, 64), contact: str = "") -> str:
    """Abertura (ou encerramento) em HD: título e subtítulo em linhas equilibradas, autoria, contato
    (ex.: endereço do LinkedIn) e aviso, com entrada suave. backdrop: None, quadro RGB ou Backdrop."""
    from PIL import Image, ImageDraw
    W, H = size
    s = H / 1080.0
    bd = _backdrop(backdrop, size)
    probe = ImageDraw.Draw(Image.new("RGBA", (8, 8)))
    title, subtitle, credit, note = br(title), br(subtitle), br(credit), br(note)
    f_t, f_s, f_c, f_n = _font(int(72 * s), True), _font(int(36 * s)), _font(int(28 * s)), _font(int(22 * s))
    t_lines = _wrap_balanced(probe, title, f_t, int(0.86 * W))
    s_lines = _wrap_balanced(probe, subtitle, f_s, int(0.80 * W))
    blocks = []      # (linhas, fonte, cor, altura de linha, atraso)
    blocks.append((t_lines, f_t, (240, 244, 250), int(f_t.size * 1.18), 0.0))
    blocks.append(([""], f_c, (0, 0, 0), int(24 * s), 0.0))
    blocks.append((s_lines, f_s, accent, int(f_s.size * 1.3), 0.4))
    if credit:
        blocks.append(([""], f_c, (0, 0, 0), int(30 * s), 0.0))
        blocks.append(([credit], f_c, (178, 186, 198), int(f_c.size * 1.3), 0.8))
    if contact:
        blocks.append(([""], f_c, (0, 0, 0), int(12 * s), 0.0))
        blocks.append(([contact], _font(int(30 * s), True), accent, int(f_c.size * 1.4), 1.0))
    total = sum(len(l) * lh for l, _, _, lh, _ in blocks)
    y0 = (H - total) / 2 - 20 * s
    pipe = FFmpegPipe(path, size, fps, crf=16)
    n = int(seconds * fps)
    for i in range(n):
        t = i / fps
        im = Image.fromarray(bd.next(place=0.5, strength=0.38, ramp_from=-1.0)).convert("RGBA")
        d = ImageDraw.Draw(im)
        y = y0
        for lines, font, col, lh, delay in blocks:
            a = float(np.clip((t - delay) / 0.6, 0, 1) * np.clip((seconds - t) / 0.4, 0, 1))
            ease = 1 - (1 - a) ** 3
            for ln in lines:
                if ln:
                    d.text((W / 2, y + lh / 2 + (1 - ease) * 18 * s), ln, font=font,
                           fill=tuple(col) + (int(255 * a),), anchor="mm")
                y += lh
        if note:
            a = float(np.clip((t - 1.1) / 0.6, 0, 1) * np.clip((seconds - t) / 0.4, 0, 1))
            for k, ln in enumerate(_wrap(d, note, f_n, int(0.86 * W))[:2]):
                d.text((W / 2, H - int((92 - 30 * k) * s)), ln, font=f_n, fill=(140, 148, 160, int(255 * a)),
                       anchor="mm")
        pipe.write(np.asarray(im.convert("RGB")))
    return pipe.close()


def render_text_cards(path: str, cards: list[dict], size=(1920, 1080), fps: int = 30,
                      seconds: float | list = 7.0, fade: float = 0.6, accent=(255, 176, 64),
                      backdrop=None) -> str:
    """Cartões de texto (kicker, título, tópicos, nota opcional) com entrada suave, em HD.

    cards = [{"kicker": "O DESAFIO", "headline": "...", "bullets": ["...", "..."], "footnote": "..."}]
    backdrop: None, quadro RGB (ex.: um quadro HD do reator) ou Backdrop (reator em movimento), mostrado
    esmaecido à direita."""
    from PIL import Image, ImageDraw
    W, H = size
    s = H / 1080.0
    pipe = FFmpegPipe(path, size, fps, crf=16)
    f_k, f_h, f_b, f_n = _font(int(30 * s), True), _font(int(62 * s), True), _font(int(38 * s)), _font(int(24 * s))
    secs = seconds if isinstance(seconds, (list, tuple)) else [seconds] * len(cards)
    bd = _backdrop(backdrop, size)
    wrap = _wrap

    for card, sec in zip(cards, secs):
        n = int(sec * fps)
        base = Image.new("RGBA", (8, 8))
        layers = []   # (y_alvo, imagem RGBA, atraso em s)
        x0, wmax = int(150 * s), int(W - 300 * s)
        y = int(230 * s)
        probe = ImageDraw.Draw(base)

        def layer(lines, font, color, gap, delay, bullet=False):
            nonlocal y
            h = int(font.size * 1.25)
            im = Image.new("RGBA", (W, h * len(lines) + int(gap)), (0, 0, 0, 0))
            d = ImageDraw.Draw(im)
            for k, ln in enumerate(lines):
                xo = x0 + (int(48 * s) if bullet else 0)
                if bullet and k == 0:
                    d.ellipse([x0 + int(6 * s), int(h * 0.42 - 8 * s), x0 + int(22 * s), int(h * 0.42 + 8 * s)],
                              fill=accent + (255,))
                d.text((xo, k * h), ln, font=font, fill=color)
            layers.append((y, im, delay))
            y += im.height

        layer([br(card.get("kicker", "")).upper()], f_k, accent + (255,), 18 * s, 0.0)
        layer(_wrap_balanced(probe, br(card["headline"]), f_h, wmax), f_h, (238, 242, 248, 255), 46 * s, 0.25)
        for j, bl in enumerate(card.get("bullets", [])):
            layer(wrap(probe, br(bl), f_b, wmax - int(48 * s)), f_b, (200, 208, 220, 255), 22 * s, 0.7 + 0.45 * j,
                  bullet=True)
        foot = br(card.get("footnote") or "")
        y_top = int(230 * s)
        shift = int((H - (y - y_top)) / 2 - y_top - 20 * s)      # centraliza o bloco na vertical
        layers = [(yl + shift, L, dl) for (yl, L, dl) in layers]
        bar = (y_top + shift, y + shift)
        icard = cards.index(card)
        for i in range(n):
            t = i / fps
            im = Image.fromarray(bd.next()).convert("RGBA")
            dr = ImageDraw.Draw(im)
            dr.rectangle([x0 - int(40 * s), bar[0], x0 - int(32 * s), bar[1]], fill=accent + (255,))
            if len(cards) > 1:   # indicador de progresso (bolinhas)
                for j in range(len(cards)):
                    cx = W - int((60 + 28 * (len(cards) - 1 - j)) * s)
                    r = int((7 if j == icard else 5) * s)
                    col = accent + (255,) if j == icard else (90, 98, 112, 255)
                    dr.ellipse([cx - r, H - int(60 * s) - r, cx + r, H - int(60 * s) + r], fill=col)
            for (yl, L, delay) in layers:
                a = float(np.clip((t - delay) / fade, 0, 1))
                a *= float(np.clip((sec - t) / fade, 0, 1))
                if a <= 0:
                    continue
                ease = 1 - (1 - a) ** 3
                Lc = L.copy()
                Lc.putalpha(Image.eval(L.getchannel("A"), lambda v, a=a: int(v * a)))
                im.alpha_composite(Lc, (0, int(yl + (1 - ease) * 24 * s)))
            if foot:
                a = float(np.clip((t - 1.2) / fade, 0, 1) * np.clip((sec - t) / fade, 0, 1))
                dr.text((x0, H - int(90 * s)), foot, font=f_n, fill=(140, 148, 160, int(255 * a)))
            pipe.write(np.asarray(im.convert("RGB")))
    return pipe.close()


# ------------------------------------------------------------ figuras com legenda
def _frames_of(src: str, seconds: float | None, fps: int):
    """Quadros RGB de um vídeo (decodificado pelo ffmpeg) ou de uma imagem repetida por `seconds`."""
    from PIL import Image
    if src.lower().endswith((".png", ".jpg", ".jpeg")):
        im = np.asarray(Image.open(src).convert("RGB"))
        for _ in range(int(round((seconds or 5.0) * fps))):
            yield im
        return
    ff = shutil.which("ffmpeg")
    probe = subprocess.run([ff, "-i", src], capture_output=True, text=True).stderr
    import re
    m = re.search(r", (\d{2,5})x(\d{2,5})", probe)
    w, h = int(m.group(1)), int(m.group(2))
    p = subprocess.Popen([ff, "-loglevel", "error", "-i", src, "-vf", f"fps={fps}", "-f", "rawvideo",
                          "-pix_fmt", "rgb24", "-"], stdout=subprocess.PIPE)
    n = w * h * 3
    while True:
        buf = p.stdout.read(n)
        if len(buf) < n:
            break
        yield np.frombuffer(buf, np.uint8).reshape(h, w, 3)
    p.wait()


def caption_segment(src: str, path: str, captions, chapter: str = "", size=(1920, 1080), fps: int = 30,
                    seconds: float | None = None, band: float = 0.14, accent=(255, 176, 64),
                    handle: str = "") -> str:
    """Coloca um vídeo ou imagem (figura do estudo) acima de uma faixa de legenda: rótulo do capítulo e
    textos que se alternam com transição suave. captions: lista de textos (distribuídos igualmente) ou
    [(t0, t1, texto)]; handle: assinatura discreta no canto da faixa (ex.: endereço do LinkedIn)."""
    from PIL import Image, ImageDraw
    W, H = size
    s = H / 1080.0
    bh = int(band * H)
    frames = list(_frames_of(src, seconds, fps))
    dur = len(frames) / fps
    caps = captions or []
    if caps and isinstance(caps[0], str):
        caps = even_captions(caps, dur)
    f_k, f_c = _font(int(24 * s), True), _font(int(34 * s))
    base = Image.new("RGBA", size, (13, 17, 23, 255))
    d0 = ImageDraw.Draw(base)
    d0.rectangle([0, H - bh, W, H], fill=(9, 12, 18, 255))
    d0.rectangle([int(60 * s), H - bh + int(22 * s), int(66 * s), H - int(22 * s)], fill=accent + (255,))
    if chapter:
        d0.text((int(84 * s), H - bh + int(20 * s)), br(chapter).upper(), font=f_k, fill=accent + (255,))
    if handle:
        d0.text((W - int(60 * s), H - bh + int(22 * s)), handle, font=_font(int(22 * s)), fill=(118, 126, 140, 255),
                anchor="ra")
    pipe = FFmpegPipe(path, size, fps, crf=16)
    area_h = H - bh
    for i, fr in enumerate(frames):
        im = base.copy()
        src_im = Image.fromarray(fr)
        sc = min(W / src_im.width, area_h / src_im.height)
        fit = src_im.resize((int(src_im.width * sc), int(src_im.height * sc)), Image.LANCZOS)
        im.paste(fit, ((W - fit.width) // 2, (area_h - fit.height) // 2))
        text, a = _caption_alpha(caps, i / fps)
        if text:
            layer = Image.new("RGBA", size, (0, 0, 0, 0))
            d = ImageDraw.Draw(layer)
            lines = _wrap(d, br(text), f_c, int(W - 180 * s))[:2]
            y = H - bh + int((56 if chapter else 30) * s)
            for k, ln in enumerate(lines):
                d.text((int(84 * s), y + k * int(f_c.size * 1.2)), ln, font=f_c, fill=(236, 240, 246, int(255 * a)))
            im.alpha_composite(layer)
        pipe.write(np.asarray(im.convert("RGB")))
    return pipe.close()


# --------------------------------------------------------------- números-chave
def render_kpi_card(path: str, title: str, kpis: list[dict], size=(1920, 1080), fps: int = 30,
                    seconds: float = 8.0, backdrop=None, note: str = "", kicker: str = "",
                    accent=(255, 176, 64)) -> str:
    """Cartão de números-chave com contagem animada.

    kpis = [{"value": 169, "fmt": "{:.0f}", "unit": "kg/h", "label": "plástico processado",
             "color": (255, 176, 64)}, ...] — value pode ser texto (sem animação)."""
    from PIL import Image, ImageDraw
    W, H = size
    s = H / 1080.0
    bd = _backdrop(backdrop, size)
    base = Image.new("RGBA", size, (0, 0, 0, 0))      # título e caixas, compostos sobre o fundo a cada quadro
    d0 = ImageDraw.Draw(base)
    y_t = int(170 * s)
    if kicker:
        d0.text((W // 2, y_t - int(48 * s)), br(kicker).upper(), font=_font(int(28 * s), True), fill=accent + (255,),
                anchor="mm")
    d0.text((W // 2, y_t + int(10 * s)), br(title), font=_font(int(56 * s), True), fill=(240, 244, 250, 255),
            anchor="mm")
    if note:
        d0.text((W // 2, H - int(70 * s)), br(note), font=_font(int(22 * s)), fill=(140, 148, 160, 255),
                anchor="mm")
    n = len(kpis)
    cols = 2 if n == 4 else n
    rows = int(math.ceil(n / cols))
    cw, ch = int(W * 0.40), int(250 * s)
    gx = (W - cols * cw) // (cols + 1)
    y0 = int(300 * s)
    boxes = []
    for k, kp in enumerate(kpis):
        r, c = divmod(k, cols)
        x = gx + c * (cw + gx)
        y = y0 + r * (ch + int(40 * s))
        d0.rounded_rectangle([x, y, x + cw, y + ch], radius=int(18 * s), fill=(18, 24, 33, 230),
                             outline=(40, 52, 68, 255), width=max(1, int(2 * s)))
        boxes.append((x, y))
    f_v, f_u, f_l = _font(int(92 * s), True), _font(int(40 * s), True), _font(int(30 * s))
    pipe = FFmpegPipe(path, size, fps, crf=16)
    N = int(seconds * fps)
    for i in range(N):
        t = i / fps
        im = Image.fromarray(bd.next(strength=0.35)).convert("RGBA")
        im.alpha_composite(base)
        d = ImageDraw.Draw(im)
        for k, (kp, (x, y)) in enumerate(zip(kpis, boxes)):
            a = float(np.clip((t - 0.3 - 0.35 * k) / 0.5, 0, 1) * np.clip((seconds - t) / 0.4, 0, 1))
            if a <= 0:
                continue
            col = tuple(kp.get("color", accent))
            v = kp["value"]
            if isinstance(v, (int, float)):
                prog = 1 - (1 - min(1.0, max(0.0, (t - 0.3 - 0.35 * k) / 1.4))) ** 3
                vs = br(kp.get("fmt", "{:.1f}").format(v * prog))
            else:
                vs = br(str(v))
            unit = kp.get("unit", "")
            wv = d.textlength(vs, font=f_v)
            wu = d.textlength(" " + unit, font=f_u) if unit else 0
            xs = x + (cw - wv - wu) / 2
            d.text((xs, y + int(40 * s)), vs, font=f_v, fill=col + (int(255 * a),))
            if unit:
                d.text((xs + wv, y + int(84 * s)), " " + unit, font=f_u, fill=col + (int(255 * a),))
            for j, ln in enumerate(_wrap(d, br(kp.get("label", "")), f_l, cw - int(40 * s))[:2]):
                d.text((x + cw / 2, y + int(170 * s) + j * int(36 * s)), ln, font=f_l,
                       fill=(200, 208, 220, int(255 * a)), anchor="mm")
        pipe.write(np.asarray(im.convert("RGB")))
    return pipe.close()

def render_still(src_png: str, path: str, seconds: float = 5.0, fps: int = 30):
    """Segmento de vídeo a partir de uma imagem fixa (1920×1080)."""
    ff = shutil.which("ffmpeg")
    subprocess.run([ff, "-y", "-loop", "1", "-framerate", str(fps), "-t", f"{seconds}", "-i", src_png,
                    "-vf", "scale=1920:1080,setsar=1,format=yuv420p", "-c:v", "libx264", "-crf", "16",
                    "-preset", "medium", path], check=True, capture_output=True)
    return path


def assemble(segments: list[str], out: str, fps: int = 30, size: tuple = (1920, 1080), crf: int = 16) -> str:
    """Concatena os segmentos em um MP4 final na resolução `size` (os de outra resolução são reescalados
    com Lanczos, preservando o aspecto)."""
    ff = shutil.which("ffmpeg")
    if ff is None:
        raise RuntimeError("ffmpeg não encontrado")
    segs = [p for p in segments if p and Path(p).exists()]
    cmd = [ff, "-y"]
    for p in segs:
        cmd += ["-i", p]
    W, H = size
    chains = "".join(f"[{i}:v]scale={W}:{H}:flags=lanczos:force_original_aspect_ratio=decrease,"
                     f"pad={W}:{H}:(ow-iw)/2:(oh-ih)/2,setsar=1,fps={fps},format=yuv420p[v{i}];"
                     for i in range(len(segs)))
    cmd += ["-filter_complex", chains + "".join(f"[v{i}]" for i in range(len(segs)))
            + f"concat=n={len(segs)}:v=1:a=0[v]", "-map", "[v]", "-c:v", "libx264", "-crf", str(crf),
            "-preset", "slow", "-tune", "film", "-movflags", "+faststart", out]
    subprocess.run(cmd, check=True, capture_output=True)
    return out

In [ ]:
%%writefile pyrokit/figures.py
"""Figuras do estudo (matplotlib, tema escuro, vírgula decimal) — 1920×1080 para o vídeo e o relatório."""
from __future__ import annotations

import math

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import ticker

from .video import br

BG, PANEL, GRID, TXT, MUTED = "#0d1117", "#131a24", "#2a3442", "#e8edf4", "#8b95a5"
ACCENT, COOL, GOOD = "#ffb040", "#6fc3ff", "#7ee0a2"


class _Comma(ticker.ScalarFormatter):
    def __call__(self, x, pos=None):
        return br(super().__call__(x, pos))


def _style(ax, title=None, xlabel=None, ylabel=None):
    ax.set_facecolor(PANEL)
    for s in ax.spines.values():
        s.set_color(GRID)
    ax.tick_params(colors=MUTED, labelsize=12)
    ax.grid(color=GRID, lw=0.6, alpha=0.6)
    for axis in (ax.xaxis, ax.yaxis):
        if type(axis.get_major_formatter()) is ticker.ScalarFormatter:
            axis.set_major_formatter(_Comma())
    if title:
        ax.set_title(br(title), color=TXT, fontsize=16, loc="left", pad=10, fontweight="bold")
    if xlabel:
        ax.set_xlabel(br(xlabel), color=MUTED, fontsize=13)
    if ylabel:
        ax.set_ylabel(br(ylabel), color=MUTED, fontsize=13)


def _fig(title, subtitle=None, size=(19.2, 10.8)):
    fig = plt.figure(figsize=size, dpi=100, facecolor=BG)
    fig.text(0.04, 0.935, br(title), color=TXT, fontsize=30, fontweight="bold")
    if subtitle:
        fig.text(0.04, 0.895, br(subtitle), color=MUTED, fontsize=17)
    return fig


def _save(fig, path):
    if path:
        fig.savefig(path, facecolor=fig.get_facecolor())
        plt.close(fig)
    return path or fig


# ------------------------------------------------------------ curva de capacidade
def capacity_figure(curve, cfd_point=None, path=None, coke_wall_C=520.0, title="Quanto plástico o reator processa",
                    subtitle=None):
    """curve: lista de CSTRResult (capacity_curve); cfd_point: kpis do reator 3D."""
    Tw = np.array([c.T_wall - 273.15 for c in curve])
    F = np.array([3600 * c.feed for c in curve])
    Tb = np.array([c.T - 273.15 for c in curve])
    Q = np.array([c.duty / 1e3 for c in curve])
    fig = _fig(title, subtitle or "reator ideal 0-D (linhas) e CFD 3D (ponto) · carga pós-consumo, parede aquecida")
    ax1 = fig.add_axes([0.06, 0.12, 0.40, 0.70])
    ax2 = fig.add_axes([0.54, 0.12, 0.38, 0.70])
    _style(ax1, "Vazão de plástico", "temperatura da parede [°C]", "kg/h")
    ax1.plot(Tw, F, color=ACCENT, lw=3.5)
    ax1.fill_between(Tw, 0, F.max() * 1.15, where=Tw >= coke_wall_C, color="#ff6b6b", alpha=0.12)
    ax1.text(coke_wall_C + 2, F.max() * 1.07, "risco de coque\nna parede", color="#ff9b9b", fontsize=12, va="top")
    _style(ax2, "Temperatura do fundido e calor", "temperatura da parede [°C]", "°C")
    ax2.plot(Tw, Tb, color=COOL, lw=3.5, label="fundido")
    ax2b = ax2.twinx()
    ax2b.plot(Tw, Q, color=GOOD, lw=2.5, ls="--", label="calor pela camisa")
    ax2b.tick_params(colors=MUTED, labelsize=12)
    ax2b.set_ylabel("kW", color=MUTED, fontsize=13)
    ax2b.yaxis.set_major_formatter(_Comma())
    for s in ax2b.spines.values():
        s.set_color(GRID)
    if cfd_point is not None:
        ax1.plot([cfd_point["T_wall_C"]], [cfd_point["feed_kgph"]], "o", ms=16, color="white", mec=ACCENT, mew=3,
                 zorder=5)
        ax1.annotate(br(f"CFD 3D: {cfd_point['feed_kgph']:.0f} kg/h"), (cfd_point["T_wall_C"], cfd_point["feed_kgph"]),
                     xytext=(-190, 30), textcoords="offset points", color=TXT, fontsize=15, fontweight="bold",
                     arrowprops=dict(arrowstyle="-", color=MUTED))
        ax2.plot([cfd_point["T_wall_C"]], [cfd_point["T_bulk_C"]], "o", ms=16, color="white", mec=COOL, mew=3, zorder=5)
        ax2.annotate(br(f"CFD 3D: {cfd_point['T_bulk_C']:.1f} °C"), (cfd_point["T_wall_C"], cfd_point["T_bulk_C"]),
                     xytext=(-210, 25), textcoords="offset points", color=TXT, fontsize=15, fontweight="bold",
                     arrowprops=dict(arrowstyle="-", color=MUTED))
    ax1.set_ylim(0, F.max() * 1.15)
    h1, l1 = ax2.get_legend_handles_labels()
    h2, l2 = ax2b.get_legend_handles_labels()
    leg = ax2.legend(h1 + h2, l1 + l2, loc="upper left", fontsize=13, facecolor=PANEL, edgecolor=GRID)
    for t in leg.get_texts():
        t.set_color(TXT)
    return _save(fig, path)


# ---------------------------------------------------------------- cortes 2D
def _plane(f, q, axis="y"):
    """Fatia vertical que passa pelo eixo (plano x–z, média das duas linhas centrais em y)."""
    q = np.asarray(q)
    j = q.shape[1] // 2
    return 0.5 * (q[:, j - 1, :] + q[:, j, :])


def sections_figure(reactor, path=None, title="Por dentro do reator", subtitle=None):
    """Plano vertical pelo eixo (referencial da fita): temperatura, massa molar e viscosidade, velocidade."""
    f = reactor.flow
    fl = (f.fluid > 0.5).cpu().numpy()
    x = f.xc
    z = f.zc
    T = reactor.T.cpu().numpy() - 273.15
    Mw = _bulk_Mw(reactor)
    eta = reactor._mu(f.gamma).cpu().numpy()
    ul, vl, wl = [a.cpu().numpy() for a in f.lab_velocity_centers()]
    U = np.sqrt(ul ** 2 + vl ** 2 + wl ** 2)
    m = _plane(f, fl) > 0.5
    fig = _fig(title, subtitle or "corte vertical pelo eixo, no referencial que gira com a fita")
    panels = [("Temperatura", T, "inferno", "°C", None),
              ("Massa molar média (Mw)", Mw, "viridis", "kg/mol", None),
              ("Viscosidade", np.log10(np.maximum(eta, 1e-6)), "magma", "log₁₀ Pa s", None),
              ("Velocidade (laboratório)", U, "cividis", "m/s", None)]
    ext = [x[0], x[-1], 0.0, reactor.tank.H_L]
    R = reactor.tank.R
    for k, (name, q, cmap, unit, _) in enumerate(panels):
        ax = fig.add_axes([0.04 + 0.24 * k, 0.10, 0.20, 0.72])
        P = np.where(m, _plane(f, q), np.nan)
        vals = P[np.isfinite(P)]
        lo, hi = np.percentile(vals, 1), np.percentile(vals, 99.5)
        im = ax.imshow(P.T, origin="lower", extent=ext, cmap=cmap, vmin=lo, vmax=hi, interpolation="bilinear",
                       aspect="equal")
        if name.startswith("Velocidade"):
            jj = slice(None, None, 3)
            Wp = _plane(f, wl)
            Up = _plane(f, ul)
            X, Z = np.meshgrid(x, z, indexing="ij")
            ax.quiver(X[jj, jj], Z[jj, jj], np.where(m, Up, 0)[jj, jj], np.where(m, Wp, 0)[jj, jj], color="white",
                      alpha=0.55, scale=6.0, width=0.004)
        _style(ax, name)
        ax.set_xlim(-R * 1.02, R * 1.02)
        ax.set_ylim(0, reactor.tank.H_L)
        ax.set_xticks([-R, 0, R])
        ax.set_xticklabels([br(f"{-R:.1f}"), "0", br(f"{R:.1f}")])
        ax.set_xlabel("r [m]", color=MUTED, fontsize=12)
        if k == 0:
            ax.set_ylabel("altura [m]", color=MUTED, fontsize=12)
        cb = fig.colorbar(im, ax=ax, orientation="horizontal", fraction=0.05, pad=0.08)
        cb.ax.tick_params(colors=MUTED, labelsize=11)
        cb.ax.xaxis.set_major_formatter(_Comma())
        cb.set_label(br(unit), color=MUTED, fontsize=12)
        cb.outline.set_edgecolor(GRID)
    return _save(fig, path)


def _bulk_Mw(reactor):
    import torch
    num = sum(s["Y"] * reactor._Mw(c, s) for (c, _), s in zip(reactor.items, reactor.state))
    den = sum(s["Y"] for s in reactor.state).clamp(min=1e-12)
    return (num / den).cpu().numpy()


# ----------------------------------------------------------------- mapa de parede
def wall_map_figure(reactor, path=None, title="Onde o calor entra", subtitle=None):
    """Fluxo de calor local na parede lateral, desenrolada (ângulo × altura), no referencial da fita."""
    import torch
    f = reactor.flow
    sc = reactor._last_sc
    q = sc.wall_flux(reactor.T) * reactor.cfg.rho0 * reactor.cp0          # W por célula
    fl = reactor.flm > 0
    side = fl & (f.r_c > reactor.tank.R - 1.5 * f.h) & (q > 0)
    X, Y = f._t(f.P_c[..., 0]), f._t(f.P_c[..., 1])
    th = torch.atan2(Y, X)[side].cpu().numpy()
    zz = f._t(f.P_c[..., 2])[side].cpu().numpy()
    qq = q[side].cpu().numpy()
    nth, nz = 48, f.nz
    H2, xe, ye = np.histogram2d(th, zz, bins=[nth, nz], range=[[-math.pi, math.pi], [0, reactor.tank.H_L]], weights=qq)
    A = 2 * math.pi * reactor.tank.R * reactor.tank.H_L / (nth * nz)           # área de cada célula do mapa
    qmap = H2 / A / 1e3                                                          # kW/m²
    fig = _fig(title, subtitle or "fluxo de calor na parede lateral, desenrolada (ângulo × altura), em kW/m²")
    ax = fig.add_axes([0.06, 0.12, 0.80, 0.70])
    im = ax.imshow(qmap.T, origin="lower", extent=[-180, 180, 0, reactor.tank.H_L], aspect="auto", cmap="inferno",
                   interpolation="bicubic")
    _style(ax, None, "ângulo em relação à fita [°]", "altura [m]")
    cax = fig.add_axes([0.88, 0.12, 0.018, 0.70])
    cb = fig.colorbar(im, cax=cax)
    cb.ax.tick_params(colors=MUTED, labelsize=12)
    cb.ax.yaxis.set_major_formatter(_Comma())
    cb.set_label("kW/m²", color=MUTED, fontsize=13)
    cb.outline.set_edgecolor(GRID)
    return _save(fig, path)


# --------------------------------------------------------------------- TGA
def tga_figure(path=None, title="Cinética: termogravimetria a 10 K/min", subtitle=None):
    from . import kinetics as kin
    fig = _fig(title, subtitle or "cisão aleatória (L = 2), E de Aboulkas et al. (2010); marcas = picos da literatura")
    ax = fig.add_axes([0.07, 0.12, 0.88, 0.70])
    _style(ax, None, "temperatura [°C]", "taxa de perda de massa [%/°C]")
    colors = {"HDPE": ACCENT, "LDPE": "#ffd27a", "PP": COOL, "PS": "#c49bff", "PET": GOOD}
    for key, col in colors.items():
        p = kin.POLYMERS[key]
        T, a, d = kin.tga(p, 10.0)
        ax.plot(T - 273.15, 100 * d, color=col, lw=3, label=f"{key} ({p.name})")
        ax.axvline(p.tpeak_10, color=col, lw=1, ls=":", alpha=0.8)
    ax.set_xlim(330, 560)
    leg = ax.legend(fontsize=13, facecolor=PANEL, edgecolor=GRID, loc="upper left")
    for t in leg.get_texts():
        t.set_color(TXT)
    return _save(fig, path)

In [ ]:
%%writefile pyrokit/validation.py
"""Validação em três níveis (escada do documento de projeto), cada item com alvo, tolerância e veredito.

L1 · componentes do CFD: Couette newtoniano e de lei de potência (solução exata), Kp da fita helicoidal em
     regime laminar (correlação, ±20%, abaixo do limite de Couette), coeficiente interno contra Nagata
     (±30%) e fechamento dos balanços de massa e energia do reator 3D.
L2 · química e propriedades: picos de DTG (consenso a 10 K/min e Westerhout et al. 1997), PET a 2 K/min
     (Dubdub & Al-Yaari 2023), Kissinger, deslocamento do pico por duplicação da taxa, constantes
     isotérmicas de Westerhout, densidades da ISO 1133, viscosidade das ceras POLYWAX e a taxa de
     volatilização por massa retida de Murata et al. (2002).
L3 · sistema: CFD 3D × reator ideal 0-D (T dentro de 3 K, vazão dentro de 10%) e faixas de produtos.

Cada verificação devolve um dicionário {nível, item, valor, alvo, ok, nota}; `dashboard` desenha a tabela.
"""
from __future__ import annotations

import math
import time

import numpy as np

from . import kinetics as kin
from . import rheology as rh

TARGETS = {
    "kp_ribbon": 269.0, "kp_tol": 0.20, "kp_couette_bound": 653.0, "h_tol": 0.30,
    "westerhout_PS": 410.9, "westerhout_HDPE": 474.0, "PET_2K_K": 680.0,
    "westerhout_k": {400: 1.61e-4, 420: 5.01e-4, 440: 1.46e-3},
    "iso1133": {"PE": (190.0, 763.6), "PP": (230.0, 738.6)},
    "polywax_mPa_s": {655: 7.0, 1000: 15.0, 2000: 55.0, 3000: 130.0},
    "murata_per_h": (0.3, 1.0),
    "slate": {"gás": (0.04, 0.10), "líquido": (0.85, 0.93), "coque": (0.005, 0.04)},
}


def check(level, item, value, target, ok, note="", fmt="{:.3g}"):
    return {"nível": level, "item": item, "valor": value if isinstance(value, str) else fmt.format(value),
            "alvo": target, "ok": bool(ok), "nota": note}


# --------------------------------------------------------------------- L2
def chemistry_checks() -> list:
    out = []
    for key in ("HDPE", "LDPE", "PP", "PS"):
        p = kin.POLYMERS[key]
        tp = kin.tga_peak(p, 10.0)
        out.append(check("L2", f"pico DTG {key}, 10 K/min (calibração)", tp, f"{p.tpeak_10:.0f} ± 3 °C",
                         abs(tp - p.tpeak_10) <= 3.0, "consenso da literatura", "{:.1f} °C"))
    tp = kin.tga_peak(kin.POLYMERS["PS"], 10.0)
    out.append(check("L2", "pico DTG PS × Westerhout (1997)", tp, f"{TARGETS['westerhout_PS']} ± 12 °C",
                     abs(tp - TARGETS["westerhout_PS"]) <= 12, "independente da calibração", "{:.1f} °C"))
    tp = kin.tga_peak(kin.POLYMERS["HDPE"], 10.0)
    out.append(check("L2", "pico DTG HDPE × Westerhout (1997)", tp, f"{TARGETS['westerhout_HDPE']} ± 5 °C",
                     abs(tp - TARGETS["westerhout_HDPE"]) <= 5, "", "{:.1f} °C"))
    tpet = kin.tga_peak(kin.POLYMERS["PET"], 2.0) + 273.15
    out.append(check("L2", "pico DTG PET a 2 K/min", tpet, f"{TARGETS['PET_2K_K']:.0f} ± 5 K",
                     abs(tpet - TARGETS["PET_2K_K"]) <= 5, "Dubdub & Al-Yaari (2023)", "{:.0f} K"))
    for key in ("HDPE", "PP"):
        p = kin.POLYMERS[key]
        E, Tp = kin.kissinger(p)
        out.append(check("L2", f"Kissinger {key}: E recuperada", E / 1e3, f"{p.E / 1e3:.0f} kJ/mol (±5%)",
                         abs(E - p.E) / p.E < 0.05, "consistência da cinética RS-L2", "{:.0f} kJ/mol"))
        shift = float(np.mean(np.diff([kin.tga_peak(p, b) for b in (5, 10, 20)])))
        out.append(check("L2", f"deslocamento do pico por 2× na taxa ({key})", shift, "12–16 K",
                         11.0 <= shift <= 17.0, "", "{:.1f} K"))
    p = kin.POLYMERS["HDPE"]
    errs = [float(kin.rate(p, Tc + 273.15)) / k - 1 for Tc, k in TARGETS["westerhout_k"].items()]
    worst = max(errs, key=abs)
    out.append(check("L2", "k isotérmico HDPE 400–440 °C × Westerhout", f"{100 * worst:+.0f}% (pior)",
                     "dentro de ±40%", abs(worst) <= 0.40, "constantes de 1ª ordem medidas"))
    for poly, (Tc, ref) in TARGETS["iso1133"].items():
        key = "HDPE" if poly == "PE" else "PP"
        rho = float(rh.density(key, Tc + 273.15))
        out.append(check("L2", f"densidade {poly} a {Tc:.0f} °C (ISO 1133)", rho, f"{ref} kg/m³ ± 2%",
                         abs(rho / ref - 1) <= 0.02, "", "{:.1f} kg/m³"))
    errs = []
    for M, mpas in TARGETS["polywax_mPa_s"].items():
        model = 1e3 * 0.015 * (M / 1080.0) ** 1.9
        errs.append(model / mpas - 1)
    worst = max(errs, key=abs)
    out.append(check("L2", "viscosidade de ceras PE a 149 °C (POLYWAX)", f"{100 * worst:+.0f}% (pior)",
                     "±30%", abs(worst) <= 0.30, "ramo de cera da reologia"))
    return out


# --------------------------------------------------------------------- L1
def kp_ribbon(tank_fn, n_diam: int = 40, Re: float = 8.0, device: str = "auto", t_max_visc: float = 2.5,
              verbose: bool = False):
    """Kp = Np·Re da fita em regime laminar, por simulação newtoniana (μ escolhido para dar Re)."""
    import torch
    from .flow import FlowConfig, TankFlow
    tank = tank_fn()
    N, d = tank.rpm / 60.0, tank.impeller_diameter
    rho = 640.0
    mu = rho * N * d * d / Re
    f = TankFlow(tank, FlowConfig(n_diam=n_diam, rho=rho, device=device, spin0=0.8),
                 lambda g: torch.full_like(g, mu))
    t_end = t_max_visc * tank.R ** 2 / (mu / rho)
    t0 = time.perf_counter()
    last = None
    while f.time < t_end:
        f.step()
        f.record()
        if f.step_n % 500 == 0:
            P = f.power(200)
            if last is not None and abs(P - last) < 0.002 * abs(P):
                break
            last = P
            if verbose:
                print(f"  t = {f.time:.2f} s, Np·Re = {P / (rho * N ** 3 * d ** 5) * Re:.1f}")
    Np = f.power(200) / (rho * N ** 3 * d ** 5)
    return {"Kp": Np * Re, "Np": Np, "Re": Re, "steps": f.step_n, "wall_s": time.perf_counter() - t0,
            "torque_balance": -f.torque_wall / max(f.torque_imp, 1e-12)}


def flow_checks(kp: dict | None = None, reactor_kpis: dict | None = None, h_nagata: float | None = None) -> list:
    out = []
    if kp is not None:
        k = kp["Kp"]
        out.append(check("L1", f"Kp da fita (laminar, Re = {kp['Re']:.0f})", k,
                         f"{TARGETS['kp_ribbon']:.0f} ± 20% e < {TARGETS['kp_couette_bound']:.0f}",
                         abs(k / TARGETS["kp_ribbon"] - 1) <= TARGETS["kp_tol"] and k < TARGETS["kp_couette_bound"],
                         "correlação de Delaplace / limite de Couette", "{:.0f}"))
    if reactor_kpis is not None:
        kp_ = reactor_kpis
        out.append(check("L1", "balanço de massa (alimentação = vapor + purga)", f"{100 * kp_['mass_balance_err']:+.2f}%",
                         "|erro| < 1%", abs(kp_["mass_balance_err"]) < 0.01, "controle de nível"))
        out.append(check("L1", "balanço de energia (parede + agitação = carga + pirólise)",
                         f"{100 * kp_['energy_balance_err']:+.2f}%", "|erro| < 3%",
                         abs(kp_["energy_balance_err"]) < 0.03, "fluxos conservativos"))
        if h_nagata:
            h = kp_["h_mean"]
            out.append(check("L1", "coeficiente interno h × Nagata (fita, turbulento)", h,
                             f"{h_nagata:.0f} W/m²K ± 30%", abs(h / h_nagata - 1) <= TARGETS["h_tol"],
                             "lei de parede de Kader + SST", "{:.0f} W/m²K"))
    return out


# --------------------------------------------------------------------- L3
def system_checks(reactor, reactor_kpis: dict) -> list:
    """CFD 3D × reator ideal 0-D com o MESMO coeficiente interno médio (isola o efeito da mistura não
    ideal: pluma fria da alimentação, gradientes junto à parede) e faixas de produtos."""
    from .chemistry import product_slate, solve_cstr
    kp = reactor_kpis
    cfg = reactor.cfg
    ideal = solve_cstr(reactor.feed, reactor.V_liq, reactor.A_wall, kp["h_mean"], T_wall=kp["T_wall_C"] + 273.15,
                       feed_rate=None if cfg.feed_kgph is None else cfg.feed_kgph / 3600.0, rho=cfg.rho0)
    out = []
    dT = kp["T_bulk_C"] - (ideal.T - 273.15)
    out.append(check("L3", "T do fundido: CFD 3D × reator ideal 0-D (mesmo h)", f"{dT:+.1f} K", "dentro de ±3 K",
                     abs(dT) <= 3.0, "efeito da mistura não ideal"))
    dF = kp["feed_kgph"] / (3600 * ideal.feed) - 1
    out.append(check("L3", "vazão processada: CFD × 0-D (mesmo h)", f"{100 * dF:+.1f}%", "dentro de ±10%",
                     abs(dF) <= 0.10, ""))
    vol_h = kp["vapour_kgph"] / (cfg.rho0 * reactor.V_liq)
    lo, hi = TARGETS["murata_per_h"]
    out.append(check("L3", "volatilização por massa retida", vol_h, f"{lo}–{hi} 1/h (Murata 2002)",
                     lo <= vol_h <= hi, "reator contínuo agitado, 400–420 °C", "{:.2f} 1/h"))
    # faixas da literatura valem para reatores a 425–450 °C: o modelo de produtos é avaliado a 435 °C
    sl = product_slate(708.15)
    liq = sl["óleo"] + sl["cera"]
    for name, val in (("gás", sl["gás"]), ("líquido", liq), ("coque", sl["coque"])):
        lo, hi = TARGETS["slate"][name]
        out.append(check("L3", f"produtos a 435 °C: {name}", f"{100 * val:.1f}%", f"{100 * lo:.1f}–{100 * hi:.0f}%",
                         lo - 1e-9 <= val <= hi + 1e-9, "faixa da literatura (seletividade calibrada)"))
    return out, ideal


# ------------------------------------------------------------------ figura
def dashboard(checks: list, path: str | None = None, title: str = "Validação", size=(19.2, 10.8)):
    """Tabela de verificações com veredito colorido (PASSOU / FORA)."""
    import matplotlib.pyplot as plt
    from .video import br
    fig = plt.figure(figsize=size, dpi=100, facecolor="#0d1117")
    ax = fig.add_axes([0.03, 0.03, 0.94, 0.86])
    ax.axis("off")
    fig.text(0.03, 0.94, title, color="#eef2f8", fontsize=30, fontweight="bold")
    n_ok = sum(c["ok"] for c in checks)
    fig.text(0.97, 0.945, f"{n_ok}/{len(checks)} dentro do alvo", color="#7ee0a2" if n_ok == len(checks) else "#ffb040",
             fontsize=22, ha="right", fontweight="bold")
    cols = ["", "verificação", "resultado", "alvo", ""]
    xs = [0.0, 0.035, 0.53, 0.68, 0.93]
    n = len(checks)
    lh = min(0.06, 0.95 / (n + 1))
    fs = max(10, min(17, int(lh * 300)))
    for j, c in enumerate(cols):
        ax.text(xs[j], 1.0, c, color="#8b95a5", fontsize=fs, fontweight="bold", va="top", transform=ax.transAxes)
    for i, c in enumerate(checks):
        y = 1.0 - (i + 1.2) * lh
        if i % 2 == 0:
            ax.add_patch(plt.Rectangle((0, y - 0.32 * lh), 1, lh, transform=ax.transAxes, color="#151b24", zorder=0))
        ax.text(xs[0], y, c["nível"], color="#ffb040", fontsize=fs, va="center", transform=ax.transAxes,
                fontweight="bold")
        ax.text(xs[1], y, br(c["item"]), color="#e6eaf0", fontsize=fs, va="center", transform=ax.transAxes)
        ax.text(xs[2], y, br(str(c["valor"])), color="#ffffff", fontsize=fs, va="center", transform=ax.transAxes,
                fontweight="bold")
        ax.text(xs[3], y, br(str(c["alvo"])), color="#aab3c2", fontsize=fs, va="center", transform=ax.transAxes)
        ax.text(xs[4], y, "PASSOU" if c["ok"] else "FORA", color="#7ee0a2" if c["ok"] else "#ff6b6b",
                fontsize=fs, va="center", transform=ax.transAxes, fontweight="bold")
    if path:
        fig.savefig(path, facecolor=fig.get_facecolor())
        plt.close(fig)
    return fig

In [ ]:
%%writefile pyrokit/render3d.py
"""Imagem 3D do reator em corte (ray marching de funções de distância em PyTorch; GPU quando houver).

Cena (referencial do laboratório; o agitador gira de θ):
  * vaso encamisado de aço com um quarto removido de frente para a câmera (corte em cunha);
  * dupla fita helicoidal (a mesma função de distância usada na malha), em aço polido;
  * o líquido aparece como um sólido opaco cortado pela cunha: as faces do corte e a superfície livre
    são coloridas pelo campo escolhido (temperatura, massa molar, viscosidade…), amostrado na solução
    do referencial girante — no laboratório, a solução estacionária só gira junto com a fita.

Sem malha poligonal: a geometria é traçada direto das SDFs (sphere tracing), com normais por diferenças,
oclusão ambiente barata, especular de Blinn-Phong e tone mapping. Antisserrilhado por supersampling.
"""
from __future__ import annotations

import math

import numpy as np
import torch
import torch.nn.functional as F


def _lut(name: str, n: int = 256, dev=None):
    import matplotlib
    cm = matplotlib.colormaps[name]
    return torch.tensor(cm(np.linspace(0, 1, n))[:, :3], dtype=torch.float32, device=dev)


def _rotz(p, ang):
    c, s = math.cos(ang), math.sin(ang)
    x, y = p[..., 0], p[..., 1]
    return torch.stack([c * x - s * y, s * x + c * y, p[..., 2]], -1)


class Cutaway:
    """Renderizador do reator em corte. fields: {nome: tensor [nx, ny, nz] nos centros da malha}."""

    MAT_LIQ, MAT_IMP, MAT_SHELL, MAT_JACKET = 0, 1, 2, 3

    def __init__(self, tank, xc, yc, zc, device=None, wall: float = 0.012, jacket_gap: float = 0.03,
                 jacket: float = 0.010, cut_deg: float = 100.0):
        self.tank = tank
        self.dev = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
        self.R, self.H = tank.R, tank.H_L
        self.wall, self.jg, self.jt = wall, jacket_gap, jacket
        self.H_top = self.H + 0.10 * tank.T           # o vaso continua acima do nível do líquido
        self.cut = math.radians(cut_deg)
        self.xc, self.yc, self.zc = (float(xc[0]), float(xc[-1])), (float(yc[0]), float(yc[-1])), \
            (float(zc[0]), float(zc[-1]))
        self.fields = {}

    # ------------------------------------------------------------------ campos
    def set_field(self, name, values, cmap="inferno", vmin=None, vmax=None, log=False, fluid=None):
        v = torch.as_tensor(values, dtype=torch.float32, device=self.dev)
        if fluid is not None:   # estende o campo um pouco para dentro das paredes (amostragem na borda)
            from .ops import extend_into_solid
            v = extend_into_solid(v, torch.as_tensor(fluid, device=self.dev) > 0.5, passes=3)
        if log:
            v = torch.log10(v.clamp(min=1e-30))
        lo = float(v.min()) if vmin is None else (math.log10(vmin) if log else vmin)
        hi = float(v.max()) if vmax is None else (math.log10(vmax) if log else vmax)
        self.fields[name] = dict(vol=v.permute(2, 1, 0)[None, None].contiguous(), lut=_lut(cmap, dev=self.dev),
                                 lo=lo, hi=hi, log=log, cmap=cmap)

    def _sample(self, name, p_rot):
        f = self.fields[name]
        g = torch.stack([(p_rot[..., 0] - self.xc[0]) / (self.xc[1] - self.xc[0]) * 2 - 1,
                         (p_rot[..., 1] - self.yc[0]) / (self.yc[1] - self.yc[0]) * 2 - 1,
                         (p_rot[..., 2] - self.zc[0]) / (self.zc[1] - self.zc[0]) * 2 - 1], -1)
        val = F.grid_sample(f["vol"], g.view(1, -1, 1, 1, 3), mode="bilinear", padding_mode="border",
                            align_corners=True).view(-1)
        x = ((val - f["lo"]) / max(f["hi"] - f["lo"], 1e-12)).clamp(0, 1)
        idx = (x * (f["lut"].shape[0] - 1)).long()
        return f["lut"][idx]

    # ------------------------------------------------------------------- SDFs
    def _wedge(self, p, phi_c):
        """SDF (negativa dentro) da cunha removida, centrada no azimute phi_c e com abertura self.cut."""
        a1, a2 = phi_c - 0.5 * self.cut, phi_c + 0.5 * self.cut
        n1 = (-math.sin(a1), math.cos(a1))           # normais para dentro da cunha
        n2 = (math.sin(a2), -math.cos(a2))
        d1 = p[..., 0] * n1[0] + p[..., 1] * n1[1]
        d2 = p[..., 0] * n2[0] + p[..., 1] * n2[1]
        return torch.maximum(-d1, -d2)

    def _scene(self, p, theta, phi_c):
        r = torch.hypot(p[..., 0], p[..., 1])
        z = p[..., 2]
        wedge = self._wedge(p, phi_c)
        # líquido (cilindro) sem a cunha
        liq = torch.maximum(torch.maximum(r - self.R, torch.maximum(-z, z - self.H)), -wedge)
        # agitador (no referencial dele) e eixo até o motor
        pr = _rotz(p, -theta)
        imp = self.tank.impeller.sdf(pr)
        imp = torch.maximum(imp, z - (self.H_top + 0.25 * self.tank.T))
        # vaso: parede e fundo, sem a cunha
        Ro = self.R + self.wall
        outer = torch.maximum(r - Ro, torch.maximum(-self.wall - z, z - self.H_top))
        inner = torch.maximum(r - self.R, torch.maximum(-z, z - self.H_top - 1.0))
        shell = torch.maximum(torch.maximum(outer, -inner), -wedge)
        # camisa: segunda casca, mais baixa, com folga
        Rj0, Rj1 = Ro + self.jg, Ro + self.jg + self.jt
        jac = torch.maximum(torch.maximum(r - Rj1, Rj0 - r), torch.maximum(0.06 * self.H - z, z - 0.96 * self.H))
        jac = torch.maximum(jac, -wedge)
        d = torch.stack([liq, imp, shell, jac], -1)
        dmin, mat = d.min(-1)
        return dmin, mat

    # ------------------------------------------------------------------ câmera
    def _rays(self, W, H, az, el, dist, fov, target):
        az, el = math.radians(az), math.radians(el)
        eye = torch.tensor([target[0] + dist * math.cos(el) * math.cos(az),
                            target[1] + dist * math.cos(el) * math.sin(az),
                            target[2] + dist * math.sin(el)], device=self.dev)
        tgt = torch.tensor(target, device=self.dev, dtype=torch.float32)
        fwd = F.normalize(tgt - eye, dim=0)
        right = F.normalize(torch.linalg.cross(fwd, torch.tensor([0.0, 0.0, 1.0], device=self.dev)), dim=0)
        up = torch.linalg.cross(right, fwd)
        th = math.tan(math.radians(fov) / 2)
        xs = (torch.arange(W, device=self.dev) + 0.5) / W * 2 - 1
        ys = 1 - (torch.arange(H, device=self.dev) + 0.5) / H * 2
        Y, X = torch.meshgrid(ys, xs, indexing="ij")
        d = fwd + X[..., None] * th * (W / H) * right + Y[..., None] * th * up
        return eye.expand(H, W, 3).reshape(-1, 3), F.normalize(d, dim=-1).reshape(-1, 3)

    # ---------------------------------------------------------------- desenho
    @torch.no_grad()
    def render(self, field: str, theta: float = 0.0, az: float = -60.0, el: float = 24.0, dist: float | None = None,
               size=(1920, 1080), ssaa: float = 1.5, fov: float = 32.0, background=None, max_steps: int = 140,
               exposure: float = 1.0) -> np.ndarray:
        W0, H0 = size
        W, H = int(W0 * ssaa), int(H0 * ssaa)
        T = self.tank.T
        dist = dist or 3.1 * T
        target = (0.0, 0.0, 0.47 * self.H)
        o, d = self._rays(W, H, az, el, dist, fov, target)
        phi_c = math.radians(az)                       # a cunha aponta para a câmera
        n = o.shape[0]
        t = torch.zeros(n, device=self.dev)
        hit = torch.zeros(n, dtype=torch.bool, device=self.dev)
        alive = torch.ones(n, dtype=torch.bool, device=self.dev)
        far = dist + 3 * T
        eps = 4e-4 * T
        idx = torch.arange(n, device=self.dev)
        for _ in range(max_steps):
            ia = idx[alive]
            if ia.numel() == 0:
                break
            p = o[ia] + t[ia, None] * d[ia]
            dd, _ = self._scene(p, theta, phi_c)
            close = dd < eps
            hit[ia[close]] = True
            t[ia] = t[ia] + dd.clamp(min=0.25 * eps) * 0.9
            gone = close | (t[ia] > far)
            alive[ia[gone]] = False
        img = self._background(H, W) if background is None else background
        img = img.reshape(-1, 3).clone()
        ih = idx[hit]
        if ih.numel():
            p = o[ih] + t[ih, None] * d[ih]
            _, mat = self._scene(p, theta, phi_c)
            e = 5e-4 * T
            nrm = torch.zeros_like(p)
            for k in range(3):
                dp = torch.zeros(3, device=self.dev)
                dp[k] = e
                nrm[:, k] = self._scene(p + dp, theta, phi_c)[0] - self._scene(p - dp, theta, phi_c)[0]
            nrm = F.normalize(nrm, dim=-1)
            col = self._shade(p, nrm, -d[ih], mat, field, theta, phi_c)
            img[ih] = col
        img = img.reshape(H, W, 3)
        img = 1 - torch.exp(-exposure * img * 1.6)                       # tone mapping suave
        img = img.clamp(0, 1) ** (1 / 2.2)
        if ssaa != 1.0:
            img = F.interpolate(img.permute(2, 0, 1)[None], size=(H0, W0), mode="area")[0].permute(1, 2, 0)
        return (img * 255).round().byte().cpu().numpy()

    def _background(self, H, W):
        yy = torch.linspace(0, 1, H, device=self.dev)[:, None, None]
        top = torch.tensor([0.020, 0.028, 0.045], device=self.dev)
        bot = torch.tensor([0.050, 0.062, 0.085], device=self.dev)
        bg = top + (bot - top) * yy
        return bg.expand(H, W, 3)

    def _ao(self, p, nrm, theta, phi_c):
        T = self.tank.T
        occ = torch.zeros(p.shape[0], device=self.dev)
        for k, h in enumerate((0.01, 0.025, 0.05, 0.09)):
            hh = h * T
            dd = self._scene(p + nrm * hh, theta, phi_c)[0]
            occ = occ + (hh - dd).clamp(min=0) / hh * 0.5 ** k
        return (1 - 0.55 * occ).clamp(0.25, 1.0)

    def _shade(self, p, nrm, view, mat, field, theta, phi_c):
        dev = self.dev
        L1 = F.normalize(torch.tensor([0.4, -0.6, 0.9], device=dev), dim=0)    # luz principal
        L2 = F.normalize(torch.tensor([-0.7, 0.5, 0.35], device=dev), dim=0)   # contraluz fria
        ndl1 = (nrm @ L1).clamp(min=0)[:, None]
        ndl2 = (nrm @ L2).clamp(min=0)[:, None]
        hv = F.normalize(L1 + view, dim=-1)
        spec = (nrm * hv).sum(-1).clamp(min=0)[:, None]
        ao = self._ao(p, nrm, theta, phi_c)[:, None]
        fres = (1 - (nrm * view).sum(-1).clamp(0, 1))[:, None] ** 5
        out = torch.zeros_like(p)
        # líquido: cor do campo, levemente sombreada (o corte é "pintado" pelo resultado)
        m = mat == self.MAT_LIQ
        if m.any():
            c = self._sample(field, _rotz(p[m], -theta)) if field in self.fields else \
                torch.tensor([0.9, 0.5, 0.2], device=dev).expand(int(m.sum()), 3)
            lin = c ** 2.2
            out[m] = lin * (0.55 + 0.45 * ndl1[m]) * ao[m] * 1.15 + 0.06 * spec[m] ** 40
        # agitador: aço polido
        m = mat == self.MAT_IMP
        if m.any():
            base = torch.tensor([0.56, 0.58, 0.62], device=dev)
            out[m] = (base * (0.10 + 0.70 * ndl1[m] + 0.25 * ndl2[m]) * ao[m] + 0.9 * spec[m] ** 60
                      + 0.25 * fres[m] * torch.tensor([0.6, 0.7, 0.9], device=dev))
        # vaso: aço escovado; faces internas mais claras
        m = mat == self.MAT_SHELL
        if m.any():
            r = torch.hypot(p[m, 0], p[m, 1])
            inside = (r < self.R + 0.5 * self.wall)[:, None]
            base = torch.where(inside, torch.tensor([0.36, 0.38, 0.42], device=dev),
                               torch.tensor([0.24, 0.26, 0.30], device=dev))
            out[m] = base * (0.12 + 0.75 * ndl1[m] + 0.2 * ndl2[m]) * ao[m] + 0.35 * spec[m] ** 30
        # camisa de aquecimento: cobre/laranja
        m = mat == self.MAT_JACKET
        if m.any():
            base = torch.tensor([0.62, 0.30, 0.12], device=dev)
            out[m] = base * (0.12 + 0.75 * ndl1[m] + 0.2 * ndl2[m]) * ao[m] + 0.5 * spec[m] ** 40
        return out

In [ ]:
%%writefile pyrokit/story.py
"""Segmentos de vídeo do reator: a fita girando em corte, com o campo escolhido e textos sobrepostos.

No referencial da fita a solução é estacionária; no laboratório ela só gira junto com a fita. Cada quadro
é então a mesma solução girada de Ω·t (em câmera lenta, para o olho acompanhar), com a câmera orbitando
devagar. Os textos (título, barra de cores, legendas e números grandes) são desenhados com PIL.
"""
from __future__ import annotations

import math

import numpy as np

from .video import FFmpegPipe, _caption_alpha, _font, br, draw_lower_third, even_captions


def colorbar_strip(cw, field, label, size, s, fmt="{:.0f}"):
    """Barra de cores vertical (PIL RGBA) com rótulo e valores do campo renderizado."""
    from PIL import Image, ImageDraw
    f = cw.fields[field]
    lut = (f["lut"].cpu().numpy() * 255).astype(np.uint8)
    h, w = int(360 * s), int(22 * s)
    im = Image.new("RGBA", (int(220 * s), h + int(90 * s)), (0, 0, 0, 0))
    d = ImageDraw.Draw(im)
    col = np.repeat(lut[::-1][np.linspace(0, len(lut) - 1, h).astype(int)][:, None, :], w, 1)
    im.paste(Image.fromarray(col), (0, int(50 * s)))
    d.text((0, 0), br(label), font=_font(int(24 * s), True), fill=(236, 240, 246, 255))
    lo, hi = f["lo"], f["hi"]
    for k, frac in enumerate((1.0, 0.5, 0.0)):
        v = lo + frac * (hi - lo)
        v = 10 ** v if f["log"] else v
        txt = br(fmt.format(v)) if not f["log"] else br(f"{v:.0e}".replace("e-0", "e-").replace("e+0", "e"))
        y = int(50 * s) + int((1 - frac) * (h - 1))
        d.text((w + int(12 * s), y), txt, font=_font(int(20 * s)), fill=(200, 208, 220, 255), anchor="lm")
    return im


def render_rotation(cw, path, field, title="", subtitle="", kicker="", captions=None, cbar_label="",
                    cbar_fmt="{:.0f}", seconds=8.0, fps=30, size=(1920, 1080), turns=0.5, az0=-70.0, az1=-40.0,
                    el=24.0, ssaa=1.25, big=None, accent=(255, 176, 64), handle="", clean_path=None, progress=True):
    """Vídeo da fita girando `turns` voltas em `seconds` s com a câmera orbitando de az0 a az1.

    big: lista de (valor, rótulo) mostrados em destaque à direita (o resultado aparece logo de cara)."""
    from PIL import Image, ImageDraw
    W, H = size
    s = H / 1080.0
    n = int(round(seconds * fps))
    caps = captions or []
    if caps and isinstance(caps[0], str):
        caps = even_captions(caps, seconds)
    static = Image.new("RGBA", size, (0, 0, 0, 0))
    d = ImageDraw.Draw(static)
    y = int(40 * s)
    if kicker:
        d.text((int(60 * s), y), br(kicker).upper(), font=_font(int(24 * s), True), fill=accent + (255,))
        y += int(38 * s)
    if title:
        d.text((int(58 * s), y), br(title), font=_font(int(46 * s), True), fill=(236, 240, 246, 255))
    if subtitle:
        d.text((int(60 * s), y + int(62 * s)), br(subtitle), font=_font(int(24 * s)), fill=(150, 160, 175, 255))
    if cbar_label and field in cw.fields:
        static.alpha_composite(colorbar_strip(cw, field, cbar_label, size, s, cbar_fmt), (int(60 * s), int(330 * s)))
    if handle:
        d.text((W - int(60 * s), H - int(40 * s)), handle, font=_font(int(22 * s)), fill=(118, 126, 140, 255),
               anchor="rb")
    pipe = FFmpegPipe(path, size, fps, crf=16)
    clean = FFmpegPipe(clean_path, size, fps, crf=20) if clean_path else None
    for i in range(n):
        t = i / fps
        u = t / max(seconds, 1e-9)
        ease = 0.5 - 0.5 * math.cos(math.pi * u)
        theta = 2 * math.pi * turns * u
        az = az0 + (az1 - az0) * ease
        rgb = cw.render(field, theta=theta, az=az, el=el, size=size, ssaa=ssaa)
        if clean is not None:
            clean.write(rgb)
        im = Image.fromarray(rgb).convert("RGBA")
        im.alpha_composite(static)
        if big:
            dd = ImageDraw.Draw(im)
            x = W - int(70 * s)
            yb = int(250 * s)
            for k, (val, lab) in enumerate(big):
                a = float(np.clip((t - 0.25 - 0.35 * k) / 0.5, 0, 1) * np.clip((seconds - t) / 0.4, 0, 1))
                if a <= 0:
                    continue
                dd.text((x, yb + k * int(190 * s)), br(val), font=_font(int(88 * s), True),
                        fill=accent + (int(255 * a),), anchor="ra")
                dd.text((x, yb + k * int(190 * s) + int(104 * s)), br(lab), font=_font(int(28 * s)),
                        fill=(220, 226, 236, int(255 * a)), anchor="ra")
        if caps:
            text, a = _caption_alpha(caps, t)
            draw_lower_third(im, text, a, s, accent)
        pipe.write(np.asarray(im.convert("RGB")))
        if progress and (i % max(1, n // 10) == 0):
            print(f"  {path}: quadro {i + 1}/{n}", flush=True)
    if clean is not None:
        clean.close()
    return pipe.close()


def cover_image(cw, path, field, title, lines, size=(1920, 1080), theta=0.6, az=-55.0, el=24.0, ssaa=1.5,
                accent=(255, 176, 64)):
    """Capa (miniatura do post): o reator em corte e o resultado principal em letras grandes."""
    from PIL import Image, ImageDraw
    W, H = size
    s = H / 1080.0
    rgb = cw.render(field, theta=theta, az=az, el=el, size=size, ssaa=ssaa)
    im = Image.fromarray(rgb).convert("RGBA")
    d = ImageDraw.Draw(im)
    d.text((int(60 * s), int(50 * s)), br(title), font=_font(int(58 * s), True), fill=(240, 244, 250, 255))
    for k, (val, lab) in enumerate(lines):
        y = int(230 * s) + k * int(200 * s)
        d.text((W - int(70 * s), y), br(val), font=_font(int(96 * s), True), fill=accent + (255,), anchor="ra")
        d.text((W - int(70 * s), y + int(112 * s)), br(lab), font=_font(int(30 * s)), fill=(225, 230, 240, 255),
               anchor="ra")
    im.convert("RGB").save(path)
    return path

In [ ]:
%%writefile video_roteiro.json
{
 "hook": {
  "kicker": "CFD 3D no Google Colab",
  "title": "Reator de pirólise de plástico",
  "subtitle": "Fita helicoidal a {rpm} rpm · parede a {Tw} °C · carga pós-consumo",
  "big": [["{vazao} kg/h", "de plástico virando óleo e cera"], ["{T} °C", "no plástico fundido"]]
 },
 "title_card": {
  "title": "Pirólise de plástico: um reator agitado simulado em 3D",
  "subtitle": "Escoamento, calor e química do fundido em um só modelo, validado com a literatura",
  "credit": "{autor}",
  "disclaimer": "Composições, reologia e cinética de artigos abertos; sem dados de projeto ou de clientes."
 },
 "context_cards": [
  {
   "kicker": "O desafio",
   "headline": "Transformar plástico em óleo exige calor no lugar certo",
   "bullets": [
    "O plástico entra derretido, a 300 °C, viscoso como mel grosso.",
    "No reator, ele precisa chegar a ~400 °C para as cadeias se quebrarem.",
    "Parede quente demais forma coque; mistura ruim deixa carga sem converter."
   ]
  },
  {
   "kicker": "O que foi feito",
   "headline": "Um reator agitado em 3D, do escoamento à química",
   "bullets": [
    "Vaso encamisado de {T_vaso} m (≈ {vol} m³) com fita helicoidal dupla.",
    "Turbulência k-ω SST, reologia do fundido e cinética de cisão de cadeias.",
    "Tudo em Python, rodando em GPU no Google Colab."
   ]
  }
 ],
 "rotation": {
  "chapter": "Dentro do reator",
  "title": "Temperatura do plástico fundido",
  "subtitle": "Corte do vaso; a fita gira em câmera lenta",
  "captions": [
   "O plástico entra a 300 °C perto da superfície e se mistura em segundos.",
   "A camisa a {Tw} °C mantém o fundido entre {T_min} e {T_max} °C."
  ]
 },
 "sections": {
  "chapter": "Escoamento e química",
  "captions": [
   "As cadeias se quebram: a massa molar cai de {Mw0} para {Mw} kg/mol no reator.",
   "Com isso a viscosidade despenca: o seio do reator vira uma cera fina, de {eta} mPa s."
  ]
 },
 "wall": {
  "chapter": "Troca térmica",
  "captions": [
   "O calor entra pela parede: {q} kW/m² em média, {duty} kW no total.",
   "Coeficiente interno de {h} W/m²K, {h_vs}."
  ]
 },
 "capacity": {
  "chapter": "Capacidade",
  "captions": [
   "Mais calor na parede, mais plástico processado: {cap_lo} a {cap_hi} kg/h entre 440 e 540 °C.",
   "No ponto simulado: {vazao} kg/h, com {duty} kW pela camisa."
  ]
 },
 "validation": {
  "chapter": "Validação",
  "captions": [
   "{n_ok} de {n} verificações dentro do alvo.",
   "Termogravimetria da literatura, potência da fita, troca térmica (Nagata) e balanços de massa e energia."
  ]
 },
 "kpis": {
  "title": "Números do ponto de operação",
  "labels": ["plástico processado", "temperatura do fundido", "calor pela camisa", "óleo e cera no vapor"]
 },
 "closing": {
  "kicker": "Para a gestão",
  "headline": "Uma ferramenta rápida para testar o reator antes de construir",
  "bullets": [
    "Compara temperaturas de parede, rotações, agitadores e cargas em horas, não semanas.",
    "Mostra onde o calor entra e onde há risco de coque ou de carga não convertida.",
    "Aberta e auditável: cada hipótese está no código e no relatório."
  ]
 },
 "end_card": {
  "line": "Simulação 3D aberta, validada e reprodutível no navegador",
  "author": "{autor}"
 },
 "caption": "Plástico pós-consumo virando óleo dentro de um reator agitado — simulado em 3D, do escoamento à química, rodando no Google Colab.\n\nO caso: um vaso encamisado de {T_vaso} m com fita helicoidal dupla a {rpm} rpm, parede a {Tw} °C e carga de poliolefinas (HDPE, LDPE e PP). O modelo junta turbulência k-ω SST, a reologia do plástico fundido (que cai de milhares de Pa s para alguns mPa s quando as cadeias se quebram) e a cinética de cisão de cadeias calibrada com termogravimetria da literatura.\n\nResultado: {vazao} kg/h de plástico processado, fundido a {T} °C e {duty} kW entrando pela camisa. {n_ok} de {n} verificações batem com a literatura (TGA, potência da fita, troca térmica de Nagata e balanços).\n\nTudo com dados abertos, sem informações de projeto. Código e relatório no GitHub.\n\n#CFD #Pirólise #Reciclagem #EconomiaCircular #EngenhariaQuímica #Simulação #Python #IndústriaQuímica"
}

In [ ]:
#@title Carrega o pacote (recarrega se as células acima mudarem)
import importlib, sys
for m in [m for m in sys.modules if m == "pyrokit" or m.startswith("pyrokit.")]:
    del sys.modules[m]
sys.path.insert(0, ".")
import numpy as np, matplotlib.pyplot as plt, torch
from IPython.display import Image, Video, display
import pyrokit
from pyrokit import chemistry as ch, figures as FG, validation as VAL, kinetics as kin
from pyrokit.geometry import ribbon_tank, standard_anchor_tank
from pyrokit.reactor import Reactor, ReactorConfig
from pyrokit.video import br
print("pyrokit", pyrokit.__version__)

## Parâmetros

In [ ]:
#@title Parâmetros do caso
PRESET = "gpu"            #@param ["teste", "gpu", "gpu_fino"]
CARGA = "F3"              #@param ["F1", "F2", "F3"]
T_PAREDE = 480.0          #@param {type:"slider", min:440, max:540, step:5}
RPM = 30.0                #@param {type:"number"}
AGITADOR = "fita"         #@param ["fita", "ancora"]
AUTOR = "Gabriel Rozo · CAExperts"   #@param {type:"string"}
LINKEDIN = "linkedin.com/in/gabrielhrozo"   #@param {type:"string"}

PRESETS = {   # células no diâmetro, voltas do agitador e iterações externas
    "teste":    dict(n_diam=40,  revs_first=8,  revs_next=2, outer=2),
    "gpu":      dict(n_diam=96,  revs_first=12, revs_next=3, outer=3),
    "gpu_fino": dict(n_diam=128, revs_first=14, revs_next=3, outer=3),
}
tank = ribbon_tank(rpm=RPM) if AGITADOR == "fita" else standard_anchor_tank(T=0.8, rpm=RPM)
feed = ch.FEEDS[CARGA]
print(tank.describe())
print("carga:", feed.name, "—", feed.describe(), "| entra a", round(feed.T_feed - 273.15), "°C")

## 1 · Cinética

In [ ]:
#@title 1 · Cinética e propriedades (validação da química)
display(FG.tga_figure("saida/fig_tga.png") and Image("saida/fig_tga.png", width=960))
cheq = VAL.chemistry_checks()
for c in cheq:
    print(("✔" if c["ok"] else "✘"), c["item"], "→", br(str(c["valor"])), "| alvo:", br(str(c["alvo"])))

## 2 · Reator ideal

In [ ]:
#@title 2 · Reator ideal 0-D: quanto plástico o reator processa
import math
V_liq = math.pi * tank.R ** 2 * tank.H_L
A_wall = 2 * math.pi * tank.R * tank.H_L + math.pi * tank.R ** 2
N, d = tank.rpm / 60, tank.impeller_diameter
h0 = 300.0
for _ in range(4):        # coeficiente interno pela correlação de Nagata, iterado com a viscosidade do fundido
    r0 = ch.solve_cstr(feed, V_liq, A_wall, h0, T_wall=T_PAREDE + 273.15)
    h0 = ch.nagata_h(r0.rho, N, d, r0.eta, 3240.0, 0.11, tank.T)[0]
curva = ch.capacity_curve(feed, V_liq, A_wall, h0)
print(br(f"h (Nagata) = {h0:.0f} W/m²K | T fundido {r0.T - 273.15:.1f} °C | {3600 * r0.feed:.0f} kg/h | "
         f"{r0.duty / 1e3:.1f} kW | residência {r0.residence_h:.2f} h | viscosidade {1e3 * r0.eta:.1f} mPa s"))

## 3 · CFD 3D

In [ ]:
#@title 3 · CFD 3D (GPU): escoamento ↔ temperatura e química, até o regime permanente
import time
cfg = ReactorConfig(T_wall_C=T_PAREDE, **PRESETS[PRESET])
R = Reactor(tank, feed, cfg)
print(R.flow.summary(), "|", f"Δ = {1e3 * R.flow.h:.1f} mm, folga da fita = {(tank.R - tank.impeller_diameter / 2) / R.flow.h:.1f} células")
t0 = time.time()
kp = R.run(verbose=True)
print(br(f"\nPronto em {(time.time() - t0) / 60:.1f} min"))
torch.save({"T": R.T.cpu(), "state": [{k: (v.cpu() if torch.is_tensor(v) else [e.cpu() for e in v]) for k, v in s.items()}
            for s in R.state], "kpis": kp}, "saida/resultado.pt")

## 4 · Resultados

In [ ]:
#@title 4 · Resultados: números do ponto de operação, cortes e mapa de parede
q_med = kp["duty_kW"] / R.A_wall
linhas = [("plástico processado", f"{kp['feed_kgph']:.0f} kg/h"), ("vapor (óleo, cera e gás)", f"{kp['vapour_kgph']:.0f} kg/h"),
          ("temperatura do fundido", f"{kp['T_bulk_C']:.1f} °C (de {kp['T_min_C']:.0f} a {kp['T_max_C']:.0f} °C)"),
          ("calor pela camisa", f"{kp['duty_kW']:.1f} kW ({q_med:.1f} kW/m²)"),
          ("coeficiente interno h", f"{kp['h_mean']:.0f} W/m²K"),
          ("massa molar no seio", f"{kp['Mw_bulk_kgmol']:.2f} kg/mol"),
          ("viscosidade no seio", f"{1e3 * kp['eta_bulk']:.1f} mPa s (Re = {kp['Re']:.0f})"),
          ("potência do agitador", f"{1e3 * kp['power_kW']:.1f} W (Np = {kp['Np']:.2f})"),
          ("tempo de residência", f"{kp['residence_h']:.2f} h"),
          ("vapor: gás / óleo / cera / coque", " / ".join(f"{100 * v:.0f}%" for v in kp["slate"].values()))]
for a, b in linhas:
    print(f"{a:34s} {br(b)}")
FG.capacity_figure(curva, kp, "saida/fig_capacidade.png")
FG.sections_figure(R, "saida/fig_cortes.png")
FG.wall_map_figure(R, "saida/fig_parede.png")
for p in ("saida/fig_capacidade.png", "saida/fig_cortes.png", "saida/fig_parede.png"):
    display(Image(p, width=960))

## 5 · Validação

In [ ]:
#@title 5 · Validação (L1 componentes · L2 química · L3 sistema)
RODAR_KP_LAMINAR = True   #@param {type:"boolean"}
kp_lam = None
if RODAR_KP_LAMINAR and AGITADOR == "fita":
    n_kp = {"teste": 32, "gpu": 48, "gpu_fino": 64}[PRESET]
    kp_lam = VAL.kp_ribbon(lambda: ribbon_tank(rpm=RPM), n_diam=n_kp, Re=8.0)
    print(br(f"Kp laminar da fita: {kp_lam['Kp']:.0f} (Re = 8, {kp_lam['steps']} passos, {kp_lam['wall_s']:.0f} s)"))
flow_c = VAL.flow_checks(kp_lam, kp, h_nagata=h0)
sys_c, ideal = VAL.system_checks(R, kp)
checks = flow_c + cheq + sys_c
VAL.dashboard(checks, "saida/fig_validacao.png", "Validação: modelo × literatura")
n_ok = sum(c["ok"] for c in checks)
print(f"{n_ok}/{len(checks)} verificações dentro do alvo")
display(Image("saida/fig_validacao.png", width=960))

## 6 · Imagem 3D

In [ ]:
#@title 6 · Imagem 3D do reator em corte (GPU)
from pyrokit.render3d import Cutaway
f = R.flow
fl = (f.fluid > 0.5)
cw = Cutaway(tank, f.xc, f.yc, f.zc)
Tc = (R.T - 273.15)
cw.set_field("T", Tc, cmap="inferno", vmin=float(Tc[fl].min()), vmax=float(Tc[fl].max()) + 1.0, fluid=fl)
Mw = FG._bulk_Mw(R)
cw.set_field("Mw", Mw, cmap="viridis", fluid=fl.cpu())
eta = R._mu(f.gamma)
cw.set_field("eta", eta, cmap="magma", log=True, fluid=fl)
for name in ("T", "Mw", "eta"):
    from PIL import Image as PILImage
    PILImage.fromarray(cw.render(name, theta=0.6, size=(1920, 1080), ssaa=1.5)).save(f"saida/reator_{name}.png")
display(Image("saida/reator_T.png", width=960))

## 7 · Vídeo

In [ ]:
#@title Roteiro do vídeo (textos editáveis; {chaves} são preenchidas com os resultados)
import json
ROTEIRO = json.load(open("video_roteiro.json", encoding="utf-8"))

In [ ]:
#@title 7 · Vídeo em HD: resultado nos primeiros segundos (completo ~1 min + curto ~25 s)
import os
from pyrokit import video as hd, story
FPS = 30                  #@param {type:"integer"}
RESOLUCAO = "1080p"       #@param ["720p", "1080p"]
size = {"720p": (1280, 720), "1080p": (1920, 1080)}[RESOLUCAO]
ssaa = 1.25 if torch.cuda.is_available() else 1.0

def nf(x, dd=0):
    return br(f"{x:.{dd}f}")

V = dict(rpm=nf(RPM), Tw=nf(T_PAREDE), vazao=nf(kp["feed_kgph"]), T=nf(kp["T_bulk_C"], 0), T_min=nf(kp["T_min_C"]),
         T_max=nf(kp["T_max_C"]), autor=AUTOR, T_vaso=nf(tank.T, 1), vol=nf(R.V_liq, 1),
         Mw0=nf(R.items[0][0].Mw0), Mw=nf(kp["Mw_bulk_kgmol"], 1), eta=nf(1e3 * kp["eta_bulk"], 1),
         q=nf(kp["duty_kW"] / R.A_wall, 1), duty=nf(kp["duty_kW"]), h=nf(kp["h_mean"]),
         h_vs=br(f"{100 * (kp['h_mean'] / h0 - 1):+.0f}% em relação à correlação de Nagata"),
         cap_lo=nf(3600 * curva[0].feed), cap_hi=nf(3600 * curva[-1].feed), n_ok=n_ok, n=len(checks))
F = lambda t: t.format(**V) if isinstance(t, str) else [F(x) for x in t]  # noqa: E731
RT = ROTEIRO
seg, curto = [], []
hk = RT["hook"]
big = [(F(a), F(b)) for a, b in hk["big"]]
seg.append(story.render_rotation(cw, "saida/s00_gancho.mp4", "T", F(hk["title"]), F(hk["subtitle"]), F(hk["kicker"]),
                                 cbar_label="T [°C]", seconds=6.0, fps=FPS, size=size, ssaa=ssaa, big=big,
                                 handle=LINKEDIN, clean_path="saida/fundo.mp4", turns=0.35))
curto.append(seg[-1])
fundo = hd.Backdrop("saida/fundo.mp4", size)
tc = RT["title_card"]
seg.append(hd.render_title_card("saida/s01_titulo.mp4", F(tc["title"]), F(tc["subtitle"]), F(tc["credit"]),
                                F(tc["disclaimer"]), size=size, fps=FPS, seconds=4.0, backdrop=fundo))
cards = [{k: F(v) for k, v in c.items()} for c in RT["context_cards"]]
seg.append(hd.render_text_cards("saida/s02_contexto.mp4", cards, size=size, fps=FPS, seconds=5.5, backdrop=fundo))
ro = RT["rotation"]
seg.append(story.render_rotation(cw, "saida/s03_rotacao.mp4", "T", F(ro["title"]), F(ro["subtitle"]), F(ro["chapter"]),
                                 captions=F(ro["captions"]), cbar_label="T [°C]", seconds=8.0, fps=FPS, size=size,
                                 ssaa=ssaa, handle=LINKEDIN, az0=-40, az1=-100, turns=0.5))
curto.append(seg[-1])
for key, fig in (("sections", "saida/fig_cortes.png"), ("wall", "saida/fig_parede.png"),
                 ("capacity", "saida/fig_capacidade.png"), ("validation", "saida/fig_validacao.png")):
    sx = RT[key]
    out = f"saida/s_{key}.mp4"
    seg.append(hd.caption_segment(fig, out, F(sx["captions"]), F(sx["chapter"]), size=size, fps=FPS,
                                  seconds=6.5, handle=LINKEDIN))
    if key in ("capacity",):
        curto.append(out)
kv = [dict(value=kp["feed_kgph"], fmt="{:.0f}", unit="kg/h"), dict(value=kp["T_bulk_C"], fmt="{:.0f}", unit="°C"),
      dict(value=kp["duty_kW"], fmt="{:.0f}", unit="kW"),
      dict(value=100 * (kp["slate"]["óleo"] + kp["slate"]["cera"]), fmt="{:.0f}", unit="%")]
for d_, lab in zip(kv, RT["kpis"]["labels"]):
    d_["label"] = lab
seg.append(hd.render_kpi_card("saida/s08_numeros.mp4", RT["kpis"]["title"], kv, size=size, fps=FPS, seconds=6.0,
                              backdrop=fundo))
curto.append(seg[-1])
seg.append(hd.render_text_cards("saida/s09_gestao.mp4", [{k: F(v) for k, v in RT["closing"].items()}], size=size,
                                fps=FPS, seconds=7.0, backdrop=fundo))
ec = RT["end_card"]
seg.append(hd.render_title_card("saida/s10_final.mp4", F(ec["line"]), "", F(ec["author"]), "", size=size, fps=FPS,
                                seconds=4.0, backdrop=fundo, contact=LINKEDIN))
curto.append(seg[-1])
hd.assemble(seg, "saida/pirolise_reator.mp4", fps=FPS, size=size)
hd.assemble(curto, "saida/pirolise_reator_curto.mp4", fps=FPS, size=size)
story.cover_image(cw, "saida/capa.png", "T", F(hk["title"]), big, size=(1920, 1080), ssaa=ssaa)
for p in ("saida/pirolise_reator.mp4", "saida/pirolise_reator_curto.mp4"):
    print(br(f"{p}: {os.path.getsize(p) / 1e6:.1f} MB"))
display(Video("saida/pirolise_reator_curto.mp4", embed=True, width=960))
display(Image("saida/capa.png", width=640))
print("\nTexto sugerido para o post:\n")
print(br(F(RT["caption"])))

## Limitações e próximos passos

* **Escoamento**: RANS k-ω SST com lei de parede numa malha cartesiana com fronteira imersa; a parede curva do vaso usa uma camada de deslizamento com a tensão de Spalding aplicada na área real. O balanço de torque fecha melhor a cada refinamento (o resíduo cai de forma aproximadamente linear com Δ).
* **Química**: cisão aleatória com fechamento em três classes de cadeias (em vez do balanço populacional completo); seletividade de gás calibrada para tanque agitado; sem craqueamento secundário no vapor.
* **Fases**: o vapor sai do líquido no ponto em que é gerado (sem bolhas nem espuma resolvidas); superfície livre plana.
* **Próximos passos**: espaço de vapor e condensador; âncora × fita; PS e PET na carga; estudo de malha (GCI) no preset `gpu_fino`.